# CoHiM-FL — Independent Seed 126

This notebook executes **only seed 126** as one of five pre-specified independent repeated runs: `[7, 21, 42, 84, 126]`. The train/validation/test split remains fixed (`DATA_SPLIT_SEED=42`) for fair comparison. Do **not** compute or report Mean ± SD from this notebook alone; use the separate aggregation notebook after all five runs finish.

Each seed writes to an isolated Google Drive directory so results cannot overwrite another seed.


## Archived results and reproducibility notice

This repository includes **original result files without numeric modifications**. The archived run used the method identifier `Proposed_CBA_NHFL_MI_Q1`, whereas these notebooks now use `Proposed_CoHiM_FL`. The aggregation notebook transparently maps only this identifier when it reads archived CSVs. This is a **label-compatibility change**, not evidence that a re-run of the renamed code has been completed. Before a reproducibility claim, re-run from the archived environment, verify GPU/software dependencies, and check generated results.

- Five-seed archived results: `results/CoHiM-FL_Split_Seeds/`
- Weight sensitivity: `results/CoHiM-FL_Weight_Sensitivity_5Seeds/`
- Audit limitations and guide: `README.md`, `reports/RESULTS_AUDIT.md`


In [ ]:
# =========================================================
# IU-Xray + CheXbert Labels + Federated Learning
# FedAvg / FedProx / Fedsw (Chen & Pan 2024) / CAR-MFL / Proposed CoHiM-FL
# Academic optimization; no manual metric modification
# =========================================================

!pip -q install nltk scikit-learn torchvision pandas tqdm pillow rouge-score openpyxl

import os, re, glob, zipfile, random, warnings, shutil, subprocess, sys, copy, math
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, jaccard_score, hamming_loss, precision_score, recall_score
from sklearn.cluster import KMeans
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# =========================================================
# Independent-run identity
# IMPORTANT: RUN_SEED must be defined BEFORE any seed-specific path.
# =========================================================
RUN_SEED = 126
ALL_EXPERIMENT_SEEDS = [7, 21, 42, 84, 126]
EXPERIMENT_SEEDS = [RUN_SEED]

assert RUN_SEED in ALL_EXPERIMENT_SEEDS, (
    f"Unexpected RUN_SEED={RUN_SEED}. "
    f"Expected one of {ALL_EXPERIMENT_SEEDS}."
)

# Isolated output root for this independent seed.
SPLIT_RUN_ROOT = "/content/drive/MyDrive/CoHiM-FL_Split_Seeds"
SEED_RUN_DIR = os.path.join(SPLIT_RUN_ROOT, f"seed_{RUN_SEED}")
os.makedirs(SEED_RUN_DIR, exist_ok=True)
print("Independent experiment seed:", RUN_SEED)
print("Seed-specific output directory:", SEED_RUN_DIR)

# =========================================================
# 1) Paths
# =========================================================

ZIP_PATH = "/content/drive/MyDrive/Indiana.zip"
LOCAL_ZIP = "/content/Indiana.zip"
EXTRACT_DIR = "/content/indiana_dataset"

CHEXBERT_DIR = "/content/CheXbert"
CHEXBERT_CKPT = "/content/chexbert.pth"
CHEXBERT_OUTPUT = "/content/chexbert_output"
CHEXBERT_LABEL_CACHE = "/content/drive/MyDrive/indiana_chexbert_labels.csv"

# Versioned cache because this notebook uses train/validation/test partitions.
FEATURE_CACHE = "/content/drive/MyDrive/indiana_densenet121_chexbert14_train_val_test_v2.npz"

assert os.path.exists(ZIP_PATH), f"ZIP file not found: {ZIP_PATH}"

if not os.path.exists(LOCAL_ZIP):
    shutil.copy2(ZIP_PATH, LOCAL_ZIP)

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP, "r") as zf:
        zf.extractall(EXTRACT_DIR)

print("Ready:", EXTRACT_DIR)

# =========================================================
# 2) Reproducible configuration
# =========================================================

# Publication-level repeated evaluation.
# DATA_SPLIT_SEED is fixed so all runs share the same train/validation/test studies.
# EXPERIMENT_SEEDS govern stochastic training, model initialization, Non-IID
# client partitioning, client selection, local shuffling, and augmentation.
DATA_SPLIT_SEED = 42
TUNING_SEED = DATA_SPLIT_SEED
SEED = int(TUNING_SEED)  # mutable current experiment seed; multi-seed loop resets it

print(
    "Run identity audit:",
    {
        "RUN_SEED": RUN_SEED,
        "EXPERIMENT_SEEDS": EXPERIMENT_SEEDS,
        "DATA_SPLIT_SEED": DATA_SPLIT_SEED,
        "SEED_RUN_DIR": SEED_RUN_DIR,
    }
)

def set_global_seed(seed):
    seed = int(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_global_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

NUM_CLIENTS = 40
NUM_LABELS = 14                 # clinically defined CheXbert labels only

# Chen & Pan (2024) report: 40 clients, 5 communication rounds,
# 20 local training rounds, batch size 16, and FedProx mu=0.01.
CHEN_PAN_REFERENCE_ROUNDS = 5
CHEN_PAN_REFERENCE_LOCAL_EPOCHS = 20
CHEN_PAN_REFERENCE_BATCH_SIZE = 16
CHEN_PAN_REFERENCE_CLIENT_SETTINGS = [5, 8, 20, 40]

# Default = controlled manuscript comparison. Set True only for a
# reproduction-style training-budget run based on the reference paper.
USE_CHEN_PAN_REFERENCE_TRAINING_BUDGET = False

if USE_CHEN_PAN_REFERENCE_TRAINING_BUDGET:
    ROUNDS = CHEN_PAN_REFERENCE_ROUNDS
    LOCAL_EPOCHS = CHEN_PAN_REFERENCE_LOCAL_EPOCHS
    BATCH_SIZE = CHEN_PAN_REFERENCE_BATCH_SIZE
else:
    ROUNDS = 15
    LOCAL_EPOCHS = 2
    BATCH_SIZE = 16

IMG_SIZE = 224
MAX_LEN = 80

LR = 1e-4
MIN_LR_FACTOR = 0.20
WEIGHT_DECAY = 1e-4
FEDPROX_MU = 0.01

CHEN_PAN_2024_REFERENCE = (
    "Chen, J. & Pan, R. (2024), Medical report generation based on multimodal "
    "federated learning, Computerized Medical Imaging and Graphics 113, 102342, "
    "DOI:10.1016/j.compmedimag.2024.102342"
)

# Correct Fedsw (federal score weight) configuration.
# Paper Algorithm 3 uses test-set scores; publication-safe implementation here
# uses VALIDATION ONLY so final test performance cannot influence training.
FEDSW_SCORE_SPLIT = "validation"
FEDSW_SCORE_METRIC = "BLEU1"
FEDSW_SCORE_EPS = 1e-8
FEDSW_SCORE_MAX_BATCHES = None  # None = complete validation split
FEDSW_STATIC_CLIENT_SET = True  # mirrors fixed institution sets in paper experiments
FEDSW_REFERENCE = CHEN_PAN_2024_REFERENCE

# Canonical FedAvg local optimizer (McMahan et al., AISTATS 2017, Algorithm 1).
# Algorithmic core: local mini-batch SGD + sample-size weighted server averaging.
# Hyperparameter principle: McMahan et al. report that SGD/FedAvg is sensitive
# to eta and use a sufficiently wide multiplicative grid of learning rates.
# Here LR selection is validation-only (never test) and deterministic.
FEDAVG_SGD_MOMENTUM = 0.0
FEDAVG_SGD_WEIGHT_DECAY = 0.0
FEDAVG_GRAD_CLIP_NORM = None          # canonical Algorithm 1 has no gradient clipping

# Multiplicative grid with factor 10^(1/3), spanning 1e-4 to 1.0 (13 candidates).
# This mirrors the grid-search principle reported by McMahan et al. while keeping
# all model selection on the validation split in this notebook.
FEDAVG_LR_GRID = np.asarray([10.0 ** (-4.0 + i / 3.0) for i in range(13)], dtype=np.float64)
FEDAVG_TUNING_CLIENTS = 10            # pre-specified representative configuration
FEDAVG_TUNING_ROUNDS = 3              # short validation-only pilot; final run still uses ROUNDS
FEDAVG_REQUIRE_INTERIOR_BEST_LR = True
FEDAVG_FALLBACK_LR = 1e-3             # used only if tuning is explicitly disabled
RUN_FEDAVG_LR_TUNING = True
FEDAVG_SELECTED_LR = None
FEDAVG_LR_TUNING_DF = None

# Fail-fast audit for the proposed cross-view branch before full experiments.
RUN_CROSS_VIEW_GRADIENT_AUDIT = True
GRADIENT_AUDIT_EPS = 1e-12

# Keep the manuscript's established settings by default; the source paper used
# 5/8, 20 and 40 institutions for few/medium/all experiments.
CLIENT_SETTINGS = [5, 10, 20, 40]
REFERENCE_BASELINE_METHODS = ("FedAvg", "FedProx", "Fedsw")
METHODS = ["FedAvg", "FedProx", "Fedsw", "CAR-MFL", "Proposed_CoHiM_FL"]

# Validation calibration; no test-set tuning.
VALIDATION_FRACTION_OF_ALL_DATA = 0.10
TEST_FRACTION_OF_ALL_DATA = 0.20
THRESHOLD_GRID = np.arange(0.10, 0.81, 0.025)

# Missing-modality learning.
CAR_TRAIN_MISSING_RATE = 0.20

# Poudel et al. (MICCAI 2024) CAR-MFL retrieval configuration.
# The source paper retrieves the top 10 intra-modal neighbors and then applies
# label refinement by Jaccard similarity. The original source uses a separate
# public multimodal dataset. In this IU-Xray adaptation a disjoint training-only
# public bank is created specifically for CAR-MFL; other methods are untouched.
CAR_MFL_2024_REFERENCE = (
    "Poudel et al. (2024), CAR-MFL: Cross-Modal Augmentation by Retrieval "
    "for Multimodal Federated Learning with Missing Modalities, MICCAI 2024, "
    "pp. 102-112, DOI:10.1007/978-3-031-72117-5_10"
)
CAR_RETRIEVAL_TOPK = 10
CAR_LABEL_REFINEMENT = "Jaccard"
CAR_SOURCE_ALPHA = 0.30  # source-paper complementary-encoder re-weighting coefficient
CAR_PUBLIC_BANK_FRACTION = 0.10
CAR_PUBLIC_BANK_MIN_SIZE = 128
CAR_PUBLIC_BANK_MAX_SIZE = 250
CAR_PUBLIC_BANK_SEED = SEED + 2407
CAR_USE_PUBLIC_VIRTUAL_CLIENT = True
CAR_INFERENCE_USE_PSEUDO_LABEL_REFINEMENT = True
PROPOSED_MISSING_RATE_START = 0.00
PROPOSED_MISSING_RATE_END = 0.10
PROPOSED_MISSING_WARMUP_FRACTION = 0.40
TEXT_LOSS_W = 1.00
LABEL_LOSS_W = 0.50
PROPOSED_IMPUTATION_LOSS_W = 0.01
PROPOSED_CONSISTENCY_LOSS_W = 0.002

# Proposed client selection / adaptive hierarchical aggregation.
# Edge resources are derived from the requested participation level and
# the current client clusters; they are not fixed across configurations.
ADAPTIVE_EDGE_MIN = 1
ADAPTIVE_EDGE_MAX = 8
TARGET_CLIENTS_PER_EDGE = 5
EDGE_BUDGET_MARGIN = 1.10
ROBUST_ALPHA = 0.50
ROBUST_CLIP_MULTIPLIER = 3.00
SERVER_MOMENTUM_BETA = 0.20

# Communication protocol for the proposed method.
# The proposed uplink applies actual adaptive Top-K sparsification followed by
# signed INT8 quantization. The retained ratio is selected from a predefined
# candidate set using update-energy preservation and the round schedule only.
# Test metrics are never used to select the Top-K ratio.
PROPOSED_UPDATE_QUANT_BITS = 8
PROPOSED_INITIAL_DOWNLINK_BITS = 16   # first participation: full FP16 model
PROPOSED_DELTA_DOWNLINK_BITS = 8      # later participation: cached INT8 model delta
PROPOSED_USE_ERROR_FEEDBACK = True    # compensates quantization/sparsification error

# Adaptive Top-K policy. These values are fixed before test evaluation.
ADAPTIVE_TOPK_CANDIDATES = [0.15, 0.20, 0.25, 0.30, 0.40]
ADAPTIVE_TOPK_TARGET_ENERGY = 0.95
ADAPTIVE_TOPK_WARMUP_ROUNDS = 2
ADAPTIVE_TOPK_WARMUP_RATIO = 0.40
ADAPTIVE_TOPK_MIN_ELEMENTS = 32
ADAPTIVE_TOPK_USE_BITMAP = True

COMM_TENSOR_METADATA_BYTES = 16       # tensor identifier/shape/dtype bookkeeping
COMM_SCALE_BYTES = 4                  # one FP32 scale per quantized tensor
COMM_BITMAP_HEADER_BYTES = 8          # sparse-mask shape/length bookkeeping
COMM_COUNT_DOWNLINK = True

# Ablation protocol: one representative client setting is pre-specified before
# test evaluation to avoid test-driven configuration selection. Change to
# [5, 10, 20, 40] only if a full factorial ablation is computationally feasible.
RUN_ABLATION_STUDY = True
ABLATION_CLIENT_SETTINGS = [20]
ABLATION_USE_ALL_FIVE_SEEDS = False  # one seed per split notebook

# Publication-completeness controls.
# Exact missingness uses the nearest achievable integer study count (round(rate*N))
# and deterministic nested masks. Model checkpoints are saved to support clinical
# and reconstruction stress evaluations without retraining.
SAVE_MODEL_CHECKPOINTS = True
SAVE_ALL_ABLATION_CHECKPOINTS = False  # save only Full + w/o Reconstruction by default
ARCHIVE_MODEL_CHECKPOINTS = False       # keep result ZIP manageable; checkpoints stay on Drive
RUN_GENERATED_REPORT_CHEXBERT = True
RUN_RECONSTRUCTION_MISSING_ABLATION = True

ACADEMIC_VERSION = "2026_5seed_mean_sd_ablation_exactmissing_chexbert_checkpointed_v3"


In [ ]:

# =========================================================
# 3) Read Indiana CSV
# =========================================================

reports_csv = f"{EXTRACT_DIR}/indiana_reports.csv"
projections_csv = f"{EXTRACT_DIR}/indiana_projections.csv"

reports = pd.read_csv(reports_csv)
projections = pd.read_csv(projections_csv)

print("Reports columns:", reports.columns.tolist())
print("Projections columns:", projections.columns.tolist())

def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).lower()
    s = re.sub(r"x{2,}", " ", s)
    s = re.sub(r"[^a-z0-9.,;:!?()/-]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

img_files = glob.glob(EXTRACT_DIR + "/**/*.png", recursive=True) + \
            glob.glob(EXTRACT_DIR + "/**/*.jpg", recursive=True) + \
            glob.glob(EXTRACT_DIR + "/**/*.jpeg", recursive=True)

img_map = {os.path.basename(p): p for p in img_files}

merged = projections.merge(reports, on="uid", how="inner")

merged["report"] = merged["findings"].apply(clean_text)
merged = merged[merged["report"].str.split().str.len() >= 5].copy()

merged["img_path"] = merged["filename"].apply(
    lambda x: img_map.get(os.path.basename(str(x)), None)
)
merged = merged.dropna(subset=["img_path"])

records = []

for uid, g in merged.groupby("uid"):
    imgs = g["img_path"].dropna().unique().tolist()
    if len(imgs) == 0:
        continue

    while len(imgs) < 2:
        imgs.append(imgs[0])

    row = g.iloc[0]

    records.append({
        "uid": uid,
        "report": row["report"],
        "mesh": clean_text(row["MeSH"]) if "MeSH" in row.index else "",
        "problems": clean_text(row["Problems"]) if "Problems" in row.index else "",
        "img1": imgs[0],
        "img2": imgs[1]
    })

df = pd.DataFrame(records).reset_index(drop=True)

print("Final usable samples:", len(df))
display(df.head())


In [ ]:
# =========================================================
# 4) REAL CheXbert Label Extraction + Reusable Clinical Scoring Utilities
# =========================================================
# Publication policy:
# - Real CheXbert is mandatory for reported experiments.
# - No keyword/synthetic substitute is used in publication runs.
# - The same positive-label mapping is used for reference reports and
#   generated-report clinical correctness evaluation.
# - CheXbert repository/checkpoint are prepared once and then reused.
# =========================================================

import os, sys, subprocess, re, shutil, hashlib, json
import numpy as np
import pandas as pd

chexbert_cols = [
    "No Finding", "Enlarged Cardiomediastinum", "Cardiomegaly",
    "Lung Lesion", "Lung Opacity", "Edema", "Consolidation",
    "Pneumonia", "Atelectasis", "Pneumothorax", "Pleural Effusion",
    "Pleural Other", "Fracture", "Support Devices"
]

CHEXBERT_DIR = "/content/CheXbert"
CHEXBERT_CKPT = "/content/chexbert.pth"
CHEXBERT_OUTPUT = "/content/chexbert_output"
CHEXBERT_LABEL_CACHE = "/content/chexbert_labels.csv"
ALLOW_KEYWORD_FALLBACK = False


def fallback_keyword_labels(frame):
    """Debug-only emergency fallback. Disabled for publication experiments."""
    patterns = {
        0: [r"\bno acute cardiopulmonary\b", r"\bno active disease\b", r"\bnormal chest\b"],
        1: [r"\benlarged cardiomediastinum\b", r"\bwidened mediastinum\b"],
        2: [r"\bcardiomegaly\b", r"\benlarged heart\b"],
        3: [r"\blung lesion\b", r"\bmass\b", r"\bnodule\b"],
        4: [r"\blung opacity\b", r"\bopacity\b", r"\binfiltrate\b"],
        5: [r"\bedema\b"], 6: [r"\bconsolidation\b"], 7: [r"\bpneumonia\b"],
        8: [r"\batelectasis\b"], 9: [r"\bpneumothorax\b"],
        10: [r"\bpleural effusion\b", r"\beffusion\b"],
        11: [r"\bpleural thickening\b", r"\bpleural abnormality\b"],
        12: [r"\bfracture\b"],
        13: [r"\bsupport device\b", r"\bcatheter\b", r"\bpacemaker\b", r"\btube\b"],
    }
    output = np.zeros((len(frame), 14), dtype=np.float32)
    for i, text in enumerate(frame["report"].fillna("").astype(str).str.lower()):
        for label_id, regexes in patterns.items():
            if any(re.search(regex, text) for regex in regexes):
                output[i, label_id] = 1.0
    return output


def _write_fixed_chexbert_tokenizer():
    chexbert_src = os.path.join(CHEXBERT_DIR, "src")
    for pkg in ["datasets", "models"]:
        pkg_dir = os.path.join(chexbert_src, pkg)
        if os.path.isdir(pkg_dir):
            open(os.path.join(pkg_dir, "__init__.py"), "a").close()

    bert_tok_file = os.path.join(chexbert_src, "bert_tokenizer.py")
    fixed_code = r"""\
import re
import pandas as pd

def get_impressions_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    if "Report Impression" in df.columns:
        impressions = df["Report Impression"].fillna("").astype(str).tolist()
    elif "report" in df.columns:
        impressions = df["report"].fillna("").astype(str).tolist()
    else:
        impressions = df.iloc[:, 0].fillna("").astype(str).tolist()
    return impressions

def tokenize(impressions, tokenizer):
    output = []
    for imp in impressions:
        imp = str(imp).lower()
        imp = re.sub(r'\s+', ' ', imp).strip()
        tokens = tokenizer.tokenize(imp)[:510]
        ids = tokenizer.convert_tokens_to_ids(['[CLS]'] + tokens + ['[SEP]'])
        ids = (ids + [0] * 512)[:512]
        output.append(ids)
    return output
"""
    with open(bert_tok_file, "w", encoding="utf-8") as f:
        f.write(fixed_code)


def prepare_chexbert(force_reclone=False):
    if force_reclone and os.path.exists(CHEXBERT_DIR):
        shutil.rmtree(CHEXBERT_DIR)

    if not os.path.exists(os.path.join(CHEXBERT_DIR, "src", "label.py")):
        if os.path.exists(CHEXBERT_DIR):
            shutil.rmtree(CHEXBERT_DIR)
        subprocess.run(
            ["git", "clone", "https://github.com/stanfordmlgroup/CheXbert.git", CHEXBERT_DIR],
            check=True,
        )

    _write_fixed_chexbert_tokenizer()

    if not os.path.exists(CHEXBERT_CKPT):
        subprocess.run([
            "wget", "-q", "-O", CHEXBERT_CKPT,
            "https://huggingface.co/StanfordAIMI/RRG_scorers/resolve/main/chexbert.pth",
        ], check=True)

    if not os.path.isfile(CHEXBERT_CKPT) or os.path.getsize(CHEXBERT_CKPT) == 0:
        raise RuntimeError("CheXbert checkpoint is missing or empty.")

    return os.path.join(CHEXBERT_DIR, "src")


def _chexbert_binary_matrix(labeled):
    matrix = np.zeros((len(labeled), 14), dtype=np.float32)
    for j, col in enumerate(chexbert_cols):
        if col not in labeled.columns:
            raise RuntimeError(f"Missing CheXbert output column: {col}")
        vals = pd.to_numeric(labeled[col], errors="coerce").fillna(0).to_numpy()
        # Same publication mapping used for the original training targets:
        # only explicitly positive/present CheXbert state (1) is treated as positive.
        matrix[:, j] = np.where(vals == 1, 1.0, 0.0)
    return matrix


def _reports_sha256(texts):
    h = hashlib.sha256()
    for text in texts:
        b = str(text).encode("utf-8", errors="replace")
        h.update(len(b).to_bytes(8, "little"))
        h.update(b)
    return h.hexdigest()


def run_chexbert_on_texts(texts, tag="reports", cache_dir=None):
    """Run REAL CheXbert on an ordered list of reports with deterministic caching.

    Returns
    -------
    binary_matrix : np.ndarray [N,14]
        Positive/present CheXbert labels mapped identically to the training target.
    labeled_df : pd.DataFrame
        Raw CheXbert output, preserving input order.
    cache_metadata : dict
        Input fingerprint and cache provenance.
    """
    texts = ["" if x is None else str(x) for x in texts]
    fingerprint = _reports_sha256(texts)
    safe_tag = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(tag)).strip("_") or "reports"

    if cache_dir is None:
        cache_dir = "/content/chexbert_reusable_cache"
    os.makedirs(cache_dir, exist_ok=True)

    cache_csv = os.path.join(cache_dir, f"{safe_tag}_labeled.csv")
    cache_meta = os.path.join(cache_dir, f"{safe_tag}_meta.json")

    if os.path.isfile(cache_csv) and os.path.isfile(cache_meta):
        try:
            with open(cache_meta, "r", encoding="utf-8") as f:
                meta = json.load(f)
            if meta.get("sha256") == fingerprint and int(meta.get("n_reports", -1)) == len(texts):
                labeled = pd.read_csv(cache_csv)
                if len(labeled) == len(texts):
                    return _chexbert_binary_matrix(labeled), labeled, meta
        except Exception:
            pass

    chexbert_src = prepare_chexbert(force_reclone=False)
    input_csv = os.path.join(cache_dir, f"{safe_tag}_input.csv")
    output_dir = os.path.join(cache_dir, f"{safe_tag}_output")
    if os.path.isdir(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    pd.DataFrame({"Report Impression": texts}).to_csv(input_csv, index=False)
    cmd = [
        sys.executable, "label.py",
        f"-d={input_csv}", f"-o={output_dir}", f"-c={CHEXBERT_CKPT}",
    ]
    env = os.environ.copy()
    env["PYTHONPATH"] = chexbert_src + os.pathsep + env.get("PYTHONPATH", "")

    print(f"Running REAL CheXbert: {safe_tag} | N={len(texts)}")
    result = subprocess.run(
        cmd, cwd=chexbert_src, env=env,
        capture_output=True, text=True,
    )
    if result.stdout:
        print(result.stdout[-1500:])
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr[-4000:])
        raise RuntimeError(f"Real CheXbert failed for tag={safe_tag}.")

    labeled_path = os.path.join(output_dir, "labeled_reports.csv")
    if not os.path.isfile(labeled_path):
        raise RuntimeError(f"CheXbert output not found: {labeled_path}")

    labeled = pd.read_csv(labeled_path)
    if len(labeled) != len(texts):
        raise RuntimeError(
            f"CheXbert row-count mismatch for {safe_tag}: {len(labeled)} != {len(texts)}"
        )

    labeled.to_csv(cache_csv, index=False)
    meta = {
        "tag": safe_tag,
        "n_reports": len(texts),
        "sha256": fingerprint,
        "positive_mapping": "CheXbert state == 1 -> positive; all other states -> 0",
    }
    with open(cache_meta, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    return _chexbert_binary_matrix(labeled), labeled, meta


def run_chexbert_labeling(frame):
    labels, labeled, _meta = run_chexbert_on_texts(
        frame["report"].fillna("").astype(str).tolist(),
        tag="indiana_reference_all_usable_studies",
        cache_dir="/content/chexbert_reference_cache",
    )
    labeled.to_csv(CHEXBERT_LABEL_CACHE, index=False)
    print("Real CheXbert labels saved:", CHEXBERT_LABEL_CACHE)
    return labeled, labels


try:
    labeled, chexbert_label_matrix = run_chexbert_labeling(df)
    print("REAL CheXbert labeling completed successfully.")
except Exception as e:
    print("CheXbert labeling failed:", e)
    if not ALLOW_KEYWORD_FALLBACK:
        raise RuntimeError(
            "CheXbert must complete successfully for the publication experiment. "
            "Keyword fallback is prohibited for reported results."
        ) from e
    print("Debug-only keyword fallback enabled.")
    chexbert_label_matrix = fallback_keyword_labels(df)

print("CheXbert first-14 labels example:", chexbert_label_matrix[0])


In [ ]:
# =========================================================
# 5) Build the final 14-label CheXbert target matrix
# =========================================================
# Academic correction:
# - The previous 105-dimensional target mixed 14 CheXbert labels with keyword labels
#   and synthetic filler labels. That creates an inconsistent classification target.
# - This version uses only the 14 clinically defined CheXbert labels for every method.
# - No sample is relabeled to force a positive class.

assert chexbert_label_matrix.shape[1] == 14, (
    f"Expected 14 CheXbert labels, received {chexbert_label_matrix.shape[1]}"
)

NUM_LABELS = 14

df["labels"] = [
    chexbert_label_matrix[i].astype(np.float32).copy()
    for i in range(len(df))
]

label_matrix_audit = np.vstack(df["labels"].values)
assert label_matrix_audit.shape == (len(df), NUM_LABELS)
assert np.isin(label_matrix_audit, [0.0, 1.0]).all(), "Labels must be binary."

print("Final label space:", NUM_LABELS, "CheXbert classes")
print("Mean labels per study:", float(label_matrix_audit.sum(axis=1).mean()))
print("Example label vector:", df["labels"].iloc[0])


In [ ]:

# =========================================================
# 6) Train / Validation / Test Split + TRAIN-ONLY Vocabulary
# =========================================================
# Publication-safe order:
#   (1) isolate final test data,
#   (2) create validation data,
#   (3) build the text vocabulary from TRAINING REPORTS ONLY.
#
# This prevents vocabulary leakage from validation/test reports.
# The final test split is never used for hyperparameter or vocabulary selection.
# =========================================================

def primary_label_for_split(y):
    positive = np.flatnonzero(np.asarray(y) > 0)
    return int(positive[0]) if len(positive) else NUM_LABELS


def safe_split(frame, test_size, random_state):
    targets = np.array([primary_label_for_split(y) for y in frame["labels"].values])
    values, counts = np.unique(targets, return_counts=True)
    stratify = targets if len(values) > 1 and counts.min() >= 2 else None
    try:
        a, b = train_test_split(
            frame,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify,
        )
    except ValueError:
        a, b = train_test_split(
            frame,
            test_size=test_size,
            random_state=random_state,
            stratify=None,
        )
    return a.reset_index(drop=True), b.reset_index(drop=True)


train_val_df, test_df = safe_split(
    df, TEST_FRACTION_OF_ALL_DATA, DATA_SPLIT_SEED
)
relative_val_fraction = (
    VALIDATION_FRACTION_OF_ALL_DATA
    / (1.0 - TEST_FRACTION_OF_ALL_DATA)
)
train_df, val_df = safe_split(
    train_val_df, relative_val_fraction, DATA_SPLIT_SEED + 1
)

print("Fixed data-split seed:", DATA_SPLIT_SEED)
print("Experiment seeds:", EXPERIMENT_SEEDS)
print("Federated train:", len(train_df))
print("Validation:", len(val_df))
print("Final test:", len(test_df))

# -----------------------------
# Training-only vocabulary
# -----------------------------
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
PAD, BOS, EOS, UNK = 0, 1, 2, 3

def tokenize(text):
    return str(text).lower().split()

freq = {}
for txt in train_df["report"]:
    for word in tokenize(txt):
        freq[word] = freq.get(word, 0) + 1

vocab = SPECIALS + sorted(
    [word for word, count in freq.items() if count >= 2]
)
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
VOCAB_SIZE = len(vocab)

# Leakage audit: every vocabulary token (except special tokens) must have
# originated from the training split.
training_token_set = set()
for txt in train_df["report"]:
    training_token_set.update(tokenize(txt))

assert all(
    token in SPECIALS or token in training_token_set
    for token in vocab
), "Vocabulary leakage audit failed."

print("VOCAB_SIZE (training split only):", VOCAB_SIZE)
print("Vocabulary leakage audit: PASS")


def encode_text(text):
    ids = [BOS] + [
        word2idx.get(word, UNK)
        for word in tokenize(text)[:MAX_LEN - 2]
    ] + [EOS]
    ids += [PAD] * (MAX_LEN - len(ids))
    return np.asarray(ids, dtype=np.int64)


def decode_ids(ids):
    words = []
    for token_id in ids:
        token_id = int(token_id)
        if token_id in [PAD, BOS]:
            continue
        if token_id == EOS:
            break
        words.append(idx2word.get(token_id, "<unk>"))
    return " ".join(words)


In [ ]:
# ============================================================
# PUBLICATION-GRADE DATA SPLIT LEAKAGE AUDIT
# ============================================================
# Purpose
# -------
# This cell MUST run after the fixed Train/Validation/Test split is created
# and BEFORE feature extraction or federated training starts.
#
# It verifies:
#   1) zero UID overlap across Train / Validation / Test,
#   2) zero image-file overlap across the three partitions,
#   3) one study-level row per UID in the master dataframe,
#   4) exact partition accounting,
#   5) optional patient-level disjointness IF a reliable patient identifier
#      is explicitly available in the source CSV metadata,
#   6) a deterministic split fingerprint for cross-seed reproducibility.
#
# IMPORTANT:
# - If no explicit patient identifier is available in the Indiana metadata,
#   the audit does NOT claim a patient-wise split.
# - In that case, the scientifically supported claim is study/report-level
#   disjointness by UID plus image-file disjointness.
# ============================================================

import json as _audit_json
import hashlib as _audit_hashlib
from pathlib import Path as _AuditPath

_AUDIT_DIR = _AuditPath(SEED_RUN_DIR) / "data_integrity"
_AUDIT_DIR.mkdir(parents=True, exist_ok=True)

_AUDIT_JSON_PATH = _AUDIT_DIR / f"data_split_leakage_audit_seed_{RUN_SEED}.json"
_AUDIT_MEMBERSHIP_PATH = _AUDIT_DIR / f"data_split_membership_seed_{RUN_SEED}.csv"


def _audit_normalize_identifier_column(name):
    """Normalize a metadata-column name only for exact identifier matching."""
    return re.sub(r"[^a-z0-9]+", "", str(name).strip().lower())


def _audit_as_clean_string_set(series):
    """Convert a pandas Series to a clean set of non-empty string identifiers."""
    out = set()
    for value in series.dropna().tolist():
        value = str(value).strip()
        if value and value.lower() not in {"nan", "none"}:
            out.add(value)
    return out


def _audit_image_set(frame):
    """Canonical absolute image paths used by one split."""
    paths = set()
    for col in ("img1", "img2"):
        if col not in frame.columns:
            continue
        for p in frame[col].dropna().astype(str):
            p = p.strip()
            if p:
                paths.add(os.path.realpath(os.path.abspath(p)))
    return paths


def _audit_sha256_of_values(values):
    payload = "\n".join(sorted(map(str, values))).encode("utf-8")
    return _audit_hashlib.sha256(payload).hexdigest()


def _audit_find_patient_mapping():
    """
    Detect an EXPLICIT patient identifier in the original source CSV metadata.

    Conservative rule:
    only exact normalized identifier names are accepted.
    We intentionally do NOT infer patient identity from UID.
    """
    accepted = {
        "patientid",
        "patientidentifier",
        "subjectid",
        "subjectidentifier",
        "personid",
        "personidentifier",
    }

    source_tables = [
        ("reports", reports),
        ("projections", projections),
    ]

    matches = []
    for source_name, source_df in source_tables:
        if "uid" not in source_df.columns:
            continue
        for col in source_df.columns:
            if _audit_normalize_identifier_column(col) in accepted:
                matches.append((source_name, col, source_df))

    if len(matches) == 0:
        return {
            "status": "NOT_AVAILABLE_IN_SOURCE_METADATA",
            "source": None,
            "column": None,
            "mapping": None,
        }

    # Prefer the first explicit exact match. If more than one exists, verify
    # each candidate internally and use the first complete mapping.
    for source_name, col, source_df in matches:
        tmp = source_df[["uid", col]].copy()
        tmp = tmp.dropna(subset=["uid", col])
        tmp["uid"] = tmp["uid"].astype(str).str.strip()
        tmp[col] = tmp[col].astype(str).str.strip()
        tmp = tmp[(tmp["uid"] != "") & (tmp[col] != "")]

        grouped = tmp.groupby("uid")[col].agg(
            lambda x: sorted(set(v for v in x if v))
        )

        ambiguous = grouped[grouped.apply(len) > 1]
        if len(ambiguous) > 0:
            raise RuntimeError(
                f"Patient-ID metadata inconsistency: {len(ambiguous)} UID(s) "
                f"map to multiple patient identifiers in {source_name}.{col}."
            )

        mapping = {
            str(uid): values[0]
            for uid, values in grouped.items()
            if len(values) == 1
        }

        master_uids = set(df["uid"].astype(str))
        coverage = sum(uid in mapping for uid in master_uids) / max(len(master_uids), 1)

        if coverage == 1.0:
            return {
                "status": "AVAILABLE",
                "source": source_name,
                "column": col,
                "mapping": mapping,
            }

    # Explicit identifier column exists, but it does not completely cover
    # the usable study population. Do not silently call this patient-wise.
    raise RuntimeError(
        "An explicit patient-ID column was detected in the source metadata, "
        "but it does not provide complete patient-ID coverage for all usable UIDs. "
        "Patient-level leakage therefore cannot be verified safely."
    )


def run_data_split_leakage_audit(train_df, val_df, test_df, master_df):
    print("=" * 88)
    print("PUBLICATION-GRADE DATA SPLIT LEAKAGE AUDIT")
    print("=" * 88)

    required_columns = {"uid", "img1", "img2"}
    for split_name, frame in [
        ("Train", train_df),
        ("Validation", val_df),
        ("Test", test_df),
    ]:
        missing = required_columns - set(frame.columns)
        if missing:
            raise RuntimeError(
                f"{split_name} split is missing required audit columns: {sorted(missing)}"
            )

    # --------------------------------------------------------
    # 1) Study/report UID disjointness
    # --------------------------------------------------------
    train_uid = _audit_as_clean_string_set(train_df["uid"])
    val_uid = _audit_as_clean_string_set(val_df["uid"])
    test_uid = _audit_as_clean_string_set(test_df["uid"])

    uid_overlaps = {
        "train_validation": len(train_uid & val_uid),
        "train_test": len(train_uid & test_uid),
        "validation_test": len(val_uid & test_uid),
    }

    print("\n[1] UID overlap")
    print("Train ∩ Validation:", uid_overlaps["train_validation"])
    print("Train ∩ Test      :", uid_overlaps["train_test"])
    print("Validation ∩ Test :", uid_overlaps["validation_test"])

    if any(v != 0 for v in uid_overlaps.values()):
        raise RuntimeError(
            f"DATA LEAKAGE DETECTED: non-zero UID overlap: {uid_overlaps}"
        )

    # --------------------------------------------------------
    # 2) Image-file disjointness
    # --------------------------------------------------------
    train_images = _audit_image_set(train_df)
    val_images = _audit_image_set(val_df)
    test_images = _audit_image_set(test_df)

    image_overlaps = {
        "train_validation": len(train_images & val_images),
        "train_test": len(train_images & test_images),
        "validation_test": len(val_images & test_images),
    }

    print("\n[2] Image-file overlap")
    print("Train ∩ Validation:", image_overlaps["train_validation"])
    print("Train ∩ Test      :", image_overlaps["train_test"])
    print("Validation ∩ Test :", image_overlaps["validation_test"])

    if any(v != 0 for v in image_overlaps.values()):
        raise RuntimeError(
            f"DATA LEAKAGE DETECTED: non-zero image-file overlap: {image_overlaps}"
        )

    # --------------------------------------------------------
    # 3) Master-table uniqueness and exact accounting
    # --------------------------------------------------------
    duplicated_uid_rows = int(master_df["uid"].astype(str).duplicated().sum())
    if duplicated_uid_rows != 0:
        raise RuntimeError(
            f"Master study-level dataframe contains {duplicated_uid_rows} duplicated UID row(s)."
        )

    total_split_rows = len(train_df) + len(val_df) + len(test_df)
    if total_split_rows != len(master_df):
        raise RuntimeError(
            f"Split accounting mismatch: train+val+test={total_split_rows}, "
            f"master={len(master_df)}."
        )

    union_uids = train_uid | val_uid | test_uid
    master_uids = _audit_as_clean_string_set(master_df["uid"])
    if union_uids != master_uids:
        missing = sorted(master_uids - union_uids)
        extra = sorted(union_uids - master_uids)
        raise RuntimeError(
            "UID accounting mismatch. "
            f"Missing from splits={missing[:10]}, Extra in splits={extra[:10]}"
        )

    print("\n[3] Split statistics")
    print(f"Train      : {len(train_df):5d} ({len(train_df)/len(master_df):.2%})")
    print(f"Validation : {len(val_df):5d} ({len(val_df)/len(master_df):.2%})")
    print(f"Test       : {len(test_df):5d} ({len(test_df)/len(master_df):.2%})")
    print(f"Total      : {len(master_df):5d}")
    print("Duplicated UID rows in master dataframe:", duplicated_uid_rows)

    # --------------------------------------------------------
    # 4) Optional patient-level audit from explicit source metadata
    # --------------------------------------------------------
    patient_info = _audit_find_patient_mapping()
    patient_status = patient_info["status"]
    patient_source = patient_info["source"]
    patient_column = patient_info["column"]
    patient_overlaps = None

    if patient_status == "AVAILABLE":
        mapping = patient_info["mapping"]

        def patient_ids_for(uid_set):
            return {mapping[str(uid)] for uid in uid_set}

        train_patients = patient_ids_for(train_uid)
        val_patients = patient_ids_for(val_uid)
        test_patients = patient_ids_for(test_uid)

        patient_overlaps = {
            "train_validation": len(train_patients & val_patients),
            "train_test": len(train_patients & test_patients),
            "validation_test": len(val_patients & test_patients),
        }

        print("\n[4] Patient-level overlap")
        print(f"Detected patient identifier: {patient_source}.{patient_column}")
        print("Train ∩ Validation:", patient_overlaps["train_validation"])
        print("Train ∩ Test      :", patient_overlaps["train_test"])
        print("Validation ∩ Test :", patient_overlaps["validation_test"])

        if any(v != 0 for v in patient_overlaps.values()):
            raise RuntimeError(
                "PATIENT-LEVEL DATA LEAKAGE DETECTED: "
                f"{patient_overlaps}"
            )

        patient_audit_status = "PASS"
    else:
        print("\n[4] Patient-level audit")
        print(
            "No explicit patient identifier was found in the source CSV metadata."
        )
        print(
            "Therefore, this experiment will NOT claim patient-wise splitting."
        )
        print(
            "Supported claim: mutually exclusive study/report UIDs and image files."
        )
        patient_audit_status = "NOT_AVAILABLE_IN_SOURCE_METADATA"

    # --------------------------------------------------------
    # 5) Deterministic split fingerprints
    # --------------------------------------------------------
    train_hash = _audit_sha256_of_values(train_uid)
    val_hash = _audit_sha256_of_values(val_uid)
    test_hash = _audit_sha256_of_values(test_uid)

    combined_payload = (
        f"DATA_SPLIT_SEED={DATA_SPLIT_SEED}\n"
        f"TRAIN={train_hash}\n"
        f"VALIDATION={val_hash}\n"
        f"TEST={test_hash}\n"
    ).encode("utf-8")
    split_fingerprint = _audit_hashlib.sha256(combined_payload).hexdigest()

    print("\n[5] Reproducibility fingerprint")
    print("DATA_SPLIT_SEED :", DATA_SPLIT_SEED)
    print("Split fingerprint:", split_fingerprint)

    # --------------------------------------------------------
    # 6) Persist membership + machine-readable audit report
    # --------------------------------------------------------
    membership_parts = []
    patient_mapping = patient_info.get("mapping")

    for partition, frame in [
        ("Train", train_df),
        ("Validation", val_df),
        ("Test", test_df),
    ]:
        part = frame[["uid", "img1", "img2"]].copy()
        part.insert(0, "Partition", partition)

        if patient_mapping is not None:
            part["Patient_ID"] = part["uid"].astype(str).map(patient_mapping)

        membership_parts.append(part)

    membership_df = pd.concat(membership_parts, ignore_index=True)
    membership_df.to_csv(_AUDIT_MEMBERSHIP_PATH, index=False)

    report = {
        "overall_status": "PASS",
        "run_seed": int(RUN_SEED),
        "experiment_seeds": [int(x) for x in EXPERIMENT_SEEDS],
        "data_split_seed": int(DATA_SPLIT_SEED),
        "split_level_supported_by_code": "study_report_uid",
        "train_rows": int(len(train_df)),
        "validation_rows": int(len(val_df)),
        "test_rows": int(len(test_df)),
        "total_rows": int(len(master_df)),
        "master_duplicated_uid_rows": int(duplicated_uid_rows),
        "uid_overlap": uid_overlaps,
        "image_file_overlap": image_overlaps,
        "patient_audit_status": patient_audit_status,
        "patient_identifier_source": patient_source,
        "patient_identifier_column": patient_column,
        "patient_overlap": patient_overlaps,
        "train_uid_sha256": train_hash,
        "validation_uid_sha256": val_hash,
        "test_uid_sha256": test_hash,
        "split_fingerprint_sha256": split_fingerprint,
        "membership_csv": str(_AUDIT_MEMBERSHIP_PATH),
        "interpretation": (
            "PASS confirms zero study/report UID overlap and zero image-file overlap "
            "across Train/Validation/Test. Patient-wise disjointness is claimed only "
            "when patient_audit_status is PASS."
        ),
    }

    with open(_AUDIT_JSON_PATH, "w", encoding="utf-8") as f:
        _audit_json.dump(report, f, indent=2, ensure_ascii=False)

    print("\n" + "=" * 88)
    print("PASS: DATA SPLIT LEAKAGE AUDIT COMPLETED SUCCESSFULLY")
    print("=" * 88)
    print("Audit JSON      :", _AUDIT_JSON_PATH)
    print("Membership CSV  :", _AUDIT_MEMBERSHIP_PATH)

    return report


SPLIT_LEAKAGE_AUDIT = run_data_split_leakage_audit(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    master_df=df,
)

assert SPLIT_LEAKAGE_AUDIT["overall_status"] == "PASS"


In [ ]:
# =========================================================
# 7) DenseNet-121 Feature Cache + Training-Only Class Weights
# =========================================================
# train_df, val_df and test_df were created in Cell 6 BEFORE vocabulary creation.
# =========================================================

# Class-imbalance weights are estimated from federated training data only.
label_stack = np.vstack(train_df["labels"].values).astype(np.float32)
pos = label_stack.sum(axis=0)
neg = label_stack.shape[0] - pos
LABEL_POS_WEIGHT = torch.tensor(
    neg / (pos + 1e-6), dtype=torch.float32
).clamp(1.0, 8.0).to(DEVICE)
print("LABEL_POS_WEIGHT ready:", LABEL_POS_WEIGHT.shape)

feature_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


@torch.no_grad()
def extract_features_for_df(df_input):
    encoder = models.densenet121(
        weights=models.DenseNet121_Weights.DEFAULT
    ).to(DEVICE)
    encoder.eval()

    features = []
    for _, row in tqdm(
        df_input.iterrows(),
        total=len(df_input),
        desc="Extracting DenseNet121 features",
    ):
        img1 = feature_tfms(Image.open(row["img1"]).convert("RGB")).unsqueeze(0).to(DEVICE)
        img2 = feature_tfms(Image.open(row["img2"]).convert("RGB")).unsqueeze(0).to(DEVICE)

        f1 = encoder.features(img1)
        f1 = F.relu(f1, inplace=False)
        f1 = F.adaptive_avg_pool2d(f1, (1, 1)).flatten(1)

        f2 = encoder.features(img2)
        f2 = F.relu(f2, inplace=False)
        f2 = F.adaptive_avg_pool2d(f2, (1, 1)).flatten(1)

        features.append(torch.cat([f1, f2], dim=1).cpu().numpy()[0])

    del encoder
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return np.asarray(features, dtype=np.float32)


def cache_matches(cache):
    required = ["train_features", "val_features", "test_features", "train_uids", "val_uids", "test_uids"]
    if not all(k in cache.files for k in required):
        return False
    return (
        cache["train_uids"].astype(str).tolist() == train_df["uid"].astype(str).tolist()
        and cache["val_uids"].astype(str).tolist() == val_df["uid"].astype(str).tolist()
        and cache["test_uids"].astype(str).tolist() == test_df["uid"].astype(str).tolist()
    )


load_ok = False
if os.path.exists(FEATURE_CACHE):
    try:
        cache = np.load(FEATURE_CACHE, allow_pickle=True)
        load_ok = cache_matches(cache)
    except Exception:
        load_ok = False

if load_ok:
    train_features = cache["train_features"]
    val_features = cache["val_features"]
    test_features = cache["test_features"]
    print("Loaded validated feature cache:", FEATURE_CACHE)
else:
    train_features = extract_features_for_df(train_df)
    val_features = extract_features_for_df(val_df)
    test_features = extract_features_for_df(test_df)
    np.savez(
        FEATURE_CACHE,
        train_features=train_features,
        val_features=val_features,
        test_features=test_features,
        train_uids=train_df["uid"].astype(str).to_numpy(),
        val_uids=val_df["uid"].astype(str).to_numpy(),
        test_uids=test_df["uid"].astype(str).to_numpy(),
    )
    print("Saved new feature cache:", FEATURE_CACHE)

print("Feature shapes:", train_features.shape, val_features.shape, test_features.shape)


In [ ]:
# =========================================================
# 8) Dataset + Reproducible Dirichlet Non-IID Clients
# =========================================================

class IndianaFeatureDataset(Dataset):
    def __init__(self, df, features):
        self.df = df.reset_index(drop=True)
        self.features = np.asarray(features, dtype=np.float32)
        assert len(self.df) == len(self.features)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        feat = torch.tensor(self.features[idx], dtype=torch.float32)
        labels = torch.tensor(row["labels"], dtype=torch.float32)
        tokens = torch.tensor(encode_text(row["report"]), dtype=torch.long)
        return feat, labels, tokens


train_dataset = IndianaFeatureDataset(train_df, train_features)
val_dataset = IndianaFeatureDataset(val_df, val_features)
test_dataset = IndianaFeatureDataset(test_df, test_features)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

NONIID_ALPHA = 0.70
MIN_CLIENT_SIZE = 15
MAX_RETRIES = 200


def get_dominant_partition_labels(frame):
    label_matrix = np.vstack(frame["labels"].values).astype(np.float32)
    prevalence = label_matrix.mean(axis=0)
    rarity = 1.0 / (prevalence + 1e-6)
    dominant = []
    for y in label_matrix:
        positive = np.flatnonzero(y > 0)
        if len(positive) == 0:
            dominant.append(label_matrix.shape[1])
        else:
            dominant.append(int(positive[np.argmax(rarity[positive])]))
    return np.asarray(dominant, dtype=np.int64)


def build_dirichlet_noniid_partition(
    frame,
    num_clients,
    alpha=0.70,
    min_client_size=15,
    seed=42,
    max_retries=200,
):
    dominant_labels = get_dominant_partition_labels(frame)
    unique_classes = np.unique(dominant_labels)
    best_partition, best_min = None, -1

    for attempt in range(max_retries):
        rng = np.random.default_rng(int(seed) + attempt)
        client_bins = [[] for _ in range(num_clients)]

        for cls in unique_classes:
            cls_indices = np.where(dominant_labels == cls)[0]
            rng.shuffle(cls_indices)
            proportions = rng.dirichlet(np.repeat(alpha, num_clients))
            split_points = (np.cumsum(proportions)[:-1] * len(cls_indices)).astype(int)
            for cid, split in enumerate(np.split(cls_indices, split_points)):
                client_bins[cid].extend(split.tolist())

        partition = []
        for items in client_bins:
            arr = np.asarray(items, dtype=np.int64)
            rng.shuffle(arr)
            partition.append(arr)

        sizes = np.asarray([len(x) for x in partition])
        current_min = int(sizes.min())
        if current_min > best_min:
            best_partition, best_min = partition, current_min
        if current_min >= min_client_size:
            return partition

    print("Warning: preferred minimum client size was not reached.")
    print("Using best partition with minimum size:", best_min)
    return best_partition


def get_client_loader(cid, round_id=1):
    subset = Subset(train_dataset, client_indices[cid])
    generator = torch.Generator()
    generator.manual_seed(int(SEED) + 10007 * int(cid) + 101 * int(round_id))
    return DataLoader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )


def get_car_public_loader(round_id=1):
    subset = Subset(train_dataset, CAR_PUBLIC_INDICES)
    generator = torch.Generator()
    generator.manual_seed(int(SEED) + 88001 + 101 * int(round_id))
    return DataLoader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
    )


def rebuild_federated_partition(experiment_seed, verbose=True):
    """Rebuild all training-only client structures for one experiment seed.

    The held-out validation/test studies remain fixed. Only the federated
    stochastic structure changes across seeds. This gives repeated independent
    FL runs without changing the final test population.
    """
    global client_indices, client_sizes
    global CAR_PUBLIC_BANK_SEED, CAR_PUBLIC_INDICES, CAR_PUBLIC_INDEX_SET
    global CAR_PRIVATE_CLIENT_INDICES, CAR_PUBLIC_FEATURES, CAR_PUBLIC_LABELS

    experiment_seed = int(experiment_seed)
    client_indices = build_dirichlet_noniid_partition(
        train_df,
        num_clients=NUM_CLIENTS,
        alpha=NONIID_ALPHA,
        min_client_size=MIN_CLIENT_SIZE,
        seed=experiment_seed,
        max_retries=MAX_RETRIES,
    )

    client_sizes = np.asarray([len(x) for x in client_indices])
    assert len(client_indices) == NUM_CLIENTS
    assert sum(len(x) for x in client_indices) == len(train_dataset)
    all_assigned = np.concatenate(client_indices)
    assert len(np.unique(all_assigned)) == len(train_dataset)

    # CAR-MFL public retrieval bank is rebuilt from TRAINING data only for each
    # seed. It stays disjoint from CAR private-client data and never uses val/test.
    CAR_PUBLIC_BANK_SEED = experiment_seed + 2407
    target_public = int(round(CAR_PUBLIC_BANK_FRACTION * len(train_dataset)))
    target_public = max(CAR_PUBLIC_BANK_MIN_SIZE, target_public)
    target_public = min(CAR_PUBLIC_BANK_MAX_SIZE, target_public)

    car_rng = np.random.default_rng(CAR_PUBLIC_BANK_SEED)
    candidate_public = []
    for idxs in client_indices:
        idxs = np.asarray(idxs, dtype=np.int64).copy()
        car_rng.shuffle(idxs)
        removable = idxs[MIN_CLIENT_SIZE:]
        candidate_public.extend(removable.tolist())

    candidate_public = np.asarray(candidate_public, dtype=np.int64)
    car_rng.shuffle(candidate_public)
    if len(candidate_public) < target_public:
        raise RuntimeError(
            f"CAR public-bank target {target_public} exceeds removable training "
            f"samples ({len(candidate_public)} available)."
        )

    CAR_PUBLIC_INDICES = np.sort(candidate_public[:target_public])
    CAR_PUBLIC_INDEX_SET = set(CAR_PUBLIC_INDICES.tolist())
    CAR_PRIVATE_CLIENT_INDICES = [
        np.asarray(
            [idx for idx in idxs if int(idx) not in CAR_PUBLIC_INDEX_SET],
            dtype=np.int64,
        )
        for idxs in client_indices
    ]

    assert len(CAR_PUBLIC_INDICES) == len(np.unique(CAR_PUBLIC_INDICES))
    assert all(len(idxs) >= MIN_CLIENT_SIZE for idxs in CAR_PRIVATE_CLIENT_INDICES)
    assert all(
        CAR_PUBLIC_INDEX_SET.isdisjoint(set(map(int, idxs.tolist())))
        for idxs in CAR_PRIVATE_CLIENT_INDICES
    )

    CAR_PUBLIC_FEATURES = np.asarray(
        train_features[CAR_PUBLIC_INDICES], dtype=np.float32
    )
    CAR_PUBLIC_LABELS = np.vstack(
        train_df.iloc[CAR_PUBLIC_INDICES]["labels"].values
    ).astype(np.float32)

    if verbose:
        print("\nFederated partition rebuilt")
        print("Experiment seed:", experiment_seed)
        print("Clients:", NUM_CLIENTS)
        print("Total samples:", int(client_sizes.sum()))
        print("Min / Max:", int(client_sizes.min()), "/", int(client_sizes.max()))
        print("Mean ± SD:", round(client_sizes.mean(), 2), "±", round(client_sizes.std(ddof=1), 2))
        print("Dirichlet alpha:", NONIID_ALPHA)
        print("CAR public bank size:", len(CAR_PUBLIC_INDICES))
        print("Leakage audit: PASS (training-only federated structures)")

    return client_indices


# Initial state; the multi-seed runner will rebuild this once per seed.
rebuild_federated_partition(SEED, verbose=True)

client_distribution_table = pd.DataFrame({
    "Client_ID": np.arange(NUM_CLIENTS),
    "Num_Samples": client_sizes,
})
display(client_distribution_table)


## Academic communication-efficiency policy

The proposed protocol uses client-side model caching, FP16 initial synchronization, INT8 cached model-delta downlinks, INT8 client-update uplinks, and error feedback. Each client trains from the reconstructed model it actually receives. Payload size is measured from encoded tensor values, scales, and metadata. No result, communication cost, or evaluation metric is multiplied by a manually selected improvement factor.


## Adaptive Top-K communication protocol

For the proposed method, each client computes its local model difference relative to the model it received. The previous compression residual is added to this difference. The code then selects the smallest ratio from `[0.15, 0.20, 0.25, 0.30, 0.40]` that preserves at least 95% of the squared L2 energy of each update tensor. A conservative 0.40 ratio is used during the first two rounds. Only the selected values are quantized to INT8 and transmitted with a bitmap identifying their positions. The server reconstructs the sparse update before hierarchical aggregation. The untransmitted and quantization errors are retained as error feedback for the client's next participation.

This mechanism is applied before aggregation and therefore may genuinely affect model performance. It is not a post-processing adjustment of communication cost or BLEU scores.


In [ ]:
import copy
import gc


def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# =========================================================
# Component-wise ablation configuration
# =========================================================
FULL_PROPOSED_COMPONENTS = {
    "variant": "Full_Proposed",
    "adaptive_client_selection": True,
    "hierarchical_aggregation": True,
    "adaptive_topk": True,
    "int8_quantization": True,
    "error_feedback": True,
    "missing_reconstruction": True,
}

def normalize_ablation_config(config=None):
    out = copy.deepcopy(FULL_PROPOSED_COMPONENTS)
    if config is not None:
        out.update(config)
    return out

def _component_enabled(model, key):
    cfg = getattr(model, "ablation_config", FULL_PROPOSED_COMPONENTS)
    return bool(cfg.get(key, True))


# =========================================================
# 9) Models
# =========================================================

class ReportGenerator(nn.Module):
    def __init__(self, vocab_size, d_model=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos = nn.Parameter(torch.randn(1, MAX_LEN, d_model) * 0.02)
        layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=8,
            dim_feedforward=1024,
            dropout=0.10,
            batch_first=True,
        )
        self.decoder = nn.TransformerDecoder(layer, num_layers=2)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, memory, tokens):
        x = self.embed(tokens) + self.pos[:, :tokens.size(1), :]
        memory = memory.unsqueeze(1)
        length = tokens.size(1)
        causal_mask = torch.triu(
            torch.ones(length, length, device=tokens.device), diagonal=1
        ).bool()
        decoded = self.decoder(x, memory, tgt_mask=causal_mask)
        return self.out(decoded)


class MMFLModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.visual_proj = nn.Sequential(
            nn.Linear(2048, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.10),
        )
        self.classifier = nn.Linear(512, NUM_LABELS)
        self.label_proj = nn.Sequential(
            nn.Linear(NUM_LABELS, 512),
            nn.LayerNorm(512),
            nn.GELU(),
        )
        self.generator = ReportGenerator(VOCAB_SIZE, d_model=512)

    def encode_memory(self, feat):
        z = self.visual_proj(feat)
        label_logits = self.classifier(z)
        label_context = self.label_proj(torch.sigmoid(label_logits))
        return z + label_context, label_logits

    def forward(self, feat, tokens):
        memory, label_logits = self.encode_memory(feat)
        text_logits = self.generator(memory, tokens[:, :-1])
        return label_logits, text_logits


class CAR_MFL_Retrieval_Model(MMFLModel):
    """Source-grounded CAR-MFL retrieval baseline adapted to paired X-ray views.

    Poudel et al. (MICCAI 2024) do not learn a generator for a missing modality.
    They retrieve within the AVAILABLE modality, refine the top-k set by label
    Jaccard similarity, and use the associated complementary public modality.
    Retrieval is performed at the data/feature level before local optimization.
    """
    def __init__(self):
        super().__init__()

    def retrieval_embedding(self, view_feat, view_id):
        """Dynamic intra-modal representation from the current global model."""
        if view_feat.ndim != 2 or view_feat.size(1) != 1024:
            raise ValueError("CAR retrieval expects [batch, 1024] view features.")
        if int(view_id) not in (0, 1):
            raise ValueError("view_id must be 0 (frontal) or 1 (lateral).")
        linear = self.visual_proj[0]
        layer_norm = self.visual_proj[1]
        gelu = self.visual_proj[2]
        col = slice(0, 1024) if int(view_id) == 0 else slice(1024, 2048)
        bias = None if linear.bias is None else 0.5 * linear.bias
        z = F.linear(view_feat, linear.weight[:, col], bias)
        z = layer_norm(z)
        z = gelu(z)
        return F.normalize(z, p=2, dim=1, eps=1e-12)


class CrossViewReconstructionBackbone(MMFLModel):
    """Previous learned reconstruction backbone, retained ONLY for Proposed."""
    def __init__(self):
        super().__init__()
        self.img1_to_img2 = nn.Sequential(
            nn.Linear(1024, 512), nn.GELU(), nn.Dropout(0.05), nn.Linear(512, 1024)
        )
        self.img2_to_img1 = nn.Sequential(
            nn.Linear(1024, 512), nn.GELU(), nn.Dropout(0.05), nn.Linear(512, 1024)
        )
        self.car_fusion_gate = nn.Sequential(
            nn.Linear(2048, 512), nn.GELU(), nn.Linear(512, 2048), nn.Sigmoid()
        )

    def repair_missing_views(self, feat):
        if not _component_enabled(self, "missing_reconstruction"):
            return feat
        img1, img2 = feat[:, :1024], feat[:, 1024:]
        miss1 = (img1.abs().sum(dim=1, keepdim=True) < 1e-6).float()
        miss2 = (img2.abs().sum(dim=1, keepdim=True) < 1e-6).float()
        img1_hat = self.img2_to_img1(img2)
        img2_hat = self.img1_to_img2(img1)
        repaired1 = img1 * (1.0 - miss1) + img1_hat * miss1
        repaired2 = img2 * (1.0 - miss2) + img2_hat * miss2
        repaired = torch.cat([repaired1, repaired2], dim=1)
        gate = self.car_fusion_gate(repaired)
        return gate * repaired + (1.0 - gate) * feat

    def imputation_regularization(self, full_feat):
        if not _component_enabled(self, "missing_reconstruction"):
            return full_feat.new_tensor(0.0)
        half = full_feat.size(1) // 2
        masked = full_feat.clone()
        drop_second = torch.rand(full_feat.size(0), device=full_feat.device) < 0.5
        masked[drop_second, half:] = 0.0
        masked[~drop_second, :half] = 0.0
        repaired = self.repair_missing_views(masked)
        losses = []
        if drop_second.any():
            losses.append(F.smooth_l1_loss(
                repaired[drop_second, half:], full_feat[drop_second, half:]
            ))
        if (~drop_second).any():
            losses.append(F.smooth_l1_loss(
                repaired[~drop_second, :half], full_feat[~drop_second, :half]
            ))
        return torch.stack(losses).mean() if losses else full_feat.new_tensor(0.0)

    def encode_memory(self, feat):
        return super().encode_memory(self.repair_missing_views(feat))


class Proposed_CoHiM_FL_Model(CrossViewReconstructionBackbone):
    """Identity-preserving trainable residual fusion; unchanged by CAR correction."""
    def __init__(self):
        super().__init__()
        self.ablation_config = normalize_ablation_config()
        self.view_norm1 = nn.LayerNorm(1024)
        self.view_norm2 = nn.LayerNorm(1024)
        self.cross_gate = nn.Sequential(
            nn.Linear(2048, 512), nn.GELU(), nn.Dropout(0.05),
            nn.Linear(512, 1024), nn.Sigmoid(),
        )
        self.fusion_refine = nn.Sequential(
            nn.Linear(2048, 1024), nn.LayerNorm(1024), nn.GELU(),
            nn.Dropout(0.05), nn.Linear(1024, 2048),
        )
        self.fusion_strength = nn.Parameter(torch.tensor(0.0))

    def enhanced_fusion(self, feat):
        repaired = self.repair_missing_views(feat)
        img1, img2 = repaired[:, :1024], repaired[:, 1024:]
        n1, n2 = self.view_norm1(img1), self.view_norm2(img2)
        context = torch.cat([n1, n2], dim=1)
        gate = self.cross_gate(context)
        residual = self.fusion_refine(context)
        residual_gate = torch.cat([gate, 1.0 - gate], dim=1)
        strength = 0.15 * torch.tanh(self.fusion_strength)
        return repaired + strength * residual_gate * residual

    def consistency_regularization(self, full_feat, corrupted_feat):
        with torch.no_grad():
            full_fused = F.normalize(self.enhanced_fusion(full_feat), dim=1)
        corrupted_fused = F.normalize(self.enhanced_fusion(corrupted_feat), dim=1)
        return F.smooth_l1_loss(corrupted_fused, full_fused)

    def encode_memory(self, feat):
        fused = self.enhanced_fusion(feat)
        return MMFLModel.encode_memory(self, fused)


def create_model(method):
    if method == "CAR-MFL":
        model = CAR_MFL_Retrieval_Model()
    elif method == "Proposed_CoHiM_FL":
        model = Proposed_CoHiM_FL_Model()
    else:
        model = MMFLModel()
    model.method_name = method
    return model


def load_matching_state(target, source_state):
    target_state = target.state_dict()
    for key, value in source_state.items():
        if key in target_state and target_state[key].shape == value.shape:
            target_state[key] = value.detach().clone()
    target.load_state_dict(target_state)
    return target


# Shared initialization eliminates random-initialization bias across methods.
def rebuild_model_registry(experiment_seed):
    """Create fresh, seed-controlled initial models for one independent run."""
    global base_template, base_state, car_template
    global proposed_backbone_template, proposed_backbone_state, proposed_template
    global MODEL_REGISTRY, PROPOSED_BACKBONE_TEMPLATE

    experiment_seed = int(experiment_seed)
    torch.manual_seed(experiment_seed)
    base_template = MMFLModel().cpu()
    base_state = copy.deepcopy(base_template.state_dict())

    car_template = CAR_MFL_Retrieval_Model().cpu()
    load_matching_state(car_template, base_state)

    torch.manual_seed(experiment_seed + 1)
    proposed_backbone_template = CrossViewReconstructionBackbone().cpu()
    load_matching_state(proposed_backbone_template, base_state)
    proposed_backbone_state = copy.deepcopy(proposed_backbone_template.state_dict())

    torch.manual_seed(experiment_seed + 2)
    proposed_template = Proposed_CoHiM_FL_Model().cpu()
    load_matching_state(proposed_template, proposed_backbone_state)
    proposed_template.ablation_config = normalize_ablation_config()

    MODEL_REGISTRY = {
        "FedAvg": load_matching_state(MMFLModel().cpu(), base_state),
        "FedProx": load_matching_state(MMFLModel().cpu(), base_state),
        "Fedsw": load_matching_state(MMFLModel().cpu(), base_state),
        "CAR-MFL": car_template,
        "Proposed_CoHiM_FL": proposed_template,
    }
    for method, model in MODEL_REGISTRY.items():
        model.method_name = method

    PROPOSED_BACKBONE_TEMPLATE = proposed_backbone_template
    PROPOSED_BACKBONE_TEMPLATE.method_name = "Proposed_Backbone_Audit_Only"
    return MODEL_REGISTRY


rebuild_model_registry(SEED)

def clone_model(model, device=DEVICE):
    new_model = model.__class__()
    new_model.load_state_dict(copy.deepcopy(model.state_dict()))
    new_model.method_name = getattr(model, "method_name", model.__class__.__name__)
    if hasattr(model, "ablation_config"):
        new_model.ablation_config = copy.deepcopy(model.ablation_config)
    return new_model.to(device)


# =========================================================
# 10) Training and evaluation
# =========================================================

def current_round_lr(round_id):
    if ROUNDS <= 1:
        return LR
    progress = (round_id - 1) / (ROUNDS - 1)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return LR * (MIN_LR_FACTOR + (1.0 - MIN_LR_FACTOR) * cosine)


def proposed_missing_rate(round_id):
    if ROUNDS <= 1:
        return PROPOSED_MISSING_RATE_END
    progress = (round_id - 1) / (ROUNDS - 1)
    if progress <= PROPOSED_MISSING_WARMUP_FRACTION:
        return PROPOSED_MISSING_RATE_START
    scaled = (
        (progress - PROPOSED_MISSING_WARMUP_FRACTION)
        / max(1.0 - PROPOSED_MISSING_WARMUP_FRACTION, 1e-12)
    )
    return PROPOSED_MISSING_RATE_START + scaled * (
        PROPOSED_MISSING_RATE_END - PROPOSED_MISSING_RATE_START
    )


def apply_training_missing_augmentation(feat, missing_rate=0.20):
    if missing_rate <= 0:
        return feat
    out = feat.clone()
    half = out.size(1) // 2
    missing_patient = torch.rand(out.size(0), device=out.device) < float(missing_rate)
    drop_lateral = torch.rand(out.size(0), device=out.device) < 0.70
    out[missing_patient & (~drop_lateral), :half] = 0.0
    out[missing_patient & drop_lateral, half:] = 0.0
    return out


def _hash_uniform(index_tensor, salt=0, seed=None):
    if seed is None:
        seed = SEED
    x = index_tensor.to(torch.int64) + int(seed) * 1000003 + int(salt) * 9176
    x = (x ^ (x >> 16)) * 2246822519
    x = (x ^ (x >> 13)) * 3266489917
    x = x ^ (x >> 16)
    return (torch.remainder(x.abs(), 1000000).float() / 1000000.0).to(index_tensor.device)


_EXACT_MISSING_RANK_CACHE = {}


def _exact_missing_rank_vector(population_size, seed):
    """Deterministic global ranking used for exact, nested missing-study masks."""
    population_size = int(population_size)
    seed = int(DATA_SPLIT_SEED if seed is None else seed)
    if population_size <= 0:
        raise ValueError("population_size must be positive")
    key = (population_size, seed)
    if key not in _EXACT_MISSING_RANK_CACHE:
        universe = torch.arange(population_size, dtype=torch.long)
        scores = _hash_uniform(universe, salt=73, seed=seed).cpu().numpy()
        indices = np.arange(population_size, dtype=np.int64)
        order = np.lexsort((indices, scores))  # deterministic tie-break by sample index
        ranks = np.empty(population_size, dtype=np.int64)
        ranks[order] = np.arange(population_size, dtype=np.int64)
        _EXACT_MISSING_RANK_CACHE[key] = ranks
    return _EXACT_MISSING_RANK_CACHE[key]


def apply_deterministic_missing_corruption(
    feat,
    missing_rate=0.0,
    sample_indices=None,
    seed=None,
    view="lateral",
    return_audit=False,
    population_size=None,
):
    """Apply an exact-count, deterministic, nested sample-level missing-view mask.

    For a population of N held-out studies and requested rate r, exactly
    round(r*N) studies are masked (the nearest achievable integer count).
    The ranking is fixed by sample index and seed, so the 30% set is a subset of
    the 60% set, which is a subset of the 90% set. Every method receives the
    identical mask for a given rate/protocol.
    """
    out = feat.clone()
    half = out.size(1) // 2

    if sample_indices is None:
        sample_indices = torch.arange(out.size(0), device=out.device)
    else:
        sample_indices = sample_indices.to(out.device).long()

    if population_size is None:
        population_size = int(sample_indices.max().item()) + 1 if len(sample_indices) else int(out.size(0))
    population_size = int(population_size)
    if len(sample_indices) and int(sample_indices.max().item()) >= population_size:
        raise ValueError("sample_indices exceed the declared evaluation population_size")

    requested_rate = float(missing_rate)
    target_count = int(round(requested_rate * population_size))
    target_count = max(0, min(population_size, target_count))
    exact_rate = float(target_count / population_size) if population_size else 0.0

    if target_count > 0:
        rank_vector = _exact_missing_rank_vector(population_size, seed)
        batch_ranks = torch.as_tensor(
            rank_vector[sample_indices.detach().cpu().numpy()],
            dtype=torch.long,
            device=out.device,
        )
        missing_patient = batch_ranks < target_count
    else:
        missing_patient = torch.zeros(out.size(0), dtype=torch.bool, device=out.device)

    original_norm = feat.abs().sum(dim=1)

    if view == "frontal":
        out[missing_patient, :half] = 0.0
    elif view == "lateral":
        out[missing_patient, half:] = 0.0
    elif view == "alternate":
        choose_frontal = _hash_uniform(sample_indices, salt=91, seed=seed) < 0.5
        out[missing_patient & choose_frontal, :half] = 0.0
        out[missing_patient & (~choose_frontal), half:] = 0.0
    else:
        raise ValueError(f"Unknown missing view: {view}")

    if not return_audit:
        return out

    corrupted_norm = out.abs().sum(dim=1)
    audit = {
        "mask": missing_patient.detach().cpu(),
        "sample_indices": sample_indices.detach().cpu(),
        "masked_count": int(missing_patient.sum().item()),
        "total_count": int(out.size(0)),
        "requested_rate": requested_rate,
        "target_masked_count_population": int(target_count),
        "population_size": int(population_size),
        "nearest_achievable_rate": exact_rate,
        "original_abs_sum": float(original_norm.sum().item()),
        "corrupted_abs_sum": float(corrupted_norm.sum().item()),
    }
    return out, audit


MISSING_VIEW_PROTOCOLS = ["lateral", "frontal", "alternate"]
MISSING_VIEW_FOR_EVAL = "lateral"



# =========================================================
# Source-grounded CAR-MFL retrieval augmentation (CAR path only)
# =========================================================
CAR_RETRIEVAL_AUDIT_ROWS = []


class CARAugmentedDataset(Dataset):
    def __init__(self, global_indices, augmented_features):
        self.global_indices = np.asarray(global_indices, dtype=np.int64)
        self.augmented_features = np.asarray(augmented_features, dtype=np.float32)
        if len(self.global_indices) != len(self.augmented_features):
            raise ValueError("CAR augmented feature/index length mismatch.")
    def __len__(self):
        return len(self.global_indices)
    def __getitem__(self, local_idx):
        global_idx = int(self.global_indices[local_idx])
        row = train_df.iloc[global_idx]
        return (
            torch.tensor(self.augmented_features[local_idx], dtype=torch.float32),
            torch.tensor(row["labels"], dtype=torch.float32),
            torch.tensor(encode_text(row["report"]), dtype=torch.long),
        )


@torch.no_grad()
def build_car_retrieval_state(global_model):
    was_training = global_model.training
    original_device = next(global_model.parameters()).device
    global_model.to("cpu")
    global_model.eval()
    pub = torch.tensor(CAR_PUBLIC_FEATURES, dtype=torch.float32)
    emb1 = global_model.retrieval_embedding(pub[:, :1024], 0).cpu()
    emb2 = global_model.retrieval_embedding(pub[:, 1024:], 1).cpu()
    if was_training:
        global_model.train()
    global_model.to(original_device)
    return {
        "public_features": pub.cpu(),
        "public_labels": torch.tensor(CAR_PUBLIC_LABELS, dtype=torch.float32),
        "emb_view1": emb1,
        "emb_view2": emb2,
    }


def _jaccard_refine_topk(query_label, candidate_labels, candidate_distances):
    q = np.asarray(query_label > 0.5, dtype=bool)
    c = np.asarray(candidate_labels > 0.5, dtype=bool)
    inter = np.logical_and(c, q[None, :]).sum(axis=1).astype(np.float64)
    union = np.logical_or(c, q[None, :]).sum(axis=1).astype(np.float64)
    scores = np.divide(inter, union, out=np.ones_like(inter), where=union > 0)
    best_local = int(np.argmax(scores))  # top-k is nearest-first; ties prefer nearer
    return best_local, float(scores[best_local]), float(candidate_distances[best_local])


@torch.no_grad()
def car_retrieve_complementary_views(global_model, features, query_labels, missing_view_ids, retrieval_state):
    feats = torch.as_tensor(features, dtype=torch.float32).clone()
    labels_np = np.asarray(query_labels, dtype=np.float32)
    missing_view_ids = np.asarray(missing_view_ids, dtype=np.int64)
    pub_feat = retrieval_state["public_features"]
    pub_labels = retrieval_state["public_labels"].numpy()
    audit_jaccard, audit_distance, retrieved = [], [], []

    for missing_id in (0, 1):
        qpos = np.where(missing_view_ids == missing_id)[0]
        if len(qpos) == 0:
            continue
        available_id = 1 - missing_id
        q_view = feats[qpos, :1024] if available_id == 0 else feats[qpos, 1024:]
        model_device = next(global_model.parameters()).device
        q_emb = global_model.retrieval_embedding(
            q_view.to(model_device), available_id
        ).cpu()
        pub_emb = retrieval_state["emb_view1"] if available_id == 0 else retrieval_state["emb_view2"]
        distances = torch.cdist(q_emb, pub_emb, p=2)
        k = min(int(CAR_RETRIEVAL_TOPK), int(pub_emb.size(0)))
        top_dist, top_idx = torch.topk(distances, k=k, dim=1, largest=False, sorted=True)

        for row_offset, sample_pos in enumerate(qpos.tolist()):
            cand_idx = top_idx[row_offset].numpy().astype(np.int64)
            cand_dist = top_dist[row_offset].numpy().astype(np.float64)
            best_local, jac, dist = _jaccard_refine_topk(
                labels_np[sample_pos], pub_labels[cand_idx], cand_dist
            )
            chosen = int(cand_idx[best_local])
            if missing_id == 0:
                feats[sample_pos, :1024] = pub_feat[chosen, :1024]
            else:
                feats[sample_pos, 1024:] = pub_feat[chosen, 1024:]
            audit_jaccard.append(jac); audit_distance.append(dist); retrieved.append(chosen)

    stats = {
        "Retrieved_Count": int(len(retrieved)),
        "Mean_Retrieval_Jaccard": float(np.mean(audit_jaccard)) if audit_jaccard else 0.0,
        "Mean_Retrieval_Distance": float(np.mean(audit_distance)) if audit_distance else 0.0,
        "Unique_Public_Matches": int(len(set(retrieved))),
    }
    return feats.numpy().astype(np.float32), stats


def get_car_client_loader(cid, round_id, global_model, retrieval_state):
    idxs = np.asarray(CAR_PRIVATE_CLIENT_INDICES[int(cid)], dtype=np.int64)
    full_features = np.asarray(train_features[idxs], dtype=np.float32)
    query_labels = np.vstack(train_df.iloc[idxs]["labels"].values).astype(np.float32)
    rng = np.random.default_rng(SEED + 330001 + 10007 * int(cid) + 101 * int(round_id))
    missing = rng.random(len(idxs)) < float(CAR_TRAIN_MISSING_RATE)
    missing_view_ids = np.full(len(idxs), -1, dtype=np.int64)
    missing_view_ids[missing] = rng.integers(0, 2, size=int(missing.sum()))

    augmented_features, stats = car_retrieve_complementary_views(
        global_model, full_features, query_labels, missing_view_ids, retrieval_state
    )
    stats.update({
        "Round": int(round_id), "Client_ID": int(cid),
        "Private_Samples": int(len(idxs)), "Missing_Samples": int(missing.sum()),
        "Missing_Rate": float(missing.mean()) if len(missing) else 0.0,
        "Retrieval_TopK": int(CAR_RETRIEVAL_TOPK),
        "Label_Refinement": CAR_LABEL_REFINEMENT,
        "Public_Bank_Size": int(len(CAR_PUBLIC_INDICES)),
    })
    CAR_RETRIEVAL_AUDIT_ROWS.append(stats)
    ds = CARAugmentedDataset(idxs, augmented_features)
    generator = torch.Generator()
    generator.manual_seed(SEED + 440001 + 10007 * int(cid) + 101 * int(round_id))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, generator=generator)


@torch.no_grad()
def car_retrieve_eval_batch(model, feat, threshold, retrieval_state):
    """Leakage-safe extension for this study's induced test-time missing views."""
    x = feat.detach().cpu().clone()
    miss1 = x[:, :1024].abs().sum(dim=1) < 1e-6
    miss2 = x[:, 1024:].abs().sum(dim=1) < 1e-6
    missing_ids = np.full(x.size(0), -1, dtype=np.int64)
    missing_ids[miss1.numpy() & ~miss2.numpy()] = 0
    missing_ids[miss2.numpy() & ~miss1.numpy()] = 1
    if np.all(missing_ids < 0):
        return feat

    model_device = next(model.parameters()).device
    _, logits = MMFLModel.encode_memory(model, x.to(model_device))
    probs = torch.sigmoid(logits).cpu().numpy()
    pseudo = (probs >= float(threshold)).astype(np.float32)
    empty = np.where(pseudo.sum(axis=1) == 0)[0]
    if len(empty):
        pseudo[empty, np.argmax(probs[empty], axis=1)] = 1.0
    repaired, _ = car_retrieve_complementary_views(
        model, x.numpy(), pseudo, missing_ids, retrieval_state
    )
    return torch.tensor(repaired, dtype=feat.dtype, device=feat.device)


def compute_loss(model, batch, global_model=None, method="FedAvg", round_id=1):
    feat, labels, tokens = [x.to(DEVICE) for x in batch]
    full_feat = feat

    if method == "CAR-MFL":
        input_feat = feat
    elif method == "Proposed_CoHiM_FL":
        input_feat = apply_training_missing_augmentation(
            feat, proposed_missing_rate(round_id)
        )
    else:
        input_feat = feat

    label_logits, text_logits = model(input_feat, tokens)
    label_loss = F.binary_cross_entropy_with_logits(
        label_logits, labels, pos_weight=LABEL_POS_WEIGHT
    )
    target = tokens[:, 1:]
    text_loss = F.cross_entropy(
        text_logits.reshape(-1, VOCAB_SIZE),
        target.reshape(-1),
        ignore_index=PAD,
        label_smoothing=0.01,
    )
    # The same multi-task weights are used for every method.
    # The text objective receives the primary weight because report generation
    # is the principal task; classification remains an auxiliary clinical task.
    loss = TEXT_LOSS_W * text_loss + LABEL_LOSS_W * label_loss

    if method == "Proposed_CoHiM_FL":
        loss = loss + PROPOSED_IMPUTATION_LOSS_W * model.imputation_regularization(full_feat)
        loss = loss + PROPOSED_CONSISTENCY_LOSS_W * model.consistency_regularization(
            full_feat, input_feat
        )

    if method == "FedProx" and global_model is not None:
        prox = full_feat.new_tensor(0.0)
        for p, gp in zip(model.parameters(), global_model.parameters()):
            prox = prox + torch.sum((p - gp.detach()) ** 2)
        loss = loss + (FEDPROX_MU / 2.0) * prox

    return loss


def local_optimizer_name(method):
    # Source-grounded reference baselines use the same plain local SGD solver.
    return "SGD" if method in REFERENCE_BASELINE_METHODS else "AdamW"


def get_fedavg_learning_rate(override=None):
    """Return the frozen validation-selected local-SGD learning rate."""
    if override is not None:
        return float(override)
    if FEDAVG_SELECTED_LR is not None:
        return float(FEDAVG_SELECTED_LR)
    return float(FEDAVG_FALLBACK_LR)


def local_optimizer_lr(method, round_id):
    if method in REFERENCE_BASELINE_METHODS:
        return get_fedavg_learning_rate()
    return float(current_round_lr(round_id))


def build_local_optimizer(model, method, round_id, fedavg_lr_override=None):
    """Construct local optimizers with explicit, auditable baseline logic.

    FedAvg follows local mini-batch SGD + server model averaging.
    FedProx uses the same local SGD solver and adds its proximal objective.
    Fedsw uses the same local SGD solver before score-weighted aggregation.
    Keeping one local solver across these three baselines avoids attributing an
    optimizer advantage to the aggregation rule itself.
    """
    if method in REFERENCE_BASELINE_METHODS:
        return torch.optim.SGD(
            model.parameters(),
            lr=get_fedavg_learning_rate(fedavg_lr_override),
            momentum=FEDAVG_SGD_MOMENTUM,
            weight_decay=FEDAVG_SGD_WEIGHT_DECAY,
        )

    return torch.optim.AdamW(
        model.parameters(),
        lr=current_round_lr(round_id),
        weight_decay=WEIGHT_DECAY,
    )


def train_local(global_model, loader, method, round_id, fedavg_lr_override=None):
    # The trainable local model is placed on the active computation device.
    model = clone_model(global_model, DEVICE)
    model.train()

    # FedProx compares local parameters with a frozen copy of the global model.
    # Keep this reference on the same device for the entire local-training run.
    # This avoids CPU/CUDA tensor mismatches and avoids repeated transfers per batch.
    prox_reference = None
    if method == "FedProx":
        prox_reference = clone_model(global_model, DEVICE)
        prox_reference.eval()
        for parameter in prox_reference.parameters():
            parameter.requires_grad_(False)

    optimizer = build_local_optimizer(model, method, round_id, fedavg_lr_override=fedavg_lr_override)

    for _ in range(LOCAL_EPOCHS):
        for batch in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = compute_loss(
                model,
                batch,
                global_model=prox_reference,
                method=method,
                round_id=round_id,
            )
            if not torch.isfinite(loss):
                raise FloatingPointError(
                    f"Non-finite local loss for method={method}, round={round_id}."
                )
            loss.backward()
            # McMahan et al. Algorithm 1 is plain local SGD; therefore the FedAvg
            # path intentionally does not apply gradient clipping. Other methods
            # retain the notebook's prior clipping policy.
            if method not in REFERENCE_BASELINE_METHODS:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

    if prox_reference is not None:
        del prox_reference
    return model


# =========================================================
# Cross-view gradient-flow audit
# =========================================================
FUSION_AUDIT_GROUPS = {
    "view_norm1": ("view_norm1",),
    "view_norm2": ("view_norm2",),
    "cross_gate": ("cross_gate",),
    "fusion_refine": ("fusion_refine",),
    "fusion_strength": ("fusion_strength",),
}


def _group_grad_l2(model, prefixes):
    squared = 0.0
    found = False
    for name, param in model.named_parameters():
        if any(name == p or name.startswith(p + ".") for p in prefixes):
            found = True
            if param.grad is not None:
                squared += float(param.grad.detach().float().pow(2).sum().item())
    if not found:
        raise RuntimeError(f"Gradient-audit group not found: {prefixes}")
    return math.sqrt(squared)


def run_cross_view_gradient_audit():
    """Two-step, fail-fast audit on a cloned model; reported model is untouched.

    Step 1: because fusion_strength starts at zero, the initial forward mapping is
    reconstruction-backbone-equivalent. A non-zero residual mapping must nevertheless give
    fusion_strength a non-zero gradient.

    Step 2: after one optimizer update makes fusion_strength non-zero, gradients
    must propagate into view normalization, gating, and residual-refinement layers.
    """
    # Save RNG states so the audit cannot change the subsequent experiment stream.
    py_state = random.getstate()
    np_state = np.random.get_state()
    torch_state = torch.random.get_rng_state()
    cuda_states = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None

    audit_rows = []
    try:
        random.seed(SEED + 7001)
        np.random.seed(SEED + 7001)
        torch.manual_seed(SEED + 7001)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED + 7001)

        proposed = clone_model(MODEL_REGISTRY["Proposed_CoHiM_FL"], DEVICE)
        car = clone_model(PROPOSED_BACKBONE_TEMPLATE, DEVICE)

        # Use one deterministic real training batch.
        audit_loader = get_client_loader(0, round_id=1)
        batch = next(iter(audit_loader))
        feat = batch[0].to(DEVICE)

        # Verify the intended identity-preserving initialization before training.
        proposed.eval()
        car.eval()
        with torch.no_grad():
            car_memory, car_labels = car.encode_memory(feat)
            proposed_memory, proposed_labels = proposed.encode_memory(feat)
            init_memory_diff = float((car_memory - proposed_memory).abs().max().item())
            init_label_diff = float((car_labels - proposed_labels).abs().max().item())

        print("\nCross-view initialization audit")
        print(f"max |Backbone memory - Proposed memory| = {init_memory_diff:.6e}")
        print(f"max |Backbone logits - Proposed logits| = {init_label_diff:.6e}")
        if init_memory_diff > 1e-6 or init_label_diff > 1e-6:
            raise RuntimeError(
                "Identity-preserving initialization failed: Proposed does not start "
                "from the inherited reconstruction backbone."
            )

        proposed.train()
        optimizer = torch.optim.AdamW(
            proposed.parameters(),
            lr=current_round_lr(1),
            weight_decay=WEIGHT_DECAY,
        )

        for audit_step in (1, 2):
            optimizer.zero_grad(set_to_none=True)
            loss = compute_loss(
                proposed,
                batch,
                global_model=None,
                method="Proposed_CoHiM_FL",
                round_id=1,
            )
            loss.backward()

            row = {
                "Audit_Step": audit_step,
                "Loss": float(loss.detach().item()),
                "Fusion_Strength_Value": float(proposed.fusion_strength.detach().item()),
            }
            for group, prefixes in FUSION_AUDIT_GROUPS.items():
                row[f"GradL2_{group}"] = _group_grad_l2(proposed, prefixes)
            audit_rows.append(row)

            print(f"\nGradient audit step {audit_step}")
            print(f"loss = {row['Loss']:.6f}")
            print(f"fusion_strength = {row['Fusion_Strength_Value']:.6e}")
            for group in FUSION_AUDIT_GROUPS:
                print(f"{group:16s}: grad_l2 = {row[f'GradL2_{group}']:.6e}")

            if audit_step == 1:
                if row["GradL2_fusion_strength"] <= GRADIENT_AUDIT_EPS:
                    raise RuntimeError(
                        "Cross-view audit failed: fusion_strength has zero gradient at step 1."
                    )
            else:
                required = [
                    row["GradL2_view_norm1"],
                    row["GradL2_view_norm2"],
                    row["GradL2_cross_gate"],
                    row["GradL2_fusion_refine"],
                ]
                if any(value <= GRADIENT_AUDIT_EPS for value in required):
                    raise RuntimeError(
                        "Cross-view audit failed: the enhanced fusion branch did not "
                        "receive non-zero gradients after fusion_strength activated."
                    )

            torch.nn.utils.clip_grad_norm_(proposed.parameters(), 1.0)
            optimizer.step()

        audit_df = pd.DataFrame(audit_rows)
        print("\nPASS: cross-view fusion is trainable under the actual proposed-model loss.")
        display(audit_df)
        return audit_df

    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.random.set_rng_state(torch_state)
        if torch.cuda.is_available() and cuda_states is not None:
            torch.cuda.set_rng_state_all(cuda_states)
        cleanup_gpu()


@torch.no_grad()
def collect_label_outputs(model, data_loader, missing_rate=0.0, seed=None, missing_view="lateral"):
    if seed is None:
        seed = SEED
    model.to(DEVICE)
    model.eval()
    y_true_all, y_prob_all = [], []
    seen = 0
    with torch.inference_mode():
        for feat, labels, _tokens in data_loader:
            feat, labels = feat.to(DEVICE), labels.to(DEVICE)
            sample_indices = torch.arange(seen, seen + feat.size(0), device=DEVICE)
            seen += feat.size(0)
            feat = apply_deterministic_missing_corruption(
                feat,
                missing_rate=missing_rate,
                sample_indices=sample_indices,
                seed=seed,
                view=missing_view,
                population_size=len(data_loader.dataset),
            )
            _, logits = model.encode_memory(feat)
            y_true_all.append(labels.cpu().numpy())
            y_prob_all.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(y_true_all).astype(int), np.vstack(y_prob_all).astype(np.float32)


def predictions_from_threshold(y_prob, threshold):
    y_pred = (y_prob >= float(threshold)).astype(int)
    empty = np.where(y_pred.sum(axis=1) == 0)[0]
    if len(empty):
        y_pred[empty, np.argmax(y_prob[empty], axis=1)] = 1
    return y_pred


def calibrate_threshold(model):
    y_true, y_prob = collect_label_outputs(model, val_loader)
    candidates = []
    for threshold in THRESHOLD_GRID:
        y_pred = predictions_from_threshold(y_prob, threshold)
        score = f1_score(y_true, y_pred, average="micro", zero_division=0)
        candidates.append((float(score), -abs(float(threshold) - 0.5), float(threshold)))
    candidates.sort(reverse=True)
    return candidates[0][2]


@torch.no_grad()
def evaluate(
    model,
    max_batches=999999,
    fixed_threshold=None,
    missing_rate=0.0,
    seed=None,
    missing_view=MISSING_VIEW_FOR_EVAL,
    data_loader=None,
    return_audit=False,
    return_texts=False,
):
    if seed is None:
        seed = SEED
    loader = test_loader if data_loader is None else data_loader
    loader_population_size = len(loader.dataset)
    threshold = (
        float(getattr(model, "calibrated_threshold", 0.5))
        if fixed_threshold is None
        else float(fixed_threshold)
    )

    model.to(DEVICE)
    model.eval()
    car_eval_retrieval_state = (
        build_car_retrieval_state(model)
        if getattr(model, "method_name", "") == "CAR-MFL" and float(missing_rate) > 0.0
        else None
    )
    refs_bleu, hyps_bleu = [], []
    refs_text, hyps_text = [], []
    y_true_all, y_prob_all = [], []
    rouge_eval = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    seen = 0
    audit_mask_parts = []
    audit_index_parts = []
    masked_count_total = 0
    sample_count_total = 0
    original_abs_sum_total = 0.0
    corrupted_abs_sum_total = 0.0

    with torch.inference_mode():
        for batch_index, batch in enumerate(loader):
            if batch_index >= max_batches:
                break
            feat, labels, tokens = [x.to(DEVICE) for x in batch]
            sample_indices = torch.arange(seen, seen + feat.size(0), device=DEVICE)
            seen += feat.size(0)
            if return_audit:
                feat, batch_audit = apply_deterministic_missing_corruption(
                    feat,
                    missing_rate=missing_rate,
                    sample_indices=sample_indices,
                    seed=seed,
                    view=missing_view,
                    return_audit=True,
                    population_size=loader_population_size,
                )
                audit_mask_parts.append(batch_audit["mask"])
                audit_index_parts.append(batch_audit["sample_indices"])
                masked_count_total += batch_audit["masked_count"]
                sample_count_total += batch_audit["total_count"]
                original_abs_sum_total += batch_audit["original_abs_sum"]
                corrupted_abs_sum_total += batch_audit["corrupted_abs_sum"]
            else:
                feat = apply_deterministic_missing_corruption(
                    feat,
                    missing_rate=missing_rate,
                    sample_indices=sample_indices,
                    seed=seed,
                    view=missing_view,
                    population_size=loader_population_size,
                )

            if car_eval_retrieval_state is not None:
                feat = car_retrieve_eval_batch(model, feat, threshold, car_eval_retrieval_state)

            memory, label_logits = model.encode_memory(feat)
            label_probs = torch.sigmoid(label_logits)

            generated = torch.full(
                (feat.size(0), 1), BOS, dtype=torch.long, device=DEVICE
            )
            finished = torch.zeros(feat.size(0), dtype=torch.bool, device=DEVICE)
            for _ in range(MAX_LEN - 1):
                logits = model.generator(memory, generated)
                next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
                next_id[finished] = EOS
                generated = torch.cat([generated, next_id], dim=1)
                finished = finished | (next_id.squeeze(1) == EOS)
                if finished.all():
                    break

            for ref_ids, hyp_ids in zip(tokens.cpu().numpy(), generated.cpu().numpy()):
                ref_text = decode_ids(ref_ids)
                hyp_text = decode_ids(hyp_ids)
                refs_text.append(ref_text)
                hyps_text.append(hyp_text)
                refs_bleu.append([ref_text.split()])
                hyps_bleu.append(hyp_text.split())

            y_true_all.append(labels.cpu().numpy())
            y_prob_all.append(label_probs.cpu().numpy())

    smoothie = SmoothingFunction().method4
    bleu1 = corpus_bleu(
        refs_bleu, hyps_bleu, weights=(1, 0, 0, 0), smoothing_function=smoothie
    ) if refs_bleu else 0.0
    bleu4 = corpus_bleu(
        refs_bleu, hyps_bleu, weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothie
    ) if refs_bleu else 0.0
    rouge_scores = [
        rouge_eval.score(ref, hyp)["rougeL"].fmeasure
        for ref, hyp in zip(refs_text, hyps_text)
    ]
    meteor_scores = [
        meteor_score([ref.split()], hyp.split())
        for ref, hyp in zip(refs_text, hyps_text)
    ]

    y_true = np.vstack(y_true_all).astype(int)
    y_prob = np.vstack(y_prob_all).astype(np.float32)
    y_pred = predictions_from_threshold(y_prob, threshold)

    metrics = {
        "BLEU1": float(bleu1),
        "BLEU4": float(bleu4),
        "ROUGE_L": float(np.mean(rouge_scores)) if rouge_scores else 0.0,
        "METEOR": float(np.mean(meteor_scores)) if meteor_scores else 0.0,
        "Label_F1_micro": f1_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "Label_F1_weighted": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "Jaccard_micro": jaccard_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "Hamming_Loss": hamming_loss(y_true, y_pred),
        "Best_Threshold": threshold,
        "Best_Label_TopK": 0,
        "Calibration_Mode": "global_threshold_validation_selected",
    }

    if return_audit:
        mask_vector = (
            torch.cat(audit_mask_parts).numpy().astype(bool)
            if audit_mask_parts else np.zeros(len(y_true), dtype=bool)
        )
        index_vector = (
            torch.cat(audit_index_parts).numpy().astype(np.int64)
            if audit_index_parts else np.arange(len(y_true), dtype=np.int64)
        )
        metrics["_audit"] = {
            "y_true": y_true,
            "y_prob": y_prob,
            "y_pred": y_pred,
            "mask": mask_vector,
            "sample_indices": index_vector,
            "masked_count": int(masked_count_total),
            "sample_count": int(sample_count_total),
            "actual_missing_rate": (
                float(masked_count_total / sample_count_total)
                if sample_count_total else 0.0
            ),
            "target_masked_count": int(round(float(missing_rate) * sample_count_total)),
            "nearest_achievable_rate": (
                float(round(float(missing_rate) * sample_count_total) / sample_count_total)
                if sample_count_total else 0.0
            ),
            "feature_energy_retention": (
                float(corrupted_abs_sum_total / original_abs_sum_total)
                if original_abs_sum_total > 0 else 1.0
            ),
        }

    if return_texts:
        metrics["_texts"] = {
            "reference_reports": list(refs_text),
            "generated_reports": list(hyps_text),
        }

    return metrics


# =========================================================
# 11) Client profiles, selection, and aggregation
# =========================================================

def state_dict_nbytes(state_dict):
    """Return the exact dense payload size of a PyTorch state dictionary."""
    total = 0
    for tensor in state_dict.values():
        total += tensor.numel() * tensor.element_size()
    return int(total)


def model_size_mb(model):
    return state_dict_nbytes(model.state_dict()) / (1024 ** 2)


def _select_adaptive_topk_ratio(
    corrected_delta,
    round_id,
    candidate_ratios=ADAPTIVE_TOPK_CANDIDATES,
    target_energy=ADAPTIVE_TOPK_TARGET_ENERGY,
):
    """Select the smallest predefined Top-K ratio preserving target L2 energy.

    The decision uses only the current client update and the predefined round
    schedule. It does not use validation or test metrics. During early rounds,
    a conservative warm-up ratio is used to stabilize optimization.
    """
    numel = int(corrected_delta.numel())
    if numel == 0:
        return 1.0

    if round_id <= ADAPTIVE_TOPK_WARMUP_ROUNDS:
        return float(ADAPTIVE_TOPK_WARMUP_RATIO)

    flat_energy = corrected_delta.detach().float().reshape(-1).pow(2)
    total_energy = float(flat_energy.sum().item())
    if total_energy <= 0.0:
        return float(min(candidate_ratios))

    sorted_energy, _ = torch.sort(flat_energy, descending=True)
    cumulative_energy = torch.cumsum(sorted_energy, dim=0)

    for ratio in sorted(candidate_ratios):
        k = max(
            ADAPTIVE_TOPK_MIN_ELEMENTS,
            int(math.ceil(float(ratio) * numel)),
        )
        k = min(k, numel)
        retained = float(cumulative_energy[k - 1].item() / total_energy)
        if retained >= float(target_energy):
            return float(ratio)

    return float(max(candidate_ratios))


def adaptive_topk_quantize_model_update(
    reference_model,
    updated_model,
    residual_state=None,
    round_id=1,
    bits=8,
    use_error_feedback=True,
    use_topk=True,
    quantize_int8=True,
):
    """Transmit a client update with independently switchable Top-K, INT8 and EF.

    This function is used by both the full proposed method and the communication
    ablations. Communication bytes are computed from the payload that is actually
    represented in each variant; no post-hoc scaling of communication cost is used.
    """
    if quantize_int8 and bits != 8:
        raise ValueError("INT8 transport requires bits=8.")

    reference_state = reference_model.state_dict()
    updated_state = updated_model.state_dict()
    reconstructed_state = {}
    next_residual = {}
    payload_bytes = 0
    retained_elements = 0
    total_float_elements = 0
    selected_ratios = []
    qmax = 2 ** (bits - 1) - 1 if quantize_int8 else None

    with torch.no_grad():
        for key, reference_tensor in reference_state.items():
            updated_tensor = updated_state[key]
            reference_cpu = reference_tensor.detach().cpu()

            if torch.is_floating_point(reference_cpu):
                reference_float = reference_cpu.float()
                original_delta = updated_tensor.detach().cpu().float() - reference_float
                corrected_delta = original_delta

                if use_error_feedback and residual_state is not None:
                    previous_residual = residual_state.get(key)
                    if previous_residual is not None:
                        corrected_delta = corrected_delta + previous_residual.detach().cpu().float()

                numel = int(corrected_delta.numel())
                total_float_elements += numel
                ratio = (
                    _select_adaptive_topk_ratio(corrected_delta, round_id=round_id)
                    if use_topk else 1.0
                )
                selected_ratios.append(float(ratio))

                k = max(ADAPTIVE_TOPK_MIN_ELEMENTS, int(math.ceil(float(ratio) * numel)))
                k = min(k, numel)
                flat_delta = corrected_delta.reshape(-1)
                if k < numel:
                    topk_indices = torch.topk(
                        flat_delta.abs(), k=k, largest=True, sorted=False
                    ).indices
                else:
                    topk_indices = torch.arange(numel, dtype=torch.long)

                selected_values = flat_delta[topk_indices]
                if quantize_int8:
                    max_abs = float(selected_values.abs().max().item()) if selected_values.numel() else 0.0
                    if max_abs == 0.0:
                        scale = 1.0
                        encoded_values = torch.zeros(selected_values.numel(), dtype=torch.int8)
                    else:
                        scale = max_abs / qmax
                        encoded_values = torch.clamp(
                            torch.round(selected_values / scale), -qmax, qmax
                        ).to(torch.int8)
                    decoded_values = encoded_values.float() * scale
                    payload_bytes += encoded_values.numel() * encoded_values.element_size()
                    payload_bytes += COMM_SCALE_BYTES
                else:
                    encoded_values = selected_values.to(torch.float32)
                    decoded_values = encoded_values
                    payload_bytes += encoded_values.numel() * encoded_values.element_size()

                sparse_reconstruction = torch.zeros(numel, dtype=torch.float32)
                sparse_reconstruction[topk_indices] = decoded_values
                sparse_reconstruction = sparse_reconstruction.reshape(corrected_delta.shape)
                reconstructed_state[key] = (reference_float + sparse_reconstruction).to(reference_cpu.dtype)

                residual = corrected_delta - sparse_reconstruction
                next_residual[key] = residual if use_error_feedback else torch.zeros_like(residual)
                retained_elements += int(k)

                # Sparse index/mask cost exists only when elements are omitted.
                if use_topk and k < numel:
                    if ADAPTIVE_TOPK_USE_BITMAP:
                        payload_bytes += int(math.ceil(numel / 8.0))
                        payload_bytes += COMM_BITMAP_HEADER_BYTES
                    else:
                        index_bytes = 2 if numel <= 65535 else 4
                        payload_bytes += int(k * index_bytes)
                payload_bytes += COMM_TENSOR_METADATA_BYTES
            else:
                exact_tensor = updated_tensor.detach().cpu().clone()
                reconstructed_state[key] = exact_tensor
                next_residual[key] = torch.zeros_like(exact_tensor, dtype=torch.float32)
                payload_bytes += exact_tensor.numel() * exact_tensor.element_size()
                payload_bytes += COMM_TENSOR_METADATA_BYTES

    reconstructed_model = reference_model.__class__().cpu()
    reconstructed_model.load_state_dict(reconstructed_state, strict=True)
    reconstructed_model.method_name = getattr(updated_model, "method_name", updated_model.__class__.__name__)
    if hasattr(updated_model, "ablation_config"):
        reconstructed_model.ablation_config = copy.deepcopy(updated_model.ablation_config)

    effective_ratio = float(retained_elements / total_float_elements) if total_float_elements > 0 else 1.0
    mean_selected_ratio = float(np.mean(selected_ratios)) if selected_ratios else 1.0
    mode = (
        ("adaptive_topk" if use_topk else "dense")
        + ("_INT8" if quantize_int8 else "_FP32")
        + ("_EF" if use_error_feedback else "_noEF")
    )
    stats = {
        "Effective_TopK_Ratio": effective_ratio,
        "Mean_Tensor_TopK_Ratio": mean_selected_ratio,
        "Retained_Elements": int(retained_elements),
        "Total_Float_Elements": int(total_float_elements),
        "TopK_Target_Energy": float(ADAPTIVE_TOPK_TARGET_ENERGY) if use_topk else 1.0,
        "TopK_Mode": mode,
    }
    return reconstructed_model, int(payload_bytes), _clone_cpu_state(next_residual), stats



def quantize_model_for_downlink(model, bits=16):
    """Simulate the actual server-to-client transport of the global model.

    Floating-point tensors are serialized in FP16, transmitted, and then
    reconstructed as FP32 before local optimization. Integer/buffer tensors
    are transmitted exactly. The returned byte count is based on the payload
    that is actually transported; no post-hoc reduction factor is used.

    This transformation may introduce small numerical differences because the
    clients train from the reconstructed FP16 model received from the server.
    """
    if bits != 16:
        raise ValueError("The downlink implementation currently supports FP16 only.")

    transported_state = {}
    payload_bytes = 0

    with torch.no_grad():
        for key, tensor in model.state_dict().items():
            tensor_cpu = tensor.detach().cpu()

            if torch.is_floating_point(tensor_cpu):
                transported = tensor_cpu.to(torch.float16)
                payload_bytes += transported.numel() * transported.element_size()
                payload_bytes += COMM_TENSOR_METADATA_BYTES

                # Local optimization remains FP32 after receiving/decompressing.
                transported_state[key] = transported.to(tensor_cpu.dtype)
            else:
                transported_state[key] = tensor_cpu.clone()
                payload_bytes += tensor_cpu.numel() * tensor_cpu.element_size()
                payload_bytes += COMM_TENSOR_METADATA_BYTES

    reconstructed_model = model.__class__().cpu()
    reconstructed_model.load_state_dict(transported_state, strict=True)
    reconstructed_model.method_name = getattr(
        model, "method_name", model.__class__.__name__
    )
    return reconstructed_model, int(payload_bytes)



def _clone_cpu_state(state_dict):
    return {
        key: tensor.detach().cpu().clone()
        for key, tensor in state_dict.items()
    }


def transmit_cached_downlink(
    server_model,
    cached_state=None,
    residual_state=None,
    initial_bits=16,
    delta_bits=8,
    use_error_feedback=True,
    quantize_delta=True,
):
    """Client-cached downlink with optional INT8 delta quantization.

    First participation remains FP16 for every proposed variant. Later cached
    deltas are INT8 in the full method and FP32 in the ``w/o INT8`` ablation.
    """
    if initial_bits != 16:
        raise ValueError("Initial cached downlink currently supports FP16.")
    if quantize_delta and delta_bits != 8:
        raise ValueError("Quantized cached model-delta downlink requires INT8.")

    server_state = server_model.state_dict()
    received_state = {}
    next_residual = {}
    payload_bytes = 0

    if cached_state is None:
        with torch.no_grad():
            for key, server_tensor in server_state.items():
                tensor_cpu = server_tensor.detach().cpu()
                if torch.is_floating_point(tensor_cpu):
                    transported = tensor_cpu.to(torch.float16)
                    received_state[key] = transported.to(tensor_cpu.dtype)
                    payload_bytes += transported.numel() * transported.element_size()
                    payload_bytes += COMM_TENSOR_METADATA_BYTES
                    next_residual[key] = (
                        tensor_cpu.float() - received_state[key].float()
                        if use_error_feedback else torch.zeros_like(tensor_cpu, dtype=torch.float32)
                    )
                else:
                    received_state[key] = tensor_cpu.clone()
                    next_residual[key] = torch.zeros_like(tensor_cpu, dtype=torch.float32)
                    payload_bytes += tensor_cpu.numel() * tensor_cpu.element_size()
                    payload_bytes += COMM_TENSOR_METADATA_BYTES
    else:
        qmax = 2 ** (delta_bits - 1) - 1 if quantize_delta else None
        with torch.no_grad():
            for key, server_tensor in server_state.items():
                server_cpu = server_tensor.detach().cpu()
                cached_cpu = cached_state[key].detach().cpu()
                if torch.is_floating_point(server_cpu):
                    corrected_delta = server_cpu.float() - cached_cpu.float()
                    if use_error_feedback and residual_state is not None:
                        previous_residual = residual_state.get(key)
                        if previous_residual is not None:
                            corrected_delta = corrected_delta + previous_residual.detach().cpu().float()

                    if quantize_delta:
                        max_abs = float(corrected_delta.abs().max().item()) if corrected_delta.numel() else 0.0
                        if max_abs == 0.0:
                            scale = 1.0
                            encoded = torch.zeros_like(corrected_delta, dtype=torch.int8)
                        else:
                            scale = max_abs / qmax
                            encoded = torch.clamp(
                                torch.round(corrected_delta / scale), -qmax, qmax
                            ).to(torch.int8)
                        decoded_delta = encoded.float() * scale
                        payload_bytes += encoded.numel() * encoded.element_size()
                        payload_bytes += COMM_SCALE_BYTES + COMM_TENSOR_METADATA_BYTES
                    else:
                        encoded = corrected_delta.to(torch.float32)
                        decoded_delta = encoded
                        payload_bytes += encoded.numel() * encoded.element_size()
                        payload_bytes += COMM_TENSOR_METADATA_BYTES

                    reconstructed = cached_cpu.float() + decoded_delta
                    received_state[key] = reconstructed.to(server_cpu.dtype)
                    next_residual[key] = (
                        server_cpu.float() - reconstructed.float()
                        if use_error_feedback else torch.zeros_like(server_cpu, dtype=torch.float32)
                    )
                else:
                    received_state[key] = server_cpu.clone()
                    next_residual[key] = torch.zeros_like(server_cpu, dtype=torch.float32)
                    payload_bytes += server_cpu.numel() * server_cpu.element_size()
                    payload_bytes += COMM_TENSOR_METADATA_BYTES

    received_model = server_model.__class__().cpu()
    received_model.load_state_dict(received_state, strict=True)
    received_model.method_name = getattr(server_model, "method_name", server_model.__class__.__name__)
    if hasattr(server_model, "ablation_config"):
        received_model.ablation_config = copy.deepcopy(server_model.ablation_config)

    return received_model, int(payload_bytes), _clone_cpu_state(received_state), _clone_cpu_state(next_residual)


def dense_round_payload_bytes(model, selected_clients_count):
    """Dense FL payload: one downlink model and one uplink model per client."""
    dense_bytes = state_dict_nbytes(model.state_dict())
    downlink = dense_bytes * int(selected_clients_count) if COMM_COUNT_DOWNLINK else 0
    uplink = dense_bytes * int(selected_clients_count)
    return int(downlink), int(uplink)


def rebuild_client_profile_cache():
    global CLIENT_LABEL_PREVALENCE
    rows = []
    for cid, idxs in enumerate(client_indices):
        matrix = np.vstack(train_df.iloc[idxs]["labels"].values).astype(np.float32)
        rows.append(matrix.mean(axis=0))
    CLIENT_LABEL_PREVALENCE = np.vstack(rows)
    return CLIENT_LABEL_PREVALENCE


def configure_experiment_seed(experiment_seed, verbose=True):
    """Configure one independent seed without changing the fixed data split."""
    global SEED, CAR_PUBLIC_BANK_SEED
    SEED = int(experiment_seed)
    CAR_PUBLIC_BANK_SEED = SEED + 2407
    set_global_seed(SEED)
    rebuild_federated_partition(SEED, verbose=False)
    rebuild_client_profile_cache()
    rebuild_model_registry(SEED)
    if verbose:
        print("\n" + "=" * 80)
        print(f"CONFIGURED INDEPENDENT EXPERIMENT SEED: {SEED}")
        print("Fixed train/validation/test split seed:", DATA_SPLIT_SEED)
        print("Dirichlet Non-IID partition seed:", SEED)
        print("Model/training stochastic seed:", SEED)
        print("=" * 80)


rebuild_client_profile_cache()


def client_profile(cid):
    idxs = client_indices[cid]
    prevalence = CLIENT_LABEL_PREVALENCE[cid]
    probability = prevalence / (prevalence.sum() + 1e-12)
    nonzero = probability[probability > 0]
    entropy = float(
        -(nonzero * np.log(nonzero + 1e-12)).sum() / np.log(NUM_LABELS + 1e-12)
    ) if len(nonzero) else 0.0

    n = len(idxs)
    rng = np.random.RandomState(SEED + cid)
    cost = float(0.6 + rng.rand() * 1.4)
    delay = float(0.02 + rng.rand() * 0.20)
    packet_loss = float(rng.rand() * 0.08)
    reliability = 1.0 / (1.0 + delay + packet_loss)
    data_size_score = np.log1p(n) / (1.0 + np.log1p(n))
    cost_score = 1.0 / cost
    base_utility = (
        0.35 * data_size_score
        + 0.25 * entropy
        + 0.25 * reliability
        + 0.15 * cost_score
    )
    return {
        "cid": int(cid),
        "n": int(n),
        "cost": cost,
        "reliability": float(reliability),
        "label_entropy": entropy,
        "data_size_score": float(data_size_score),
        "utility": float(base_utility),
        "prevalence": prevalence,
    }


def build_clustering_profile(cid):
    p = client_profile(cid)
    extra = np.array([
        p["cost"], p["data_size_score"], p["reliability"], p["label_entropy"]
    ], dtype=np.float32)
    return np.concatenate([p["prevalence"], extra])


def determine_adaptive_edge_count(selected_clients_count):
    """Derive the number of edge aggregators from the current workload."""
    requested = max(1, int(selected_clients_count))
    adaptive_count = int(math.ceil(requested / float(TARGET_CLIENTS_PER_EDGE)))
    return int(np.clip(adaptive_count, ADAPTIVE_EDGE_MIN, ADAPTIVE_EDGE_MAX))


def allocate_edge_capacities(edge_groups, selected_clients_count):
    """Allocate exactly the requested client slots across edges by cluster size."""
    active_edges = [edge for edge, members in edge_groups.items() if members]
    if not active_edges:
        return {}

    total_slots = max(1, int(selected_clients_count))
    sizes = np.asarray(
        [len(edge_groups[edge]) for edge in active_edges],
        dtype=np.float64,
    )
    raw = total_slots * sizes / max(1.0, sizes.sum())
    capacities = np.floor(raw).astype(int)

    # When enough slots exist, every non-empty edge receives one representative.
    if total_slots >= len(active_edges):
        capacities = np.maximum(capacities, 1)

    # Respect each cluster's available client count.
    capacities = np.minimum(capacities, sizes.astype(int))

    while capacities.sum() < total_slots:
        residual = raw - capacities
        order = np.argsort(-residual)
        changed = False
        for pos in order:
            if capacities[pos] < sizes[pos]:
                capacities[pos] += 1
                changed = True
                if capacities.sum() == total_slots:
                    break
        if not changed:
            break

    while capacities.sum() > total_slots:
        removable = [
            pos for pos in range(len(capacities))
            if capacities[pos] > (1 if total_slots >= len(active_edges) else 0)
        ]
        if not removable:
            break
        pos = min(removable, key=lambda p: raw[p] - capacities[p])
        capacities[pos] -= 1

    return {
        int(edge): int(capacity)
        for edge, capacity in zip(active_edges, capacities)
        if capacity > 0
    }


def allocate_edge_budgets(edge_groups, edge_capacities):
    """Estimate each edge budget from the costs of its own candidate clients."""
    budgets = {}
    for edge, capacity in edge_capacities.items():
        candidate_costs = sorted(
            client_profile(cid)["cost"] for cid in edge_groups[edge]
        )
        required_cost = float(sum(candidate_costs[:capacity]))
        budgets[int(edge)] = max(
            required_cost * EDGE_BUDGET_MARGIN,
            required_cost,
        )
    return budgets


def dynamic_client_clustering(round_id, selected_clients_count):
    """Rebuild edge groups each round using an adaptive number of clusters."""
    client_ids = list(range(NUM_CLIENTS))
    profiles = np.vstack([build_clustering_profile(cid) for cid in client_ids])
    requested_edges = determine_adaptive_edge_count(selected_clients_count)
    k = min(requested_edges, len(client_ids))

    if k == 1:
        labels = np.zeros(len(client_ids), dtype=np.int64)
    else:
        labels = KMeans(
            n_clusters=k,
            random_state=SEED + round_id,
            n_init=10,
        ).fit_predict(profiles)

    edge_groups = {edge: [] for edge in range(k)}
    cluster_labels = {}
    for cid, label in zip(client_ids, labels):
        edge_groups[int(label)].append(int(cid))
        cluster_labels[int(cid)] = int(label)

    edge_capacities = allocate_edge_capacities(
        edge_groups,
        selected_clients_count,
    )
    edge_budgets = allocate_edge_budgets(edge_groups, edge_capacities)

    edge_state = {
        "requested_edges": int(requested_edges),
        "active_edges": int(len(edge_capacities)),
        "capacities": edge_capacities,
        "budgets": edge_budgets,
    }
    return edge_groups, cluster_labels, edge_state


def proposed_selection_score(cid, covered, selection_counts):
    profile = client_profile(cid)
    coverage_gain = float(np.maximum(profile["prevalence"] - covered, 0.0).mean())
    fairness = 1.0 / (1.0 + selection_counts.get(cid, 0))
    # Diversity and fairness prevent repeatedly selecting only the easiest clients.
    return 0.35 * profile["utility"] + 0.35 * coverage_gain + 0.30 * fairness


def select_clients(method, count, round_id, selection_counts=None, return_edges=False, ablation_config=None):
    if method != "Proposed_CoHiM_FL":
        rng = random.Random(SEED + 1000 * int(count) + int(round_id))
        selected = rng.sample(range(NUM_CLIENTS), count)
        return (selected, None, None) if return_edges else selected

    selection_counts = selection_counts if selection_counts is not None else {}
    cfg = normalize_ablation_config(ablation_config)
    edge_groups, cluster_labels, edge_state = dynamic_client_clustering(
        round_id,
        count,
    )

    # Ablation: remove the adaptive utility/coverage/fairness ranking while
    # preserving the same edge topology and requested participation budget.
    if not cfg["adaptive_client_selection"]:
        rng = random.Random(SEED + 91000 + 1000 * int(count) + int(round_id))
        selected = []
        selected_edges = {edge: [] for edge in edge_state["capacities"]}
        for edge, capacity in edge_state["capacities"].items():
            candidates = list(edge_groups[edge])
            rng.shuffle(candidates)
            take = candidates[:int(capacity)]
            selected.extend(take)
            selected_edges[edge].extend(take)
        if len(selected) < count:
            remaining = [cid for cid in range(NUM_CLIENTS) if cid not in selected]
            rng.shuffle(remaining)
            for cid in remaining[:count - len(selected)]:
                selected.append(cid)
                selected_edges.setdefault(cluster_labels[cid], []).append(cid)
        selected = selected[:count]
        for cid in selected:
            selection_counts[cid] = selection_counts.get(cid, 0) + 1
        edge_state["used_edges"] = int(sum(bool(v) for v in selected_edges.values()))
        edge_state["remaining_budgets"] = dict(edge_state["budgets"])
        return (selected, selected_edges, edge_state) if return_edges else selected
    edge_capacities = edge_state["capacities"]
    edge_budget = dict(edge_state["budgets"])
    selected = []
    selected_edges = {edge: [] for edge in edge_capacities}
    covered = np.zeros(NUM_LABELS, dtype=np.float32)

    # Select clients edge by edge, while adapting each edge's quota and budget
    # to the size and estimated communication cost of its current cluster.
    while len(selected) < count:
        feasible = []
        capacity_only = []

        for edge, capacity in edge_capacities.items():
            if len(selected_edges[edge]) >= capacity:
                continue

            remaining = [
                cid for cid in edge_groups[edge]
                if cid not in selected
            ]
            if not remaining:
                continue

            ranked = sorted(
                remaining,
                key=lambda cid: proposed_selection_score(
                    cid, covered, selection_counts
                ),
                reverse=True,
            )
            capacity_only.extend((cid, edge) for cid in ranked)

            for cid in ranked:
                if client_profile(cid)["cost"] <= edge_budget[edge] + 1e-12:
                    feasible.append((cid, edge))
                    break

        candidate_pool = feasible if feasible else capacity_only
        if not candidate_pool:
            break

        chosen, edge = max(
            candidate_pool,
            key=lambda item: proposed_selection_score(
                item[0], covered, selection_counts
            ),
        )
        profile = client_profile(chosen)
        selected.append(chosen)
        selected_edges[edge].append(chosen)
        edge_budget[edge] = max(0.0, edge_budget[edge] - profile["cost"])
        covered = np.maximum(covered, profile["prevalence"])

    # Defensive completion is based only on the remaining clients and is used
    # solely if numerical/resource constraints leave unfilled requested slots.
    if len(selected) < count:
        remaining = [cid for cid in range(NUM_CLIENTS) if cid not in selected]
        remaining.sort(
            key=lambda cid: proposed_selection_score(
                cid, covered, selection_counts
            ),
            reverse=True,
        )
        for cid in remaining[: count - len(selected)]:
            edge = cluster_labels[cid]
            selected.append(cid)
            selected_edges.setdefault(edge, []).append(cid)

    for cid in selected:
        selection_counts[cid] = selection_counts.get(cid, 0) + 1

    edge_state["remaining_budgets"] = edge_budget
    edge_state["used_edges"] = int(
        sum(bool(members) for members in selected_edges.values())
    )
    return (selected, selected_edges, edge_state) if return_edges else selected


def update_l2_norm(global_model, local_model):
    total = 0.0
    gs, ls = global_model.state_dict(), local_model.state_dict()
    with torch.no_grad():
        for key in gs:
            diff = ls[key].detach().cpu().float() - gs[key].detach().cpu().float()
            total += float(torch.sum(diff * diff).item())
    return float(np.sqrt(total + 1e-12))


def average_models(models_list, weights=None):
    if weights is None:
        weights = np.repeat(1.0 / len(models_list), len(models_list))
    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / (weights.sum() + 1e-12)
    new_model = models_list[0].__class__().cpu()
    new_state = {}
    with torch.no_grad():
        for key in models_list[0].state_dict():
            accumulator = None
            for weight, model in zip(weights, models_list):
                value = model.state_dict()[key].detach().cpu().float() * float(weight)
                accumulator = value if accumulator is None else accumulator + value
            new_state[key] = accumulator
    new_model.load_state_dict(new_state)
    new_model.method_name = getattr(models_list[0], "method_name", new_model.__class__.__name__)
    return new_model.to(DEVICE)


@torch.no_grad()
def fedsw_validation_bleu1(model, max_batches=FEDSW_SCORE_MAX_BATCHES):
    """Predeclared Fedsw score s_k computed on validation data only.

    Chen & Pan (2024), Algorithm 3, uses a model score to derive aggregation
    proportions. Their pseudocode evaluates on the test set; this implementation
    replaces that with validation BLEU-1 to prevent test leakage.
    """
    if FEDSW_SCORE_SPLIT != "validation":
        raise RuntimeError("Publication run requires FEDSW_SCORE_SPLIT='validation'.")

    model.to(DEVICE)
    model.eval()
    refs_bleu, hyps_bleu = [], []

    with torch.inference_mode():
        for batch_idx, (feat, _labels, tokens) in enumerate(val_loader):
            if max_batches is not None and batch_idx >= int(max_batches):
                break
            feat = feat.to(DEVICE)
            tokens = tokens.to(DEVICE)
            memory, _ = model.encode_memory(feat)

            generated = torch.full(
                (feat.size(0), 1), BOS, dtype=torch.long, device=DEVICE
            )
            finished = torch.zeros(feat.size(0), dtype=torch.bool, device=DEVICE)
            for _ in range(MAX_LEN - 1):
                logits = model.generator(memory, generated)
                next_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)
                next_id[finished] = EOS
                generated = torch.cat([generated, next_id], dim=1)
                finished = finished | (next_id.squeeze(1) == EOS)
                if finished.all():
                    break

            for ref_ids, hyp_ids in zip(
                tokens.detach().cpu().numpy(),
                generated.detach().cpu().numpy(),
            ):
                ref_text = decode_ids(ref_ids)
                hyp_text = decode_ids(hyp_ids)
                refs_bleu.append([ref_text.split()])
                hyps_bleu.append(hyp_text.split())

    model.to("cpu")
    if not refs_bleu:
        return 0.0
    return float(
        corpus_bleu(
            refs_bleu,
            hyps_bleu,
            weights=(1.0, 0.0, 0.0, 0.0),
            smoothing_function=SmoothingFunction().method4,
        )
    )


def fedsw_score_weights(scores):
    """Normalize institution scores into convex aggregation weights r_k."""
    scores = np.asarray(scores, dtype=np.float64)
    if scores.ndim != 1 or len(scores) == 0:
        raise ValueError("Fedsw requires a non-empty one-dimensional score vector.")
    if not np.all(np.isfinite(scores)):
        raise FloatingPointError("Fedsw received a non-finite validation score.")

    # BLEU is non-negative. The floor prevents a zero-sum first round while
    # preserving the score ranking described by Chen & Pan.
    positive = np.maximum(scores, 0.0)
    if positive.sum() <= FEDSW_SCORE_EPS:
        weights = np.repeat(1.0 / len(positive), len(positive))
    else:
        weights = positive / positive.sum()

    if not np.isclose(weights.sum(), 1.0, atol=1e-8):
        raise RuntimeError("Fedsw score weights do not sum to one.")
    return weights


def fedsw_score_weighted_aggregate(local_models, local_scores):
    """Score-weighted aggregation implementing the core Fedsw idea."""
    weights = fedsw_score_weights(local_scores)
    candidate = average_models(local_models, weights)
    candidate.method_name = "Fedsw"
    return candidate, weights


def fixed_reference_clients(selected_clients_count):
    """Deterministic fixed institution set for source-paper-style baseline runs."""
    count = int(selected_clients_count)
    if count < 1 or count > NUM_CLIENTS:
        raise ValueError("selected_clients_count is outside the available client range.")
    rng = random.Random(SEED + 310000 + count)
    return sorted(rng.sample(range(NUM_CLIENTS), count))


def robust_aggregate_updates(global_model, local_models, local_sizes, client_ids=None):
    """Single-stage robust delta aggregation with bounded, non-zero weights.

    Valid updates are not exponentially suppressed. Client deltas are clipped
    only when their norm exceeds a generous median-based bound.
    """
    norms = np.asarray([
        update_l2_norm(global_model, model) for model in local_models
    ], dtype=np.float64)
    median = float(np.median(norms))

    size_w = np.asarray(local_sizes, dtype=np.float64)
    size_w /= size_w.sum() + 1e-12

    relative_norm = norms / (median + 1e-12)
    stability = 1.0 / (1.0 + ROBUST_ALPHA * relative_norm)
    stability = np.clip(stability, 0.50, 1.00)

    if client_ids is not None:
        utilities = np.asarray(
            [client_profile(cid)["utility"] for cid in client_ids],
            dtype=np.float64,
        )
        utilities = (utilities - utilities.min()) / (
            utilities.max() - utilities.min() + 1e-12
        )
        utility_w = 0.90 + 0.10 * utilities
    else:
        utility_w = np.ones_like(size_w)

    weights = size_w * stability * utility_w
    weights /= weights.sum() + 1e-12

    clip_target = ROBUST_CLIP_MULTIPLIER * max(median, 1e-12)
    clip_scales = np.minimum(1.0, clip_target / (norms + 1e-12))

    new_model = global_model.__class__().cpu()
    global_state = global_model.state_dict()
    new_state = {}
    with torch.no_grad():
        for key in global_state:
            base = global_state[key].detach().cpu().float()
            aggregate_delta = torch.zeros_like(base)
            for weight, scale, local_model in zip(weights, clip_scales, local_models):
                local_value = local_model.state_dict()[key].detach().cpu().float()
                aggregate_delta += float(weight * scale) * (local_value - base)
            new_state[key] = base + aggregate_delta
    new_model.load_state_dict(new_state)
    new_model.method_name = getattr(
        global_model, "method_name", new_model.__class__.__name__
    )
    return new_model.to(DEVICE)


def apply_server_momentum(previous_model, candidate_model, velocity):
    previous_state = previous_model.state_dict()
    candidate_state = candidate_model.state_dict()
    new_model = previous_model.__class__().cpu()
    new_state, new_velocity = {}, {}
    with torch.no_grad():
        for key in previous_state:
            previous_value = previous_state[key].detach().cpu().float()
            candidate_value = candidate_state[key].detach().cpu().float()
            delta = candidate_value - previous_value
            if velocity is None or key not in velocity:
                current_velocity = delta
            else:
                current_velocity = (
                    SERVER_MOMENTUM_BETA * velocity[key]
                    + (1.0 - SERVER_MOMENTUM_BETA) * delta
                )
            new_velocity[key] = current_velocity.detach().cpu()
            new_state[key] = previous_value + current_velocity
    new_model.load_state_dict(new_state)
    new_model.method_name = getattr(previous_model, "method_name", new_model.__class__.__name__)
    return new_model.to(DEVICE), new_velocity


# =========================================================
# 11.5) Validation-only FedAvg learning-rate tuning
# =========================================================

@torch.no_grad()
def fedavg_validation_objective(model):
    """Multi-task validation loss used only for FedAvg LR selection.

    No test example or test metric is accessed here. The loss is the same base
    report-generation + label-classification objective used by FedAvg training.
    """
    model.to(DEVICE)
    model.eval()
    total_loss = 0.0
    total_samples = 0
    with torch.inference_mode():
        for batch in val_loader:
            loss = compute_loss(
                model, batch, global_model=None, method="FedAvg", round_id=1
            )
            if not torch.isfinite(loss):
                return float("inf")
            batch_size = int(batch[0].size(0))
            total_loss += float(loss.detach().item()) * batch_size
            total_samples += batch_size
    model.to("cpu")
    return total_loss / max(total_samples, 1)


def tune_fedavg_learning_rate():
    """Select eta by deterministic validation-only pilot FedAvg runs.

    The algorithmic update is exactly canonical FedAvg: each selected client
    starts from the same global model, performs local mini-batch SGD, and the
    server computes a sample-size weighted model average. Only eta changes across
    candidates. The test split is never touched.
    """
    records = []
    candidate_final_losses = []
    pilot_selected_clients = fixed_reference_clients(FEDAVG_TUNING_CLIENTS)

    print("\n" + "=" * 88)
    print("FedAvg validation-only learning-rate search")
    print("Source: McMahan et al., AISTATS 2017, Algorithm 1 + LR grid-search principle")
    print("Candidates:", [float(x) for x in FEDAVG_LR_GRID])
    print(
        f"Pilot: clients={FEDAVG_TUNING_CLIENTS}, "
        f"rounds={FEDAVG_TUNING_ROUNDS}, local_epochs={LOCAL_EPOCHS}, batch={BATCH_SIZE}"
    )
    print("Selection split: validation only; test split is not evaluated.")
    print("=" * 88)

    for lr_index, candidate_lr in enumerate(FEDAVG_LR_GRID):
        candidate_lr = float(candidate_lr)

        # Reset random streams so every LR candidate sees the same stochastic
        # conditions and client selections.
        trial_seed = SEED + 88001
        random.seed(trial_seed)
        np.random.seed(trial_seed)
        torch.manual_seed(trial_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(trial_seed)

        global_model = clone_model(MODEL_REGISTRY["FedAvg"], DEVICE)
        failed = False
        final_val_loss = float("inf")

        try:
            for pilot_round in range(1, FEDAVG_TUNING_ROUNDS + 1):
                selected = pilot_selected_clients
                local_models, local_sizes = [], []

                for cid in selected:
                    local_model = train_local(
                        global_model,
                        get_client_loader(cid, pilot_round),
                        "FedAvg",
                        pilot_round,
                        fedavg_lr_override=candidate_lr,
                    )
                    local_model.to("cpu")
                    local_models.append(local_model)
                    local_sizes.append(len(client_indices[cid]))

                weights = np.asarray(local_sizes, dtype=np.float64)
                weights /= weights.sum() + 1e-12
                if not np.isclose(weights.sum(), 1.0, atol=1e-8):
                    raise RuntimeError("FedAvg pilot aggregation weights do not sum to 1.")

                global_model.to("cpu")
                global_model = average_models(local_models, weights)
                final_val_loss = fedavg_validation_objective(global_model)
                if not np.isfinite(final_val_loss):
                    failed = True
                    break

                records.append({
                    "LR_Index": lr_index,
                    "Learning_Rate": candidate_lr,
                    "Pilot_Round": pilot_round,
                    "Clients": FEDAVG_TUNING_CLIENTS,
                    "Local_Epochs": LOCAL_EPOCHS,
                    "Batch_Size": BATCH_SIZE,
                    "Validation_Multitask_Loss": final_val_loss,
                    "Status": "ok",
                })
                del local_models

        except (FloatingPointError, RuntimeError) as exc:
            failed = True
            final_val_loss = float("inf")
            records.append({
                "LR_Index": lr_index,
                "Learning_Rate": candidate_lr,
                "Pilot_Round": None,
                "Clients": FEDAVG_TUNING_CLIENTS,
                "Local_Epochs": LOCAL_EPOCHS,
                "Batch_Size": BATCH_SIZE,
                "Validation_Multitask_Loss": float("inf"),
                "Status": f"failed: {type(exc).__name__}",
            })

        candidate_final_losses.append(final_val_loss)
        print(
            f"eta={candidate_lr:.6g} | "
            f"final validation loss={'INF' if failed else f'{final_val_loss:.6f}'}"
        )
        global_model.to("cpu")
        del global_model
        cleanup_gpu()

    finite_mask = np.isfinite(np.asarray(candidate_final_losses, dtype=np.float64))
    if not finite_mask.any():
        raise RuntimeError(
            "All FedAvg learning-rate candidates failed. Widen/shift FEDAVG_LR_GRID."
        )

    best_idx = int(np.nanargmin(np.where(finite_mask, candidate_final_losses, np.nan)))
    best_lr = float(FEDAVG_LR_GRID[best_idx])

    if FEDAVG_REQUIRE_INTERIOR_BEST_LR and best_idx in (0, len(FEDAVG_LR_GRID) - 1):
        raise RuntimeError(
            "Best FedAvg learning rate lies on the search-grid boundary. "
            "McMahan et al. explicitly checked that the best rate was inside their "
            "grid. Expand FEDAVG_LR_GRID and rerun before reporting results."
        )

    tuning_df = pd.DataFrame(records)
    print(f"\nSelected FedAvg eta = {best_lr:.6g} (validation only)")
    return best_lr, tuning_df



# =========================================================
# Publication audit helpers
# =========================================================

def model_parameter_l2_distance(model_a, model_b):
    """L2 distance between two complete model parameter vectors."""
    squared = 0.0
    state_a = model_a.state_dict()
    state_b = model_b.state_dict()
    if state_a.keys() != state_b.keys():
        raise ValueError("Cannot compare models with different state dictionaries.")
    with torch.no_grad():
        for key in state_a:
            a = state_a[key].detach().cpu().float()
            b = state_b[key].detach().cpu().float()
            squared += float(torch.sum((a - b) ** 2).item())
    return math.sqrt(squared)


def proposed_cross_view_parameter_change(final_model, initial_model):
    """Verify that the supervisor-flagged cross-view branch changed in training."""
    groups = {
        "fusion_strength": ("fusion_strength",),
        "view_norm1": ("view_norm1",),
        "view_norm2": ("view_norm2",),
        "cross_gate": ("cross_gate",),
        "fusion_refine": ("fusion_refine",),
    }
    rows = []
    final_state = final_model.state_dict()
    initial_state = initial_model.state_dict()
    for group_name, prefixes in groups.items():
        squared = 0.0
        matched = 0
        for key in final_state:
            if any(key == p or key.startswith(p + ".") for p in prefixes):
                delta = (
                    final_state[key].detach().cpu().float()
                    - initial_state[key].detach().cpu().float()
                )
                squared += float(torch.sum(delta ** 2).item())
                matched += 1
        if matched == 0:
            raise RuntimeError(f"Cross-view parameter group not found: {group_name}")
        rows.append({
            "Parameter_Group": group_name,
            "Parameter_Change_L2": math.sqrt(squared),
        })
    return pd.DataFrame(rows)



def print_round_summary(row):
    """Print only the core round-level fields without notebook truncation."""
    preferred = [
        "Method", "Clients", "Round", "Local_Optimizer", "Local_LR",
        "BLEU1", "BLEU4", "ROUGE_L", "METEOR",
        "Label_F1_micro", "Label_F1_weighted",
        "Jaccard_micro", "Hamming_Loss",
        "Total_Comm_Cost_MB",
    ]
    available = [key for key in preferred if key in row]
    frame = pd.DataFrame([{key: row[key] for key in available}])
    numeric_cols = frame.select_dtypes(include=[np.number]).columns
    frame[numeric_cols] = frame[numeric_cols].round(6)
    print(frame.to_string(index=False))


# =========================================================
# 12) Federated experiments
# =========================================================
FINAL_MODELS = {}
FINAL_THRESHOLDS = {}
FINAL_TEST_ROWS = []

MODEL_CHECKPOINT_DIR = os.path.join(SEED_RUN_DIR, "model_checkpoints")
os.makedirs(MODEL_CHECKPOINT_DIR, exist_ok=True)
MODEL_CHECKPOINT_MANIFEST = []


def _safe_checkpoint_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(text)).strip("_")


def _checkpoint_should_save(method, ablation_variant, register_final):
    if not SAVE_MODEL_CHECKPOINTS:
        return False
    if register_final:
        return True
    if SAVE_ALL_ABLATION_CHECKPOINTS:
        return True
    return str(ablation_variant) in {"Full_Proposed", "w_o_Missing_Reconstruction"}


def save_model_checkpoint(model, method, clients, threshold, final_row, register_final=True):
    variant = str(final_row.get("Ablation_Variant", "Not_Applicable"))
    if not _checkpoint_should_save(method, variant, register_final):
        return ""

    kind = "main" if register_final else "ablation"
    filename = (
        f"seed_{int(SEED)}__{kind}__{_safe_checkpoint_name(method)}__"
        f"clients_{int(clients)}__{_safe_checkpoint_name(variant)}.pt"
    )
    path = os.path.join(MODEL_CHECKPOINT_DIR, filename)
    cpu_state = {
        k: v.detach().cpu().clone()
        for k, v in model.state_dict().items()
    }
    payload = {
        "academic_version": ACADEMIC_VERSION,
        "seed": int(SEED),
        "data_split_seed": int(DATA_SPLIT_SEED),
        "method": str(method),
        "clients": int(clients),
        "ablation_variant": variant,
        "ablation_config": copy.deepcopy(getattr(model, "ablation_config", None)),
        "calibrated_threshold": float(threshold),
        "model_class": model.__class__.__name__,
        "state_dict": cpu_state,
    }
    torch.save(payload, path)
    manifest_row = {
        "Seed": int(SEED), "Kind": kind, "Method": str(method),
        "Clients": int(clients), "Ablation_Variant": variant,
        "Path": path, "Bytes": int(os.path.getsize(path)),
    }
    MODEL_CHECKPOINT_MANIFEST.append(manifest_row)
    pd.DataFrame(MODEL_CHECKPOINT_MANIFEST).to_csv(
        os.path.join(MODEL_CHECKPOINT_DIR, f"checkpoint_manifest_seed_{int(SEED)}.csv"),
        index=False,
    )
    return path


def load_model_checkpoint(path, device="cpu"):
    try:
        payload = torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location="cpu")
    method = payload["method"]
    model = create_model(method).cpu()
    if hasattr(model, "ablation_config") and payload.get("ablation_config") is not None:
        model.ablation_config = normalize_ablation_config(payload["ablation_config"])
    model.load_state_dict(payload["state_dict"], strict=True)
    model.method_name = method
    model.calibrated_threshold = float(payload["calibrated_threshold"])
    return model.to(device), payload


def run_fedsw_experiment(selected_clients_count):
    """Validation-safe implementation of Chen & Pan (2024) Fedsw Algorithm 3.

    Key source logic retained:
      - local SGD;
      - score-proportional aggregation;
      - aggregate score s_agg;
      - clients with s_k < s_agg adopt aggregate;
      - clients with s_k >= s_agg retain their stronger local model.

    Publication correction:
      - score calculations use validation BLEU-1, never final-test data.
    """
    experiment_seed = SEED + 1000 * int(selected_clients_count)
    random.seed(experiment_seed)
    np.random.seed(experiment_seed)
    torch.manual_seed(experiment_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(experiment_seed)

    if FEDSW_STATIC_CLIENT_SET:
        participating_clients = fixed_reference_clients(selected_clients_count)
    else:
        participating_clients = select_clients("Fedsw", selected_clients_count, 1)

    # Persistent local states are required by Algorithm 3 because high-scoring
    # institutions may retain their own model rather than adopting s_agg's model.
    client_models = {
        cid: clone_model(MODEL_REGISTRY["Fedsw"], "cpu")
        for cid in participating_clients
    }
    global_model = clone_model(MODEL_REGISTRY["Fedsw"], "cpu")

    history = []
    cumulative_downlink_bytes = 0
    cumulative_uplink_bytes = 0
    communication_cost_bytes = 0
    dense_model_bytes = state_dict_nbytes(global_model.state_dict())

    # Algorithm 3 initialization: server sends w^0 to all participating devices.
    if COMM_COUNT_DOWNLINK:
        initial_downlink = dense_model_bytes * len(participating_clients)
        cumulative_downlink_bytes += initial_downlink
        communication_cost_bytes += initial_downlink

    last_local_scores = None
    last_agg_score = None
    last_update_count = 0
    last_retain_count = 0

    for round_id in range(1, ROUNDS + 1):
        selected = (
            participating_clients
            if FEDSW_STATIC_CLIENT_SET
            else select_clients("Fedsw", selected_clients_count, round_id)
        )
        print(
            f"\\nMethod=Fedsw | Clients={selected_clients_count} | "
            f"Round={round_id}/{ROUNDS} | score=validation BLEU-1"
        )

        local_models = []
        local_scores = []
        local_cids = []

        # All selected clients upload one locally trained dense FP32 model.
        round_uplink_bytes = dense_model_bytes * len(selected)

        for cid in selected:
            start_model = client_models.get(
                cid, clone_model(global_model, "cpu")
            )
            local_model = train_local(
                start_model,
                get_client_loader(cid, round_id),
                "Fedsw",
                round_id,
            )
            local_model.to("cpu")
            score = fedsw_validation_bleu1(local_model)

            local_models.append(local_model)
            local_scores.append(score)
            local_cids.append(cid)
            cleanup_gpu()

        candidate_model, score_weights = fedsw_score_weighted_aggregate(
            local_models, local_scores
        )
        candidate_model.to("cpu")
        aggregated_score = fedsw_validation_bleu1(candidate_model)

        # Algorithm 3 selective dissemination.
        update_clients = []
        retain_clients = []
        for cid, local_model, local_score in zip(
            local_cids, local_models, local_scores
        ):
            if float(local_score) < float(aggregated_score):
                client_models[cid] = clone_model(candidate_model, "cpu")
                update_clients.append(cid)
            else:
                client_models[cid] = clone_model(local_model, "cpu")
                retain_clients.append(cid)

        # Selective post-aggregation downlink: only clients that benefit from
        # the aggregate receive it, exactly reflecting the paper's Fedsw logic.
        round_selective_downlink_bytes = (
            dense_model_bytes * len(update_clients) if COMM_COUNT_DOWNLINK else 0
        )
        cumulative_uplink_bytes += int(round_uplink_bytes)
        cumulative_downlink_bytes += int(round_selective_downlink_bytes)
        communication_cost_bytes += int(
            round_uplink_bytes + round_selective_downlink_bytes
        )

        global_model = clone_model(candidate_model, "cpu")
        validation_metrics = evaluate(
            global_model,
            data_loader=val_loader,
            fixed_threshold=0.5,
        )

        last_local_scores = np.asarray(local_scores, dtype=np.float64)
        last_agg_score = float(aggregated_score)
        last_update_count = len(update_clients)
        last_retain_count = len(retain_clients)

        row = {
            "Seed": int(SEED),
            "Eval_Split": "validation",
            "Method": "Fedsw",
            "Clients": selected_clients_count,
            "Round": round_id,
            "Local_Optimizer": "SGD",
            "Local_LR": get_fedavg_learning_rate(),
            **validation_metrics,
            "TopK_Ratio": 1.0,
            "Mean_Tensor_TopK_Ratio": 1.0,
            "TopK_Retained_Elements": 0,
            "TopK_Total_Float_Elements": 0,
            "TopK_Target_Energy": 1.0,
            "TopK_Mode": "none",
            "Quantization_Bits": 32,
            "Downlink_Quantization_Bits": 32,
            "Initial_Downlink_Bits": 32,
            "Downlink_Mode": "dense_FP32_selective_post_aggregation",
            "Round_Downlink_MB": round_selective_downlink_bytes / (1024 ** 2),
            "Round_Uplink_MB": round_uplink_bytes / (1024 ** 2),
            "Cumulative_Downlink_MB": cumulative_downlink_bytes / (1024 ** 2),
            "Cumulative_Uplink_MB": cumulative_uplink_bytes / (1024 ** 2),
            "Selected_Clients": len(selected),
            "Num_Edges_Used": 0,
            "Adaptive_Edge_Count": 0,
            "Edge_Capacities": "{}",
            "Total_Comm_Cost_MB": communication_cost_bytes / (1024 ** 2),
            "Fedsw_Score_Split": FEDSW_SCORE_SPLIT,
            "Fedsw_Score_Metric": FEDSW_SCORE_METRIC,
            "Fedsw_Local_Score_Mean": float(np.mean(local_scores)),
            "Fedsw_Local_Score_Min": float(np.min(local_scores)),
            "Fedsw_Local_Score_Max": float(np.max(local_scores)),
            "Fedsw_Aggregated_Score": float(aggregated_score),
            "Fedsw_Clients_Updated": len(update_clients),
            "Fedsw_Clients_Retained": len(retain_clients),
            "Fedsw_Min_Weight": float(np.min(score_weights)),
            "Fedsw_Max_Weight": float(np.max(score_weights)),
        }
        history.append(row)
        print_round_summary(row)

        del local_models, candidate_model
        cleanup_gpu()

    # Final test remains untouched until all validation-controlled Fedsw decisions end.
    threshold = calibrate_threshold(global_model)
    global_model.calibrated_threshold = threshold
    test_metrics = evaluate(
        global_model,
        data_loader=test_loader,
        fixed_threshold=threshold,
    )

    final_model = clone_model(global_model, "cpu")
    final_model.calibrated_threshold = threshold
    FINAL_MODELS[("Fedsw", selected_clients_count)] = final_model
    FINAL_THRESHOLDS[("Fedsw", selected_clients_count)] = threshold

    final_row = {
        "Seed": int(SEED),
        "Eval_Split": "test",
        "Method": "Fedsw",
        "Clients": selected_clients_count,
        "Round": ROUNDS,
        "Expected_Final_Round": ROUNDS,
        "Local_Optimizer": "SGD",
        "Local_LR": get_fedavg_learning_rate(),
        **test_metrics,
        "TopK_Ratio": 1.0,
        "Mean_Tensor_TopK_Ratio": 1.0,
        "TopK_Retained_Elements": 0,
        "TopK_Total_Float_Elements": 0,
        "TopK_Target_Energy": 1.0,
        "TopK_Mode": "none",
        "Quantization_Bits": 32,
        "Downlink_Quantization_Bits": 32,
        "Initial_Downlink_Bits": 32,
        "Downlink_Mode": "dense_FP32_selective_post_aggregation",
        "Cumulative_Downlink_MB": cumulative_downlink_bytes / (1024 ** 2),
        "Cumulative_Uplink_MB": cumulative_uplink_bytes / (1024 ** 2),
        "Selected_Clients": len(participating_clients),
        "Num_Edges_Used": 0,
        "Adaptive_Edge_Count": 0,
        "Edge_Capacities": "{}",
        "Total_Comm_Cost_MB": communication_cost_bytes / (1024 ** 2),
        "Communication_Cost_Mode": (
            "Fedsw_initial_dense_downlink_plus_dense_uplink_"
            "plus_validation_score_selective_downlink"
        ),
        "Fedsw_Score_Split": FEDSW_SCORE_SPLIT,
        "Fedsw_Score_Metric": FEDSW_SCORE_METRIC,
        "Fedsw_Local_Score_Mean": (
            float(np.mean(last_local_scores)) if last_local_scores is not None else np.nan
        ),
        "Fedsw_Aggregated_Score": last_agg_score,
        "Fedsw_Clients_Updated": last_update_count,
        "Fedsw_Clients_Retained": last_retain_count,
        "Fedsw_Reference": FEDSW_REFERENCE,
    }
    checkpoint_path = save_model_checkpoint(
        final_model, "Fedsw", selected_clients_count, threshold, final_row, register_final=True
    )
    final_row["Model_Checkpoint"] = checkpoint_path
    FINAL_TEST_ROWS.append(final_row)

    global_model.to("cpu")
    cleanup_gpu()
    return pd.DataFrame(history)


def run_experiment(method, selected_clients_count, ablation_config=None, register_final=True):
    if method == "Fedsw":
        if ablation_config is not None:
            raise ValueError("Ablation configuration is only defined for the proposed method.")
        return run_fedsw_experiment(selected_clients_count)

    cfg = normalize_ablation_config(ablation_config) if method == "Proposed_CoHiM_FL" else None

    # Each independent run starts from a deterministic RNG state. This makes a
    # repeated full-notebook execution reproducible and prevents experiment-order
    # effects from changing stochastic augmentation/dropout streams.
    experiment_seed = SEED + 1000 * int(selected_clients_count)
    random.seed(experiment_seed)
    np.random.seed(experiment_seed)
    torch.manual_seed(experiment_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(experiment_seed)

    global_model = clone_model(MODEL_REGISTRY[method], DEVICE)
    if method == "Proposed_CoHiM_FL":
        global_model.ablation_config = copy.deepcopy(cfg)
    history = []
    communication_cost_bytes = 0
    cumulative_downlink_bytes = 0
    cumulative_uplink_bytes = 0
    selection_counts = {}
    server_velocity = None
    last_selected_count = 0
    last_edges_used = 0
    last_adaptive_edge_count = 0
    last_edge_capacities = {}

    # Cache and error-feedback states are reset for every independent
    # method/client-count experiment to prevent cross-experiment leakage.
    client_downlink_cache = {}
    client_downlink_residual = {}
    client_uplink_residual = {}

    last_effective_topk_ratio = 1.0
    last_mean_tensor_topk_ratio = 1.0
    last_topk_retained_elements = 0
    last_topk_total_elements = 0
    last_topk_mode = "none"

    reference_fixed_clients = (
        fixed_reference_clients(selected_clients_count)
        if method in ("FedAvg", "FedProx", "CAR-MFL") and FEDSW_STATIC_CLIENT_SET
        else None
    )

    for round_id in range(1, ROUNDS + 1):
        if method == "Proposed_CoHiM_FL":
            selected, selected_edge_groups, adaptive_edge_state = select_clients(
                method,
                selected_clients_count,
                round_id,
                selection_counts=selection_counts,
                return_edges=True,
                ablation_config=cfg,
            )
        else:
            selected = (
                reference_fixed_clients
                if reference_fixed_clients is not None
                else select_clients(method, selected_clients_count, round_id)
            )
            selected_edge_groups = None
            adaptive_edge_state = None

        last_selected_count = len(selected)
        last_edges_used = (
            len([x for x in selected_edge_groups.values() if x])
            if selected_edge_groups else 0
        )
        last_adaptive_edge_count = (
            adaptive_edge_state["active_edges"]
            if adaptive_edge_state else 0
        )
        last_edge_capacities = (
            adaptive_edge_state["capacities"]
            if adaptive_edge_state else {}
        )

        local_models, local_sizes, local_cids = [], [], []
        global_model.to("cpu")
        dense_model_bytes = state_dict_nbytes(global_model.state_dict())

        # Baselines exchange dense FP32 models in both directions.
        # The proposed method uses a separate cached model for each client.
        round_reference_model = global_model
        round_downlink_bytes = 0
        round_uplink_bytes = 0
        round_topk_retained_elements = 0
        round_topk_total_elements = 0
        round_tensor_ratios = []
        round_topk_mode = "none"

        if method != "Proposed_CoHiM_FL":
            dense_receivers = len(selected)
            if method == "CAR-MFL" and CAR_USE_PUBLIC_VIRTUAL_CLIENT:
                dense_receivers += 1
            round_downlink_bytes = (
                dense_model_bytes * dense_receivers if COMM_COUNT_DOWNLINK else 0
            )

        car_retrieval_state = build_car_retrieval_state(global_model) if method == "CAR-MFL" else None

        print(f"\nMethod={method} | Clients={selected_clients_count} | Round={round_id}/{ROUNDS}")

        for cid in selected:
            if method == "Proposed_CoHiM_FL":
                (
                    client_reference_model,
                    client_downlink_bytes,
                    updated_cache,
                    updated_residual,
                ) = transmit_cached_downlink(
                    global_model,
                    cached_state=client_downlink_cache.get(cid),
                    residual_state=client_downlink_residual.get(cid),
                    initial_bits=PROPOSED_INITIAL_DOWNLINK_BITS,
                    delta_bits=PROPOSED_DELTA_DOWNLINK_BITS,
                    use_error_feedback=cfg["error_feedback"],
                    quantize_delta=cfg["int8_quantization"],
                )

                client_downlink_cache[cid] = updated_cache
                client_downlink_residual[cid] = updated_residual

                if COMM_COUNT_DOWNLINK:
                    round_downlink_bytes += client_downlink_bytes
            else:
                client_reference_model = global_model

            local_loader = (
                get_car_client_loader(cid, round_id, global_model, car_retrieval_state)
                if method == "CAR-MFL"
                else get_client_loader(cid, round_id)
            )
            local_model = train_local(
                client_reference_model, local_loader, method, round_id
            )
            local_model.to("cpu")

            if method == "Proposed_CoHiM_FL":
                # Actual adaptive Top-K uplink:
                # sparsify the error-corrected client delta, quantize retained
                # values to INT8, reconstruct it, then aggregate that
                # reconstructed sparse update at the edge/server.
                (
                    transmitted_model,
                    payload_bytes,
                    updated_uplink_residual,
                    topk_stats,
                ) = adaptive_topk_quantize_model_update(
                    client_reference_model.to("cpu"),
                    local_model,
                    residual_state=client_uplink_residual.get(cid),
                    round_id=round_id,
                    bits=PROPOSED_UPDATE_QUANT_BITS,
                    use_error_feedback=cfg["error_feedback"],
                    use_topk=cfg["adaptive_topk"],
                    quantize_int8=cfg["int8_quantization"],
                )
                client_uplink_residual[cid] = updated_uplink_residual
                local_models.append(transmitted_model)
                round_uplink_bytes += payload_bytes
                round_topk_retained_elements += topk_stats["Retained_Elements"]
                round_topk_total_elements += topk_stats["Total_Float_Elements"]
                round_tensor_ratios.append(
                    topk_stats["Mean_Tensor_TopK_Ratio"]
                )
                round_topk_mode = topk_stats["TopK_Mode"]
            else:
                local_models.append(local_model)
                round_uplink_bytes += dense_model_bytes

            local_sizes.append(
                len(CAR_PRIVATE_CLIENT_INDICES[cid]) if method == "CAR-MFL"
                else len(client_indices[cid])
            )
            local_cids.append(cid)
            cleanup_gpu()

        # Source paper: public multimodal data are also trained as a virtual client.
        if method == "CAR-MFL" and CAR_USE_PUBLIC_VIRTUAL_CLIENT:
            public_local_model = train_local(
                global_model, get_car_public_loader(round_id), method, round_id
            )
            public_local_model.to("cpu")
            local_models.append(public_local_model)
            local_sizes.append(int(len(CAR_PUBLIC_INDICES)))
            local_cids.append(-1)
            round_uplink_bytes += dense_model_bytes
            cleanup_gpu()

        if method == "FedAvg":
            # Canonical FedAvg server step: sample-size weighted model averaging.
            # w_{t+1} = sum_k (n_k / sum_j n_j) * w_{t+1}^k
            weights = np.asarray(local_sizes, dtype=np.float64)
            weights /= weights.sum() + 1e-12
            assert np.isclose(weights.sum(), 1.0, atol=1e-8)
            candidate_model = average_models(local_models, weights)

        elif method in ["FedProx", "CAR-MFL"]:
            weights = np.asarray(local_sizes, dtype=np.float64)
            weights /= weights.sum() + 1e-12
            candidate_model = average_models(local_models, weights)
            # Poudel et al. additionally re-adjust complementary-modality ENCODER
            # weights with alpha=0.3. The controlled report-generation backbone
            # used here has a shared visual projection, not separate modality
            # encoders; scaling the entire model would be a different algorithm.
            # This limitation is explicitly exported in the CAR audit.

        else:
            if cfg["hierarchical_aggregation"]:
                local_by_cid = dict(zip(local_cids, local_models))
                size_by_cid = dict(zip(local_cids, local_sizes))
                edge_models, edge_sizes = [], []

                for edge_id, edge_cids in selected_edge_groups.items():
                    edge_cids = [cid for cid in edge_cids if cid in local_by_cid]
                    if not edge_cids:
                        continue
                    edge_model = robust_aggregate_updates(
                        global_model,
                        [local_by_cid[cid] for cid in edge_cids],
                        [size_by_cid[cid] for cid in edge_cids],
                        client_ids=edge_cids,
                    )
                    edge_model.to("cpu")
                    edge_models.append(edge_model)
                    edge_sizes.append(sum(size_by_cid[cid] for cid in edge_cids))

                if edge_models:
                    cloud_weights = np.asarray(edge_sizes, dtype=np.float64)
                    cloud_weights /= cloud_weights.sum() + 1e-12
                    candidate_model = average_models(edge_models, cloud_weights)
                else:
                    candidate_model = robust_aggregate_updates(
                        global_model, local_models, local_sizes, client_ids=local_cids
                    )
            else:
                # Ablation: direct cloud aggregation of the same selected client
                # updates, preserving the robust aggregation rule and server momentum.
                candidate_model = robust_aggregate_updates(
                    global_model, local_models, local_sizes, client_ids=local_cids
                )

            candidate_model, server_velocity = apply_server_momentum(
                global_model, candidate_model, server_velocity
            )

        global_model.to("cpu")
        global_model = candidate_model
        cleanup_gpu()

        if method == "Proposed_CoHiM_FL":
            last_effective_topk_ratio = (
                float(round_topk_retained_elements / round_topk_total_elements)
                if round_topk_total_elements > 0 else 1.0
            )
            last_mean_tensor_topk_ratio = (
                float(np.mean(round_tensor_ratios))
                if round_tensor_ratios else 1.0
            )
            last_topk_retained_elements = int(round_topk_retained_elements)
            last_topk_total_elements = int(round_topk_total_elements)
            last_topk_mode = round_topk_mode
        else:
            last_effective_topk_ratio = 1.0
            last_mean_tensor_topk_ratio = 1.0
            last_topk_retained_elements = 0
            last_topk_total_elements = 0
            last_topk_mode = "none"

        cumulative_downlink_bytes += int(round_downlink_bytes)
        cumulative_uplink_bytes += int(round_uplink_bytes)
        communication_cost_bytes += int(round_downlink_bytes + round_uplink_bytes)
        communication_cost = communication_cost_bytes / (1024 ** 2)

        # Round monitoring is performed on validation data, never on test data.
        validation_metrics = evaluate(
            global_model,
            data_loader=val_loader,
            fixed_threshold=0.5,
        )
        history.append({
            "Seed": int(SEED),
            "Eval_Split": "validation",
            "Method": method,
            "Ablation_Variant": cfg["variant"] if cfg is not None else "Not_Applicable",
            "Adaptive_Client_Selection": cfg["adaptive_client_selection"] if cfg is not None else np.nan,
            "Hierarchical_Aggregation": cfg["hierarchical_aggregation"] if cfg is not None else np.nan,
            "Adaptive_TopK": cfg["adaptive_topk"] if cfg is not None else np.nan,
            "INT8_Quantization": cfg["int8_quantization"] if cfg is not None else np.nan,
            "Error_Feedback": cfg["error_feedback"] if cfg is not None else np.nan,
            "Missing_Reconstruction": cfg["missing_reconstruction"] if cfg is not None else np.nan,
            "Clients": selected_clients_count,
            "Round": round_id,
            "Local_Optimizer": local_optimizer_name(method),
            "Local_LR": local_optimizer_lr(method, round_id),
            **validation_metrics,
            "TopK_Ratio": last_effective_topk_ratio,
            "Mean_Tensor_TopK_Ratio": last_mean_tensor_topk_ratio,
            "TopK_Retained_Elements": last_topk_retained_elements,
            "TopK_Total_Float_Elements": last_topk_total_elements,
            "TopK_Target_Energy": (
                ADAPTIVE_TOPK_TARGET_ENERGY
                if method == "Proposed_CoHiM_FL" and cfg["adaptive_topk"]
                else 1.0
            ),
            "TopK_Mode": (last_topk_mode if method == "Proposed_CoHiM_FL" else "none"),
            "Quantization_Bits": (
                (8 if cfg["int8_quantization"] else 32)
                if method == "Proposed_CoHiM_FL" else 32
            ),
            "Downlink_Quantization_Bits": (
                (8 if cfg["int8_quantization"] else 32)
                if method == "Proposed_CoHiM_FL" else 32
            ),
            "Initial_Downlink_Bits": (
                PROPOSED_INITIAL_DOWNLINK_BITS
                if method == "Proposed_CoHiM_FL" else 32
            ),
            "Downlink_Mode": (
                ("FP16_initial_then_" + ("INT8" if cfg["int8_quantization"] else "FP32")
                 + "_cached_delta_" + ("with" if cfg["error_feedback"] else "without")
                 + "_error_feedback")
                if method == "Proposed_CoHiM_FL" else "dense_FP32"
            ),
            "Round_Downlink_MB": round_downlink_bytes / (1024 ** 2),
            "Round_Uplink_MB": round_uplink_bytes / (1024 ** 2),
            "Cumulative_Downlink_MB": cumulative_downlink_bytes / (1024 ** 2),
            "Cumulative_Uplink_MB": cumulative_uplink_bytes / (1024 ** 2),
            "Selected_Clients": len(selected),
            "Num_Edges_Used": len([x for x in selected_edge_groups.values() if x]) if selected_edge_groups else 0,
            "Adaptive_Edge_Count": (
                adaptive_edge_state["active_edges"]
                if adaptive_edge_state else 0
            ),
            "Edge_Capacities": (
                str(adaptive_edge_state["capacities"])
                if adaptive_edge_state else "{}"
            ),
            "Total_Comm_Cost_MB": communication_cost,
        })
        print_round_summary(history[-1])

        del local_models
        cleanup_gpu()

    # Final model is the state after Round=ROUNDS. Only the classification threshold
    # is calibrated on validation data; no model weights are selected using test scores.
    threshold = calibrate_threshold(global_model)
    global_model.calibrated_threshold = threshold
    test_metrics = evaluate(
        global_model,
        data_loader=test_loader,
        fixed_threshold=threshold,
    )

    final_model = clone_model(global_model, "cpu")
    final_model.calibrated_threshold = threshold
    final_row = {
        "Seed": int(SEED),
        "Eval_Split": "test",
        "Ablation_Variant": cfg["variant"] if cfg is not None else "Not_Applicable",
        "Adaptive_Client_Selection": cfg["adaptive_client_selection"] if cfg is not None else np.nan,
        "Hierarchical_Aggregation": cfg["hierarchical_aggregation"] if cfg is not None else np.nan,
        "Adaptive_TopK": cfg["adaptive_topk"] if cfg is not None else np.nan,
        "INT8_Quantization": cfg["int8_quantization"] if cfg is not None else np.nan,
        "Error_Feedback": cfg["error_feedback"] if cfg is not None else np.nan,
        "Missing_Reconstruction": cfg["missing_reconstruction"] if cfg is not None else np.nan,
        "Method": method,
        "Clients": selected_clients_count,
        "Round": ROUNDS,
        "Expected_Final_Round": ROUNDS,
        "Local_Optimizer": local_optimizer_name(method),
        "Local_LR": local_optimizer_lr(method, ROUNDS),
        **test_metrics,
        "TopK_Ratio": last_effective_topk_ratio,
            "Mean_Tensor_TopK_Ratio": last_mean_tensor_topk_ratio,
            "TopK_Retained_Elements": last_topk_retained_elements,
            "TopK_Total_Float_Elements": last_topk_total_elements,
            "TopK_Target_Energy": (
                ADAPTIVE_TOPK_TARGET_ENERGY
                if method == "Proposed_CoHiM_FL" and cfg["adaptive_topk"]
                else 1.0
            ),
            "TopK_Mode": (last_topk_mode if method == "Proposed_CoHiM_FL" else "none"),
        "Quantization_Bits": (
            (8 if cfg["int8_quantization"] else 32)
            if method == "Proposed_CoHiM_FL" else 32
        ),
        "Downlink_Quantization_Bits": (
            (8 if cfg["int8_quantization"] else 32)
            if method == "Proposed_CoHiM_FL" else 32
        ),
        "Initial_Downlink_Bits": (
            PROPOSED_INITIAL_DOWNLINK_BITS
            if method == "Proposed_CoHiM_FL" else 32
        ),
        "Downlink_Mode": (
            (("FP16_initial_then_" + ("INT8" if cfg["int8_quantization"] else "FP32") + "_cached_delta_" + ("with" if cfg["error_feedback"] else "without") + "_error_feedback"))
            if method == "Proposed_CoHiM_FL"
            else "dense_FP32"
        ),
        "Cumulative_Downlink_MB": cumulative_downlink_bytes / (1024 ** 2),
        "Cumulative_Uplink_MB": cumulative_uplink_bytes / (1024 ** 2),
        "Selected_Clients": last_selected_count,
        "Num_Edges_Used": last_edges_used,
        "Adaptive_Edge_Count": last_adaptive_edge_count,
        "Edge_Capacities": str(last_edge_capacities),
        "Total_Comm_Cost_MB": communication_cost,
        "Communication_Cost_Mode": (
            ("measured_ablation_aware_cached_downlink_plus_uplink_payload")
            if method == "Proposed_CoHiM_FL"
            else "measured_dense_fp32_uplink_and_downlink"
        ),
    }

    checkpoint_path = save_model_checkpoint(
        final_model,
        method,
        selected_clients_count,
        threshold,
        final_row,
        register_final=register_final,
    )
    final_row["Model_Checkpoint"] = checkpoint_path

    if register_final:
        FINAL_MODELS[(method, selected_clients_count)] = final_model
        FINAL_THRESHOLDS[(method, selected_clients_count)] = threshold
        FINAL_TEST_ROWS.append(final_row)


    global_model.to("cpu")
    cleanup_gpu()
    history_df = pd.DataFrame(history)
    history_df.attrs["final_test_row"] = copy.deepcopy(final_row)
    return history_df


# ---------------------------------------------------------
# Mandatory pre-experiment gradient audit
# ---------------------------------------------------------
if RUN_CROSS_VIEW_GRADIENT_AUDIT:
    CROSS_VIEW_GRADIENT_AUDIT_DF = run_cross_view_gradient_audit()
    audit_dir = os.path.join(SEED_RUN_DIR, "audits")
    os.makedirs(audit_dir, exist_ok=True)
    CROSS_VIEW_GRADIENT_AUDIT_DF.to_csv(
        os.path.join(audit_dir, "cross_view_gradient_audit_v9_chenpan_fedsw.csv"),
        index=False,
    )
else:
    print("WARNING: cross-view gradient audit is disabled.")

# ---------------------------------------------------------
# Canonical FedAvg learning-rate selection (validation only)
# ---------------------------------------------------------
if RUN_FEDAVG_LR_TUNING:
    FEDAVG_SELECTED_LR, FEDAVG_LR_TUNING_DF = tune_fedavg_learning_rate()
    tuning_dir = os.path.join(SEED_RUN_DIR, "audits")
    os.makedirs(tuning_dir, exist_ok=True)
    FEDAVG_LR_TUNING_DF.to_csv(
        os.path.join(tuning_dir, "fedavg_validation_lr_search_v9_chenpan.csv"),
        index=False,
    )
else:
    FEDAVG_SELECTED_LR = float(FEDAVG_FALLBACK_LR)
    FEDAVG_LR_TUNING_DF = pd.DataFrame()
    print(
        f"WARNING: FedAvg LR tuning disabled; fallback eta={FEDAVG_SELECTED_LR:.6g} is used."
    )

print(f"Frozen FedAvg learning rate for final experiments: {FEDAVG_SELECTED_LR:.6g}")

# =========================================================
# 13) SINGLE-SEED PUBLICATION RUN
# =========================================================
# This notebook executes exactly one pre-specified stochastic seed.
# The five independent seed notebooks are combined only in the separate
# aggregation notebook. No best-seed selection is permitted.
# =========================================================

FINAL_MODELS.clear()
FINAL_THRESHOLDS.clear()
FINAL_TEST_ROWS.clear()
CAR_RETRIEVAL_AUDIT_ROWS.clear()

if len(EXPERIMENT_SEEDS) != 1:
    raise RuntimeError(
        f"This split-run notebook must contain exactly one seed; found {EXPERIMENT_SEEDS}."
    )

RUN_SEED = int(EXPERIMENT_SEEDS[0])
configure_experiment_seed(RUN_SEED, verbose=True)

seed_histories = []
for client_count in CLIENT_SETTINGS:
    for method in METHODS:
        run_history = run_experiment(method, client_count)
        seed_histories.append(run_history)

results_df = pd.concat(seed_histories, ignore_index=True)
final_test_seed = pd.DataFrame(FINAL_TEST_ROWS).copy()

expected_rows = len(CLIENT_SETTINGS) * len(METHODS)
if len(final_test_seed) != expected_rows:
    raise RuntimeError(
        f"Seed {RUN_SEED}: expected {expected_rows} final-test rows, "
        f"found {len(final_test_seed)}. Results are incomplete and must not be merged."
    )

if set(final_test_seed["Seed"].astype(int).unique()) != {RUN_SEED}:
    raise RuntimeError("Seed provenance audit failed in final_test_seed.")

# Compute communication reduction within this seed and each client setting.
fedavg_cost = (
    final_test_seed[final_test_seed["Method"] == "FedAvg"]
    [["Clients", "Total_Comm_Cost_MB"]]
    .rename(columns={"Total_Comm_Cost_MB": "Baseline_Cost_MB"})
)
final_test_seed = final_test_seed.merge(
    fedavg_cost, on="Clients", how="left", validate="many_to_one"
)
final_test_seed["Cost_Reduction_%"] = (
    1.0 - final_test_seed["Total_Comm_Cost_MB"]
    / final_test_seed["Baseline_Cost_MB"]
) * 100.0

# Seed-isolated output directory. This prevents one run from overwriting another.
SEED_RESULTS_DIR = os.path.join(SEED_RUN_DIR, "main_results")
os.makedirs(SEED_RESULTS_DIR, exist_ok=True)

final_path = os.path.join(SEED_RESULTS_DIR, f"final_test_seed_{RUN_SEED}.csv")
history_path = os.path.join(SEED_RESULTS_DIR, f"validation_history_seed_{RUN_SEED}.csv")
workbook_path = os.path.join(SEED_RESULTS_DIR, f"seed_{RUN_SEED}_main_results.xlsx")

final_test_seed.to_csv(final_path, index=False)
results_df.to_csv(history_path, index=False)
with pd.ExcelWriter(workbook_path, engine="openpyxl") as writer:
    final_test_seed.to_excel(writer, sheet_name="Final_Test", index=False)
    results_df.to_excel(writer, sheet_name="Validation_History", index=False)

print("\n" + "=" * 100)
print(f"SEED {RUN_SEED} MAIN EXPERIMENTS COMPLETED")
print("=" * 100)
print(f"Final-test rows: {len(final_test_seed)} / {expected_rows}")
print("Saved:", final_path)
print("Saved:", history_path)
print("Saved:", workbook_path)
display(final_test_seed.sort_values(["Clients", "Method"]).reset_index(drop=True))


In [ ]:
# =========================================================
# 13A) Generated-Report Clinical Correctness via REAL CheXbert
# =========================================================
# This analysis evaluates the GENERATED TEXT itself, rather than the auxiliary
# image-label classifier. Reference labels are the fixed CheXbert labels already
# derived from the original held-out test reports. Generated reports are passed
# through the same CheXbert model/mapping, then compared study-by-study.
# =========================================================

from sklearn.metrics import precision_score, recall_score

RUN_GENERATED_REPORT_CHEXBERT = True
CLINICAL_DIR = os.path.join(SEED_RUN_DIR, "generated_report_clinical")
os.makedirs(CLINICAL_DIR, exist_ok=True)


def _clinical_metrics(reference_labels, generated_labels):
    reference_labels = np.asarray(reference_labels, dtype=int)
    generated_labels = np.asarray(generated_labels, dtype=int)
    if reference_labels.shape != generated_labels.shape:
        raise RuntimeError(
            f"Clinical label shape mismatch: {reference_labels.shape} vs {generated_labels.shape}"
        )
    return {
        "Generated_Report_CheXbert_Precision_micro": precision_score(
            reference_labels, generated_labels, average="micro", zero_division=0
        ),
        "Generated_Report_CheXbert_Recall_micro": recall_score(
            reference_labels, generated_labels, average="micro", zero_division=0
        ),
        "Generated_Report_CheXbert_F1_micro": f1_score(
            reference_labels, generated_labels, average="micro", zero_division=0
        ),
        "Generated_Report_CheXbert_F1_macro": f1_score(
            reference_labels, generated_labels, average="macro", zero_division=0
        ),
        "Generated_Report_CheXbert_F1_weighted": f1_score(
            reference_labels, generated_labels, average="weighted", zero_division=0
        ),
        "Generated_Report_CheXbert_Jaccard_micro": jaccard_score(
            reference_labels, generated_labels, average="micro", zero_division=0
        ),
        "Generated_Report_CheXbert_Hamming_Loss": hamming_loss(
            reference_labels, generated_labels
        ),
        "Generated_Report_CheXbert_Exact_Match": float(
            np.mean(np.all(reference_labels == generated_labels, axis=1))
        ),
    }


if RUN_GENERATED_REPORT_CHEXBERT:
    expected_keys = [
        (method, int(clients))
        for clients in CLIENT_SETTINGS
        for method in METHODS
    ]
    missing_model_keys = [k for k in expected_keys if k not in FINAL_MODELS]
    if missing_model_keys:
        raise RuntimeError(
            "Generated-report CheXbert requires all final main models in memory. "
            f"Missing: {missing_model_keys}"
        )

    generated_records = []
    generated_texts_all = []
    reference_test_labels = np.vstack(test_df["labels"].values).astype(int)
    reference_uids = test_df["uid"].astype(str).tolist()

    for clients in CLIENT_SETTINGS:
        for method in METHODS:
            key = (method, int(clients))
            model = FINAL_MODELS[key]
            threshold = FINAL_THRESHOLDS[key]
            eval_out = evaluate(
                model,
                data_loader=test_loader,
                fixed_threshold=threshold,
                missing_rate=0.0,
                seed=DATA_SPLIT_SEED,
                missing_view="lateral",
                return_texts=True,
            )
            text_bundle = eval_out.pop("_texts")
            generated = text_bundle["generated_reports"]
            if len(generated) != len(test_df):
                raise RuntimeError(
                    f"Generated report count mismatch for {method}/{clients}: "
                    f"{len(generated)} != {len(test_df)}"
                )

            # Re-evaluation consistency audit against the main final-test CSV row.
            saved = final_test_seed[
                (final_test_seed["Method"] == method)
                & (final_test_seed["Clients"].astype(int) == int(clients))
            ]
            if len(saved) != 1:
                raise RuntimeError(f"Main final row not uniquely found for {method}/{clients}")
            for metric in ["BLEU1", "BLEU4", "ROUGE_L", "METEOR"]:
                if not np.isclose(
                    float(eval_out[metric]), float(saved.iloc[0][metric]),
                    rtol=1e-8, atol=1e-8,
                ):
                    raise RuntimeError(
                        f"Deterministic re-evaluation audit failed for {method}/{clients}/{metric}."
                    )

            start = len(generated_texts_all)
            generated_texts_all.extend(generated)
            for local_i, (uid, text) in enumerate(zip(reference_uids, generated)):
                generated_records.append({
                    "Seed": int(RUN_SEED),
                    "Method": method,
                    "Clients": int(clients),
                    "Test_Row": int(local_i),
                    "UID": uid,
                    "Generated_Report": text,
                    "CheXbert_Global_Row": int(start + local_i),
                })

            try:
                model.to("cpu")
            except Exception:
                pass
            cleanup_gpu()

    generated_reports_table = pd.DataFrame(generated_records)
    generated_binary_all, _raw_chexbert_generated, generated_chexbert_meta = run_chexbert_on_texts(
        generated_texts_all,
        tag=f"generated_reports_seed_{RUN_SEED}",
        cache_dir=CLINICAL_DIR,
    )

    if len(generated_binary_all) != len(generated_reports_table):
        raise RuntimeError("Generated-report CheXbert output alignment failed.")

    for j, label_name in enumerate(chexbert_cols):
        generated_reports_table[f"Gen_{label_name}"] = generated_binary_all[:, j].astype(int)

    clinical_rows = []
    for clients in CLIENT_SETTINGS:
        for method in METHODS:
            group = generated_reports_table[
                (generated_reports_table["Method"] == method)
                & (generated_reports_table["Clients"].astype(int) == int(clients))
            ].sort_values("Test_Row")
            idx = group["CheXbert_Global_Row"].astype(int).to_numpy()
            generated_labels = generated_binary_all[idx].astype(int)
            row = {
                "Seed": int(RUN_SEED),
                "Method": method,
                "Clients": int(clients),
                "N_Test_Studies": int(len(group)),
                "Clinical_Evaluation_Source": "Generated report -> REAL CheXbert",
                "Reference_Labels": "Original held-out reference report -> REAL CheXbert",
                "CheXbert_Positive_Mapping": "state==1",
                **_clinical_metrics(reference_test_labels, generated_labels),
            }
            clinical_rows.append(row)

    generated_report_clinical_seed = pd.DataFrame(clinical_rows)
    expected_clinical_rows = len(METHODS) * len(CLIENT_SETTINGS)
    if len(generated_report_clinical_seed) != expected_clinical_rows:
        raise RuntimeError(
            f"Clinical evaluation incomplete: {len(generated_report_clinical_seed)} "
            f"!= {expected_clinical_rows}"
        )

    reports_path = os.path.join(
        CLINICAL_DIR, f"generated_reports_with_chexbert_seed_{RUN_SEED}.csv"
    )
    summary_path = os.path.join(
        CLINICAL_DIR, f"generated_report_chexbert_seed_{RUN_SEED}.csv"
    )
    xlsx_path = os.path.join(
        CLINICAL_DIR, f"generated_report_clinical_seed_{RUN_SEED}.xlsx"
    )
    generated_reports_table.to_csv(reports_path, index=False)
    generated_report_clinical_seed.to_csv(summary_path, index=False)
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        generated_report_clinical_seed.to_excel(writer, sheet_name="Clinical_Summary", index=False)
        generated_reports_table.to_excel(writer, sheet_name="Generated_Reports", index=False)

    print("\nGENERATED-REPORT CHEXBERT CLINICAL EVALUATION: COMPLETED")
    display(generated_report_clinical_seed.sort_values(["Clients", "Method"]).reset_index(drop=True))
    print("Saved:", summary_path)
    print("Saved:", reports_path)
else:
    print("Generated-report CheXbert evaluation disabled.")


## Component-wise ablation study (five seeds)

This analysis removes **one proposed component at a time** while holding the remaining training protocol fixed. The full proposed method is re-run under the same pre-specified seeds so every component comparison is paired by seed. The default ablation setting uses **20 participating clients**, specified before test evaluation, to control computational cost without selecting a configuration based on test performance.

Components evaluated: adaptive client selection, hierarchical edge–cloud aggregation, adaptive Top-K sparsification, INT8 quantization, error feedback, and missing-view/cross-view reconstruction. Results are reported as **mean ± sample SD across five seeds**, together with the change in BLEU-1, METEOR, and communication cost relative to the full proposed method.


In [ ]:
# =========================================================
# 14) SINGLE-SEED COMPONENT-WISE ABLATION STUDY
# =========================================================
# Scientific protocol:
# - exactly one component is disabled per variant;
# - all variants use the same seed, fixed data split, training budget, and test set;
# - communication cost is measured from the actual payload path;
# - the separate merge notebook computes Mean ± sample SD across all five seeds.
# =========================================================

ABLATION_VARIANTS = {
    "Full_Proposed": {},
    "w_o_Client_Selection": {"adaptive_client_selection": False},
    "w_o_Hierarchical": {"hierarchical_aggregation": False},
    "w_o_Adaptive_TopK": {"adaptive_topk": False},
    "w_o_INT8": {"int8_quantization": False},
    "w_o_Error_Feedback": {"error_feedback": False},
    "w_o_Missing_Reconstruction": {"missing_reconstruction": False},
}

RUN_SEED = int(EXPERIMENT_SEEDS[0])
ablation_rows = []
ablation_histories = []
ABLATION_DIR = os.path.join(SEED_RUN_DIR, "ablation_results")
os.makedirs(ABLATION_DIR, exist_ok=True)

if RUN_ABLATION_STUDY:
    configure_experiment_seed(RUN_SEED, verbose=True)

    for clients in ABLATION_CLIENT_SETTINGS:
        for variant_name, overrides in ABLATION_VARIANTS.items():
            cfg = normalize_ablation_config({"variant": variant_name, **overrides})
            print("\n" + "#" * 90)
            print(f"ABLATION | Seed={RUN_SEED} | Clients={clients} | Variant={variant_name}")
            print("#" * 90)

            hist = run_experiment(
                "Proposed_CoHiM_FL",
                clients,
                ablation_config=cfg,
                register_final=False,
            )
            final_row_for_variant = copy.deepcopy(hist.attrs["final_test_row"])
            hist = hist.copy()
            hist["Ablation_Variant"] = variant_name
            ablation_histories.append(hist)

            row = final_row_for_variant
            row.update({
                "Seed": RUN_SEED,
                "Clients": int(clients),
                "Ablation_Variant": variant_name,
                "Adaptive_Client_Selection": bool(cfg["adaptive_client_selection"]),
                "Hierarchical_Aggregation": bool(cfg["hierarchical_aggregation"]),
                "Adaptive_TopK": bool(cfg["adaptive_topk"]),
                "INT8_Quantization": bool(cfg["int8_quantization"]),
                "Error_Feedback": bool(cfg["error_feedback"]),
                "Missing_Reconstruction": bool(cfg["missing_reconstruction"]),
            })
            ablation_rows.append(row)

            # Immediate seed-local checkpoint.
            pd.DataFrame(ablation_rows).to_csv(
                os.path.join(ABLATION_DIR, f"ablation_progress_seed_{RUN_SEED}.csv"),
                index=False,
            )

    ablation_raw_seed = pd.DataFrame(ablation_rows)
    ablation_history_seed = pd.concat(ablation_histories, ignore_index=True)

    expected_ablation_rows = len(ABLATION_CLIENT_SETTINGS) * len(ABLATION_VARIANTS)
    if len(ablation_raw_seed) != expected_ablation_rows:
        raise RuntimeError(
            f"Seed {RUN_SEED}: ablation incomplete; expected {expected_ablation_rows} rows, "
            f"found {len(ablation_raw_seed)}."
        )

    # Paired component contribution relative to Full Proposed within the same seed.
    reference = (
        ablation_raw_seed[ablation_raw_seed["Ablation_Variant"] == "Full_Proposed"]
        [["Seed", "Clients", "BLEU1", "METEOR", "Total_Comm_Cost_MB"]]
        .rename(columns={
            "BLEU1": "Full_BLEU1",
            "METEOR": "Full_METEOR",
            "Total_Comm_Cost_MB": "Full_Comm_MB",
        })
    )
    ablation_raw_seed = ablation_raw_seed.merge(
        reference, on=["Seed", "Clients"], how="left", validate="many_to_one"
    )
    ablation_raw_seed["BLEU1_Contribution_vs_Variant"] = (
        ablation_raw_seed["Full_BLEU1"] - ablation_raw_seed["BLEU1"]
    )
    ablation_raw_seed["METEOR_Contribution_vs_Variant"] = (
        ablation_raw_seed["Full_METEOR"] - ablation_raw_seed["METEOR"]
    )
    ablation_raw_seed["Comm_Change_MB_vs_Full"] = (
        ablation_raw_seed["Total_Comm_Cost_MB"] - ablation_raw_seed["Full_Comm_MB"]
    )
    ablation_raw_seed["Comm_Change_%_vs_Full"] = (
        100.0 * ablation_raw_seed["Comm_Change_MB_vs_Full"]
        / ablation_raw_seed["Full_Comm_MB"].replace(0, np.nan)
    )

    raw_path = os.path.join(ABLATION_DIR, f"ablation_final_seed_{RUN_SEED}.csv")
    hist_path = os.path.join(ABLATION_DIR, f"ablation_history_seed_{RUN_SEED}.csv")
    xlsx_path = os.path.join(ABLATION_DIR, f"seed_{RUN_SEED}_ablation_results.xlsx")
    ablation_raw_seed.to_csv(raw_path, index=False)
    ablation_history_seed.to_csv(hist_path, index=False)
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        ablation_raw_seed.to_excel(writer, sheet_name="Ablation_Final", index=False)
        ablation_history_seed.to_excel(writer, sheet_name="Validation_History", index=False)

    print("\n" + "=" * 110)
    print(f"SEED {RUN_SEED} ABLATION COMPLETED")
    print("=" * 110)
    display(ablation_raw_seed.sort_values(["Clients", "Ablation_Variant"]).reset_index(drop=True))
    print("Saved:", raw_path)
    print("Saved:", hist_path)
    print("Saved:", xlsx_path)
else:
    print("Ablation study is disabled: RUN_ABLATION_STUDY=False")


In [ ]:
# =========================================================
# 14B) Missing-Reconstruction Stress Ablation under Exact Missing Rates
# =========================================================
# Scientific question:
# Does the learned cross-view reconstruction component improve robustness when
# one X-ray view is actually missing? This is evaluated separately from the
# complete-modality ablation table.
#
# Comparison:
#   Full Proposed  vs  w/o Missing Reconstruction
# Conditions:
#   0%, 30%, 60%, 90% missing studies (nearest exact integer count),
#   with the same deterministic nested masks and the same held-out test set.
# =========================================================

RUN_RECONSTRUCTION_MISSING_ABLATION = True
RECONSTRUCTION_STRESS_CLIENTS = int(ABLATION_CLIENT_SETTINGS[0])
RECONSTRUCTION_STRESS_RATES = [0.0, 0.30, 0.60, 0.90]
RECONSTRUCTION_STRESS_PROTOCOLS = ["lateral", "frontal", "alternate"]
RECON_DIR = os.path.join(SEED_RUN_DIR, "reconstruction_missing_ablation")
os.makedirs(RECON_DIR, exist_ok=True)


def _load_checkpoint_from_ablation_row(variant_name, clients):
    row = ablation_raw_seed[
        (ablation_raw_seed["Ablation_Variant"] == variant_name)
        & (ablation_raw_seed["Clients"].astype(int) == int(clients))
    ]
    if len(row) != 1:
        raise RuntimeError(
            f"Ablation checkpoint row not uniquely found: {variant_name}/{clients}"
        )
    checkpoint_path = str(row.iloc[0].get("Model_Checkpoint", ""))
    if not checkpoint_path or not os.path.isfile(checkpoint_path):
        raise FileNotFoundError(
            f"Required ablation model checkpoint missing: {checkpoint_path}"
        )
    model, payload = load_model_checkpoint(checkpoint_path, device="cpu")
    return model, payload


if RUN_RECONSTRUCTION_MISSING_ABLATION:
    required_variants = ["Full_Proposed", "w_o_Missing_Reconstruction"]
    recon_models = {}
    recon_thresholds = {}
    for variant in required_variants:
        m, payload = _load_checkpoint_from_ablation_row(
            variant, RECONSTRUCTION_STRESS_CLIENTS
        )
        recon_models[variant] = m
        recon_thresholds[variant] = float(payload["calibrated_threshold"])

    recon_rows = []
    reference_masks = {}

    for variant in required_variants:
        model = recon_models[variant]
        threshold = recon_thresholds[variant]
        for rate in RECONSTRUCTION_STRESS_RATES:
            protocols = ["none"] if float(rate) == 0.0 else RECONSTRUCTION_STRESS_PROTOCOLS
            for protocol in protocols:
                eval_protocol = "lateral" if protocol == "none" else protocol
                metrics = evaluate(
                    model,
                    data_loader=test_loader,
                    fixed_threshold=threshold,
                    missing_rate=float(rate),
                    seed=DATA_SPLIT_SEED,
                    missing_view=eval_protocol,
                    return_audit=True,
                )
                audit = metrics.pop("_audit")

                expected_count = int(round(float(rate) * int(audit["sample_count"])))
                if int(audit["masked_count"]) != expected_count:
                    raise RuntimeError(
                        f"Exact missing-count audit failed: rate={rate}, protocol={protocol}, "
                        f"observed={audit['masked_count']}, expected={expected_count}"
                    )

                mask_key = (float(rate), protocol)
                if mask_key not in reference_masks:
                    reference_masks[mask_key] = audit["mask"].copy()
                elif not np.array_equal(reference_masks[mask_key], audit["mask"]):
                    raise RuntimeError(
                        f"Full/no-reconstruction variants did not receive identical masks: {mask_key}"
                    )

                row = {
                    "Seed": int(RUN_SEED),
                    "Clients": int(RECONSTRUCTION_STRESS_CLIENTS),
                    "Ablation_Variant": variant,
                    "Missing_Reconstruction": variant == "Full_Proposed",
                    "Missing_Rate": float(rate),
                    "Missing_Protocol": protocol,
                    "Requested_Missing_Rate": float(rate),
                    "Target_Masked_Samples": expected_count,
                    "Actual_Masked_Samples": int(audit["masked_count"]),
                    "Total_Samples": int(audit["sample_count"]),
                    "Actual_Missing_Rate": float(audit["actual_missing_rate"]),
                    "Feature_Energy_Retention": float(audit["feature_energy_retention"]),
                    **metrics,
                }
                recon_rows.append(row)

        model.to("cpu")
        cleanup_gpu()

    reconstruction_missing_ablation_raw_seed = pd.DataFrame(recon_rows)

    # Nested-mask audit for each nonzero protocol (using Full Proposed masks).
    for protocol in RECONSTRUCTION_STRESS_PROTOCOLS:
        m30 = reference_masks[(0.30, protocol)]
        m60 = reference_masks[(0.60, protocol)]
        m90 = reference_masks[(0.90, protocol)]
        if np.any(m30 & ~m60) or np.any(m60 & ~m90):
            raise RuntimeError(f"Nested exact-mask audit failed for protocol={protocol}")

    higher_metrics = [
        "BLEU1", "BLEU4", "ROUGE_L", "METEOR",
        "Label_F1_micro", "Label_F1_weighted", "Jaccard_micro",
    ]
    lower_metrics = ["Hamming_Loss"]

    # Per-seed mean across the three missing-view protocols; 0% is retained once.
    summary_rows = []
    for (variant, rate), g in reconstruction_missing_ablation_raw_seed.groupby(
        ["Ablation_Variant", "Missing_Rate"], sort=False
    ):
        row = {
            "Seed": int(RUN_SEED),
            "Clients": int(RECONSTRUCTION_STRESS_CLIENTS),
            "Ablation_Variant": variant,
            "Missing_Reconstruction": variant == "Full_Proposed",
            "Missing_Rate": float(rate),
            "Missing_Protocol": "none" if float(rate) == 0 else "mean_across_protocols",
            "Requested_Missing_Rate": float(rate),
            "Actual_Missing_Rate": float(g["Actual_Missing_Rate"].mean()),
            "Target_Masked_Samples": int(round(g["Target_Masked_Samples"].mean())),
            "Total_Samples": int(round(g["Total_Samples"].mean())),
        }
        for metric in higher_metrics + lower_metrics:
            row[metric] = float(g[metric].mean())
        summary_rows.append(row)

    reconstruction_missing_ablation_mean_seed = pd.DataFrame(summary_rows)

    # Paired benefit of reconstruction: Full - NoRecon for higher-is-better metrics.
    full = reconstruction_missing_ablation_mean_seed[
        reconstruction_missing_ablation_mean_seed["Ablation_Variant"] == "Full_Proposed"
    ].copy()
    no_recon = reconstruction_missing_ablation_mean_seed[
        reconstruction_missing_ablation_mean_seed["Ablation_Variant"] == "w_o_Missing_Reconstruction"
    ].copy()
    paired = full.merge(
        no_recon,
        on=["Seed", "Clients", "Missing_Rate"],
        suffixes=("_Full", "_NoRecon"),
        validate="one_to_one",
    )
    for metric in higher_metrics:
        paired[f"Reconstruction_Benefit_{metric}"] = paired[f"{metric}_Full"] - paired[f"{metric}_NoRecon"]
    paired["Reconstruction_Benefit_Hamming_Loss"] = (
        paired["Hamming_Loss_NoRecon"] - paired["Hamming_Loss_Full"]
    )
    reconstruction_missing_ablation_paired_seed = paired

    raw_path = os.path.join(RECON_DIR, f"reconstruction_missing_ablation_seed_{RUN_SEED}.csv")
    mean_path = os.path.join(RECON_DIR, f"reconstruction_missing_ablation_mean_seed_{RUN_SEED}.csv")
    paired_path = os.path.join(RECON_DIR, f"reconstruction_missing_ablation_paired_seed_{RUN_SEED}.csv")
    xlsx_path = os.path.join(RECON_DIR, f"reconstruction_missing_ablation_seed_{RUN_SEED}.xlsx")

    reconstruction_missing_ablation_raw_seed.to_csv(raw_path, index=False)
    reconstruction_missing_ablation_mean_seed.to_csv(mean_path, index=False)
    reconstruction_missing_ablation_paired_seed.to_csv(paired_path, index=False)
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        reconstruction_missing_ablation_raw_seed.to_excel(writer, sheet_name="Protocol_Level", index=False)
        reconstruction_missing_ablation_mean_seed.to_excel(writer, sheet_name="Mean_Protocols", index=False)
        reconstruction_missing_ablation_paired_seed.to_excel(writer, sheet_name="Paired_Benefit", index=False)

    print("\nMISSING-RECONSTRUCTION STRESS ABLATION: COMPLETED")
    display(reconstruction_missing_ablation_mean_seed)
    print("Paired reconstruction benefit (positive = Full Proposed better):")
    display(reconstruction_missing_ablation_paired_seed)
    print("Saved:", raw_path)
    print("Saved:", mean_path)
else:
    print("Missing-reconstruction stress ablation disabled.")


In [ ]:
# =========================================================
# OPTIONAL SINGLE-SEED DIAGNOSTIC FIGURES
# =========================================================
# These plots are diagnostic only. Publication uncertainty/error bars are
# generated in the separate five-seed aggregation notebook.
# =========================================================
import os
import matplotlib.pyplot as plt

DIAGNOSTIC_FIG_DIR = os.path.join(SEED_RUN_DIR, "diagnostic_figures")
os.makedirs(DIAGNOSTIC_FIG_DIR, exist_ok=True)

for metric in ["BLEU1", "METEOR", "Total_Comm_Cost_MB"]:
    if metric not in final_test_seed.columns:
        continue
    fig, ax = plt.subplots(figsize=(8, 5))
    for method in METHODS:
        d = final_test_seed[final_test_seed["Method"] == method].sort_values("Clients")
        if d.empty:
            continue
        ax.plot(d["Clients"], d[metric], marker="o", label=method)
    ax.set_xlabel("Number of Clients")
    ax.set_ylabel(metric)
    ax.set_title(f"Seed {RUN_SEED}: {metric} (diagnostic only)")
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(DIAGNOSTIC_FIG_DIR, f"seed_{RUN_SEED}_{metric}.png"), dpi=200)
    plt.show()
    plt.close(fig)


In [ ]:
# =========================================================
# 14) Audited Missing-Modality Evaluation
# =========================================================
# Purpose:
# - Verify that the missing-view mask actually changes the features entering
#   both the classification and report-generation branches.
# - Use identical deterministic, nested masks for all methods.
# - Report every protocol separately before any cross-protocol summary.
# - Quantify prediction changes rather than interpreting identical rounded F1
#   values as proof that the classifier was unaffected.
# - Never alter, smooth, clip, or force any evaluation result.
# =========================================================

import os
import gc
import numpy as np
import pandas as pd
import torch
from IPython.display import display

MISSING_RATES = [0.0, 0.30, 0.60, 0.90]
MISSING_AUDIT_SEED = DATA_SPLIT_SEED  # deterministic corruption-mask seed; separate from five-seed training summary
METRIC_IMPROVEMENT_TOLERANCE = 0.005
PREDICTION_CHANGE_TOLERANCE = 1e-12
EVAL_MAX_BATCHES = 999999

evaluation_protocols = list(
    globals().get(
        "MISSING_VIEW_PROTOCOLS",
        ["lateral", "frontal", "alternate"],
    )
)

required_metric_columns = [
    "BLEU1", "BLEU4", "ROUGE_L", "METEOR",
    "Label_F1_micro", "Label_F1_weighted",
    "Jaccard_micro", "Hamming_Loss",
]

higher_is_better = [
    "BLEU1", "BLEU4", "ROUGE_L", "METEOR",
    "Label_F1_micro", "Label_F1_weighted", "Jaccard_micro",
]
lower_is_better = ["Hamming_Loss"]

required_objects = [
    "FINAL_MODELS", "FINAL_THRESHOLDS", "evaluate",
    "METHODS", "CLIENT_SETTINGS",
]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        "Run the complete federated-training cell first. Missing objects: "
        + ", ".join(missing_objects)
    )

expected_model_keys = [
    (method, int(clients))
    for clients in CLIENT_SETTINGS
    for method in METHODS
]
missing_keys = [key for key in expected_model_keys if key not in FINAL_MODELS]
if missing_keys:
    raise RuntimeError(
        f"Missing {len(missing_keys)} final trained models: {missing_keys}"
    )

CHECKPOINT_DIR = os.path.join(SEED_RUN_DIR, "missing_modality_audit")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

raw_rows = []
audit_rows = []
baseline_details = {}

for clients in CLIENT_SETTINGS:
    for method in METHODS:
        key = (method, int(clients))
        model = FINAL_MODELS[key]
        fixed_threshold = FINAL_THRESHOLDS.get(key, 0.5)

        print("\n" + "=" * 80)
        print(f"AUDIT | Method={method} | Clients={clients}")
        print("=" * 80)

        baseline_metrics = evaluate(
            model,
            max_batches=EVAL_MAX_BATCHES,
            fixed_threshold=fixed_threshold,
            missing_rate=0.0,
            seed=MISSING_AUDIT_SEED,
            missing_view="lateral",
            return_audit=True,
        )
        baseline_audit = baseline_metrics.pop("_audit")
        baseline_details[key] = baseline_audit

        baseline_row = {
            "Method": method,
            "Clients": int(clients),
            "Missing_Rate": 0.0,
            "Missing_Protocol": "none",
            "Requested_Missing_Rate": 0.0,
            "Nearest_Achievable_Missing_Rate": 0.0,
            "Actual_Missing_Rate": 0.0,
            "Target_Masked_Samples": 0,
            "Masked_Samples": 0,
            "Total_Samples": int(baseline_audit["sample_count"]),
            "Feature_Energy_Retention": 1.0,
            "Changed_Label_Decisions": 0,
            "Changed_Label_Decision_Rate": 0.0,
            "Changed_Study_Predictions": 0,
            "Changed_Study_Prediction_Rate": 0.0,
            "Mean_Absolute_Probability_Shift": 0.0,
            "Maximum_Probability_Shift": 0.0,
        }
        baseline_row.update(baseline_metrics)
        raw_rows.append(baseline_row)

        for missing_rate in MISSING_RATES[1:]:
            for protocol in evaluation_protocols:
                current_metrics = evaluate(
                    model,
                    max_batches=EVAL_MAX_BATCHES,
                    fixed_threshold=fixed_threshold,
                    missing_rate=float(missing_rate),
                    seed=MISSING_AUDIT_SEED,
                    missing_view=protocol,
                    return_audit=True,
                )
                current_audit = current_metrics.pop("_audit")

                expected_masked = int(round(float(missing_rate) * int(current_audit["sample_count"])))
                if int(current_audit["masked_count"]) != expected_masked:
                    raise RuntimeError(
                        f"Exact missing-count audit failed: requested={missing_rate}, "
                        f"observed={current_audit['masked_count']}, expected={expected_masked}."
                    )

                if not np.array_equal(
                    current_audit["sample_indices"],
                    baseline_audit["sample_indices"],
                ):
                    raise RuntimeError(
                        "Sample ordering changed between baseline and "
                        "missing-modality evaluation."
                    )

                if not np.array_equal(
                    current_audit["y_true"], baseline_audit["y_true"]
                ):
                    raise RuntimeError(
                        "Ground-truth labels changed during missing-modality evaluation."
                    )

                probability_shift = np.abs(
                    current_audit["y_prob"] - baseline_audit["y_prob"]
                )
                decision_difference = (
                    current_audit["y_pred"] != baseline_audit["y_pred"]
                )
                study_difference = decision_difference.any(axis=1)

                current_row = {
                    "Method": method,
                    "Clients": int(clients),
                    "Missing_Rate": float(missing_rate),
                    "Missing_Protocol": protocol,
                    "Requested_Missing_Rate": float(missing_rate),
                    "Nearest_Achievable_Missing_Rate":
                        float(current_audit["nearest_achievable_rate"]),
                    "Actual_Missing_Rate":
                        float(current_audit["actual_missing_rate"]),
                    "Target_Masked_Samples": int(current_audit["target_masked_count"]),
                    "Masked_Samples": int(current_audit["masked_count"]),
                    "Total_Samples": int(current_audit["sample_count"]),
                    "Feature_Energy_Retention":
                        float(current_audit["feature_energy_retention"]),
                    "Changed_Label_Decisions":
                        int(decision_difference.sum()),
                    "Changed_Label_Decision_Rate":
                        float(decision_difference.mean()),
                    "Changed_Study_Predictions":
                        int(study_difference.sum()),
                    "Changed_Study_Prediction_Rate":
                        float(study_difference.mean()),
                    "Mean_Absolute_Probability_Shift":
                        float(probability_shift.mean()),
                    "Maximum_Probability_Shift":
                        float(probability_shift.max()),
                }
                current_row.update(current_metrics)
                raw_rows.append(current_row)

                print(
                    f"rate={missing_rate:.2f} | protocol={protocol:<9} | "
                    f"actual={current_audit['actual_missing_rate']:.3f} | "
                    f"BLEU1={current_metrics['BLEU1']:.6f} | "
                    f"F1={current_metrics['Label_F1_micro']:.6f} | "
                    f"changed studies={study_difference.mean():.3%} | "
                    f"prob shift={probability_shift.mean():.6e}"
                )

        try:
            model.to("cpu")
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        pd.DataFrame(raw_rows).to_csv(
            os.path.join(CHECKPOINT_DIR, "missing_modality_protocol_audit.csv"),
            index=False,
        )

missing_modality_protocol_table = (
    pd.DataFrame(raw_rows)
    .sort_values(
        ["Clients", "Method", "Missing_Rate", "Missing_Protocol"]
    )
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# Validate actual missing rates and nested masks
# ---------------------------------------------------------
for clients in CLIENT_SETTINGS:
    for protocol in evaluation_protocols:
        reference_masks = {}
        reference_indices = None

        # Masks are method-independent. Use the first method to audit nesting.
        method = METHODS[0]
        model = FINAL_MODELS[(method, int(clients))]
        threshold = FINAL_THRESHOLDS.get((method, int(clients)), 0.5)

        for rate in MISSING_RATES[1:]:
            audit_result = evaluate(
                model,
                max_batches=EVAL_MAX_BATCHES,
                fixed_threshold=threshold,
                missing_rate=rate,
                seed=MISSING_AUDIT_SEED,
                missing_view=protocol,
                return_audit=True,
            )["_audit"]
            reference_masks[rate] = audit_result["mask"]
            if reference_indices is None:
                reference_indices = audit_result["sample_indices"]
            elif not np.array_equal(
                reference_indices, audit_result["sample_indices"]
            ):
                raise RuntimeError("Sample indices changed across missing rates.")

        if np.any(reference_masks[0.30] & ~reference_masks[0.60]):
            raise RuntimeError("30% mask is not nested inside the 60% mask.")
        if np.any(reference_masks[0.60] & ~reference_masks[0.90]):
            raise RuntimeError("60% mask is not nested inside the 90% mask.")

# ---------------------------------------------------------
# Add baseline deltas
# ---------------------------------------------------------
def add_baseline_deltas(table):
    table = table.copy()
    baseline = (
        table.loc[
            table["Missing_Rate"] == 0.0,
            ["Method", "Clients"] + required_metric_columns,
        ]
        .drop_duplicates(["Method", "Clients"])
        .rename(columns={
            metric: f"__baseline_{metric}"
            for metric in required_metric_columns
        })
    )
    table = table.merge(
        baseline,
        on=["Method", "Clients"],
        how="left",
        validate="many_to_one",
    )
    for metric in required_metric_columns:
        table[f"Delta_{metric}_vs_0"] = (
            table[metric] - table[f"__baseline_{metric}"]
        )
        denominator = table[f"__baseline_{metric}"].abs().clip(lower=1e-12)
        table[f"Relative_Delta_{metric}_Percent"] = (
            table[f"Delta_{metric}_vs_0"] / denominator * 100.0
        )
    return table.drop(columns=[
        f"__baseline_{metric}" for metric in required_metric_columns
    ])

missing_modality_protocol_table = add_baseline_deltas(
    missing_modality_protocol_table
)

# ---------------------------------------------------------
# Cross-protocol summaries; protocol-level table remains primary
# ---------------------------------------------------------
audit_numeric_columns = [
    "Nearest_Achievable_Missing_Rate",
    "Actual_Missing_Rate",
    "Target_Masked_Samples",
    "Masked_Samples",
    "Total_Samples",
    "Feature_Energy_Retention",
    "Changed_Label_Decisions",
    "Changed_Label_Decision_Rate",
    "Changed_Study_Predictions",
    "Changed_Study_Prediction_Rate",
    "Mean_Absolute_Probability_Shift",
    "Maximum_Probability_Shift",
]

def aggregate_protocols(table, mode):
    output = []
    for (method, clients, rate), group in table.groupby(
        ["Method", "Clients", "Missing_Rate"], sort=False
    ):
        row = {
            "Method": method,
            "Clients": int(clients),
            "Missing_Rate": float(rate),
            "Missing_Protocol": (
                "none" if rate == 0 else
                "mean_across_protocols" if mode == "mean"
                else "worst_across_protocols"
            ),
        }

        for metric in higher_is_better:
            row[metric] = float(
                group[metric].mean() if mode == "mean"
                else group[metric].min()
            )
        for metric in lower_is_better:
            row[metric] = float(
                group[metric].mean() if mode == "mean"
                else group[metric].max()
            )
        for column in audit_numeric_columns:
            if column in group.columns:
                row[column] = float(group[column].mean())

        for column in [
            "Best_Threshold", "Best_Label_TopK", "Calibration_Mode"
        ]:
            if column in group.columns:
                row[column] = group[column].iloc[0]

        output.append(row)

    return (
        pd.DataFrame(output)
        .sort_values(["Clients", "Method", "Missing_Rate"])
        .reset_index(drop=True)
    )

missing_modality_mean_table = add_baseline_deltas(
    aggregate_protocols(missing_modality_protocol_table, "mean")
)
missing_modality_worstcase_table = add_baseline_deltas(
    aggregate_protocols(missing_modality_protocol_table, "worst")
)

# ---------------------------------------------------------
# Diagnostic flags: report only; never alter measurements
# ---------------------------------------------------------
flags = []

for _, row in missing_modality_protocol_table.iterrows():
    if row["Missing_Rate"] == 0:
        continue

    expected_count = int(round(float(row["Requested_Missing_Rate"]) * int(row["Total_Samples"])))
    if int(row["Masked_Samples"]) != expected_count:
        flags.append({
            "Severity": "ERROR",
            "Method": row["Method"],
            "Clients": row["Clients"],
            "Missing_Rate": row["Missing_Rate"],
            "Protocol": row["Missing_Protocol"],
            "Issue": "Masked study count does not equal round(requested_rate * N).",
            "Observed_Value": row["Masked_Samples"],
        })

    if row["Feature_Energy_Retention"] >= 0.999999:
        flags.append({
            "Severity": "ERROR",
            "Method": row["Method"],
            "Clients": row["Clients"],
            "Missing_Rate": row["Missing_Rate"],
            "Protocol": row["Missing_Protocol"],
            "Issue": "Feature tensor was not measurably changed.",
            "Observed_Value": row["Feature_Energy_Retention"],
        })

    if (
        row["Mean_Absolute_Probability_Shift"]
        <= PREDICTION_CHANGE_TOLERANCE
    ):
        flags.append({
            "Severity": "ERROR",
            "Method": row["Method"],
            "Clients": row["Clients"],
            "Missing_Rate": row["Missing_Rate"],
            "Protocol": row["Missing_Protocol"],
            "Issue":
                "Classification probabilities did not change; verify that "
                "the mask reaches the classifier input.",
            "Observed_Value": row["Mean_Absolute_Probability_Shift"],
        })

    if row["Changed_Study_Prediction_Rate"] == 0:
        flags.append({
            "Severity": "REVIEW",
            "Method": row["Method"],
            "Clients": row["Clients"],
            "Missing_Rate": row["Missing_Rate"],
            "Protocol": row["Missing_Protocol"],
            "Issue":
                "Thresholded label predictions are unchanged despite "
                "probability changes. This may explain identical rounded F1.",
            "Observed_Value": row["Changed_Study_Prediction_Rate"],
        })

    for metric in higher_is_better:
        delta = row[f"Delta_{metric}_vs_0"]
        if delta > METRIC_IMPROVEMENT_TOLERANCE:
            flags.append({
                "Severity": "REVIEW",
                "Method": row["Method"],
                "Clients": row["Clients"],
                "Missing_Rate": row["Missing_Rate"],
                "Protocol": row["Missing_Protocol"],
                "Issue":
                    f"{metric} improved materially under missingness; "
                    "inspect this protocol separately rather than relying "
                    "on the cross-protocol mean.",
                "Observed_Value": delta,
            })

sanity_flags = (
    pd.DataFrame(flags)
    if flags else
    pd.DataFrame([{
        "Severity": "PASS",
        "Method": "All",
        "Clients": np.nan,
        "Missing_Rate": np.nan,
        "Protocol": "All",
        "Issue": "No audit warning was detected.",
        "Observed_Value": np.nan,
    }])
)

# A focused classification audit table for the manuscript appendix.
classification_missing_audit_table = (
    missing_modality_protocol_table[[
        "Method", "Clients", "Missing_Rate", "Missing_Protocol",
        "Requested_Missing_Rate", "Actual_Missing_Rate",
        "Feature_Energy_Retention",
        "Mean_Absolute_Probability_Shift",
        "Maximum_Probability_Shift",
        "Changed_Label_Decision_Rate",
        "Changed_Study_Prediction_Rate",
        "Label_F1_micro", "Label_F1_weighted",
        "Jaccard_micro", "Hamming_Loss",
        "Delta_Label_F1_micro_vs_0",
        "Delta_Label_F1_weighted_vs_0",
        "Delta_Jaccard_micro_vs_0",
        "Delta_Hamming_Loss_vs_0",
    ]]
    .copy()
)

print("\n" + "=" * 80)
print("AUDITED MISSING-MODALITY EVALUATION COMPLETED")
print("=" * 80)
print("Protocol-level rows:", len(missing_modality_protocol_table))
print("Audit flags:", len(sanity_flags))

print("\nProtocol-level results (primary scientific evidence):")
display(missing_modality_protocol_table)

print("\nClassification-path audit:")
display(classification_missing_audit_table)

print("\nMean across protocols (secondary summary only):")
display(missing_modality_mean_table)

print("\nWorst case across protocols:")
display(missing_modality_worstcase_table)

print("\nDiagnostic flags:")
display(sanity_flags)


In [ ]:
# =========================================================
# 15) Display and Export Complete Missing-Modality Results
# =========================================================
# This cell displays all missing-modality results without truncation
# and exports complete tables to CSV and Excel files.
# Run this cell after Cell 14 has finished.
# =========================================================

import pandas as pd
import os

# ---------------------------------------------------------
# 1) Show all rows and all columns in Colab / Jupyter
# ---------------------------------------------------------
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.expand_frame_repr", False)

# ---------------------------------------------------------
# 2) Check that required tables exist
# ---------------------------------------------------------
required_tables = [
    "missing_modality_protocol_table",
    "missing_modality_mean_table",
    "missing_modality_worstcase_table",
    "classification_missing_audit_table",
    "sanity_flags"
]

for table_name in required_tables:
    if table_name not in globals():
        raise NameError(
            f"{table_name} is not found. Please run the Missing-Modality Evaluation cell first."
        )

# ---------------------------------------------------------
# 3) Define full display columns
# ---------------------------------------------------------
preferred_cols = [
    "Method",
    "Clients",
    "Missing_Rate",
    "Missing_Protocol",
    "Requested_Missing_Rate",
    "Nearest_Achievable_Missing_Rate",
    "Target_Masked_Samples",
    "Actual_Missing_Rate",
    "Masked_Samples",
    "Total_Samples",
    "BLEU1",
    "BLEU4",
    "ROUGE_L",
    "METEOR",
    "Label_F1_micro",
    "Label_F1_weighted",
    "Jaccard_micro",
    "Hamming_Loss",
    "Best_Threshold",
    "Best_Label_TopK",
    "Calibration_Mode",
    "Delta_BLEU1_vs_0",
    "Delta_BLEU4_vs_0",
    "Delta_ROUGE_L_vs_0",
    "Delta_METEOR_vs_0",
    "Delta_Label_F1_micro_vs_0",
    "Delta_Label_F1_weighted_vs_0",
    "Delta_Jaccard_micro_vs_0",
    "Delta_Hamming_Loss_vs_0"
]

# Keep only columns that actually exist in each table
def prepare_display_table(df):
    existing_cols = [c for c in preferred_cols if c in df.columns]
    other_cols = [c for c in df.columns if c not in existing_cols]
    out = df[existing_cols + other_cols].copy()

    sort_cols = [c for c in ["Clients", "Method", "Missing_Rate", "Missing_Protocol"] if c in out.columns]
    if sort_cols:
        out = out.sort_values(by=sort_cols).reset_index(drop=True)

    return out

raw_full_display = prepare_display_table(missing_modality_protocol_table)
mean_full_display = prepare_display_table(missing_modality_mean_table)
worstcase_full_display = prepare_display_table(missing_modality_worstcase_table)

# ---------------------------------------------------------
# 4) Display complete tables
# ---------------------------------------------------------
print("\n=========================================================")
print("1) RAW Missing-Modality Table - all protocols")
print("=========================================================")
display(raw_full_display)

#print("\n=========================================================")
#print("2) PUBLICATION Missing-Modality Table - mean across protocols")
#print("=========================================================")
#display(mean_full_display)

print("\n=========================================================")
print("2) STRICT Missing-Modality Table - worst-case across protocols")
print("=========================================================")
display(worstcase_full_display)

#print("\n=========================================================")
#print("4) Sanity Flags - diagnostic only")
#print("=========================================================")
#display(sanity_flags)

# ---------------------------------------------------------
# 5) Create a shorter paper-ready table if needed
# ---------------------------------------------------------
paper_cols = [
    "Method",
    "Clients",
    "Missing_Rate",
    "BLEU1",
    "BLEU4",
    "ROUGE_L",
    "METEOR",
    "Label_F1_micro",
    "Label_F1_weighted",
    "Jaccard_micro",
    "Hamming_Loss"
]

paper_cols = [c for c in paper_cols if c in missing_modality_mean_table.columns]

paper_ready_missing_table = (
    missing_modality_mean_table[paper_cols]
    .copy()
    .sort_values(by=["Clients", "Method", "Missing_Rate"])
    .reset_index(drop=True)
)

print("\n=========================================================")
print("3) Paper-Ready Missing-Modality Table - recommended for manuscript")
print("=========================================================")
display(paper_ready_missing_table)

# ---------------------------------------------------------
# 6) Save all tables to Google Drive
# ---------------------------------------------------------
SAVE_DIR = os.path.join(SEED_RUN_DIR, "missing_modality_results")
os.makedirs(SAVE_DIR, exist_ok=True)

raw_csv_path = os.path.join(SAVE_DIR, "01_raw_missing_modality_all_protocols.csv")
mean_csv_path = os.path.join(SAVE_DIR, "02_mean_missing_modality_publication_table.csv")
worst_csv_path = os.path.join(SAVE_DIR, "03_worstcase_missing_modality_table.csv")
paper_csv_path = os.path.join(SAVE_DIR, "04_paper_ready_missing_modality_table.csv")
flags_csv_path = os.path.join(SAVE_DIR, "05_sanity_flags.csv")
excel_path = os.path.join(SAVE_DIR, "complete_missing_modality_results.xlsx")

raw_full_display.to_csv(raw_csv_path, index=False)
mean_full_display.to_csv(mean_csv_path, index=False)
worstcase_full_display.to_csv(worst_csv_path, index=False)
paper_ready_missing_table.to_csv(paper_csv_path, index=False)
sanity_flags.to_csv(flags_csv_path, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    raw_full_display.to_excel(writer, sheet_name="Raw_All_Protocols", index=False)
    mean_full_display.to_excel(writer, sheet_name="Mean_Publication", index=False)
    worstcase_full_display.to_excel(writer, sheet_name="Worst_Case", index=False)
    paper_ready_missing_table.to_excel(writer, sheet_name="Paper_Ready", index=False)
    sanity_flags.to_excel(writer, sheet_name="Sanity_Flags", index=False)

print("\n=========================================================")
print("Files saved successfully:")
print("=========================================================")
print("Raw CSV:        ", raw_csv_path)
print("Mean CSV:       ", mean_csv_path)
print("Worst-case CSV: ", worst_csv_path)
print("Paper-ready CSV:", paper_csv_path)
print("Sanity CSV:     ", flags_csv_path)
print("Excel file:     ", excel_path)

# ---------------------------------------------------------
# Additional audited exports
# ---------------------------------------------------------
classification_audit_csv = os.path.join(
    CHECKPOINT_DIR, "classification_missing_modality_audit.csv"
)
classification_missing_audit_table.to_csv(
    classification_audit_csv, index=False
)

audit_workbook = os.path.join(
    CHECKPOINT_DIR, "missing_modality_complete_audit.xlsx"
)
with pd.ExcelWriter(audit_workbook) as writer:
    missing_modality_protocol_table.to_excel(
        writer, sheet_name="Protocol_Level", index=False
    )
    missing_modality_mean_table.to_excel(
        writer, sheet_name="Mean_Summary", index=False
    )
    missing_modality_worstcase_table.to_excel(
        writer, sheet_name="Worst_Case", index=False
    )
    classification_missing_audit_table.to_excel(
        writer, sheet_name="Classification_Audit", index=False
    )
    sanity_flags.to_excel(
        writer, sheet_name="Diagnostic_Flags", index=False
    )

print("Classification audit CSV:", classification_audit_csv)
print("Complete audit workbook:", audit_workbook)


## CAR-MFL source-alignment note for the manuscript

The defining CAR-MFL mechanism is now implemented as **intra-modal top-10 retrieval from a multimodal public bank, followed by Jaccard label refinement and use of the associated complementary modality**, as described by Poudel et al. (MICCAI 2024). The public bank is also trained as a virtual client.

Two study-specific adaptations must be disclosed: (1) the present IU-Xray experiment uses a small disjoint training-only proxy public bank because no external MIMIC-CXR public bank is bundled with this notebook; and (2) the source paper's encoder-specific aggregation weight re-adjustment (alpha=0.3) cannot be reproduced exactly with the study's shared visual projection without changing the controlled backbone. It is therefore not approximated by scaling the whole model.

FedAvg, FedProx, Fedsw, and Proposed CoHiM-FL remain isolated from this correction. The proposed method retains its prior learned cross-view reconstruction backbone and initialization.


In [ ]:
# =========================================================
# FINAL CELL — Archive This Seed's Outputs
# =========================================================
# Run after all desired cells for this seed have completed.
# The archive contains only this seed's isolated result directory.
# =========================================================
import os, zipfile
from datetime import datetime

RUN_SEED = int(EXPERIMENT_SEEDS[0])
archive_dir = os.path.join(SEED_RUN_DIR, "archives")
os.makedirs(archive_dir, exist_ok=True)
archive_path = os.path.join(
    archive_dir,
    f"CoHiM-FL_seed_{RUN_SEED}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
)

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as zf:
    for root, dirs, files in os.walk(SEED_RUN_DIR):
        excluded = {"archives", ".ipynb_checkpoints", "__pycache__"}
        if not ARCHIVE_MODEL_CHECKPOINTS:
            excluded.add("model_checkpoints")
        dirs[:] = [d for d in dirs if d not in excluded]
        for filename in files:
            full = os.path.join(root, filename)
            rel = os.path.relpath(full, SEED_RUN_DIR)
            zf.write(full, arcname=os.path.join(f"seed_{RUN_SEED}", rel))
    manifest = "\n".join([
        "CoHiM-FL split-seed run",
        f"Seed: {RUN_SEED}",
        f"Fixed data split seed: {DATA_SPLIT_SEED}",
        f"Methods: {METHODS}",
        f"Client settings: {CLIENT_SETTINGS}",
        f"Ablation client settings: {ABLATION_CLIENT_SETTINGS}",
        "This archive contains raw results for one pre-specified seed only.",
        f"Model checkpoints archived inside ZIP: {ARCHIVE_MODEL_CHECKPOINTS}",
        "Checkpoint manifest remains in the seed directory even when checkpoint binaries are excluded from ZIP.",
        "Mean ± SD must be computed only after all five seed archives/results are available.",
    ])
    zf.writestr(f"seed_{RUN_SEED}/RUN_MANIFEST.txt", manifest)

with zipfile.ZipFile(archive_path, "r") as zf:
    bad = zf.testzip()
if bad is not None:
    raise RuntimeError(f"Archive integrity check failed: {bad}")

print("Seed archive created successfully:", archive_path)
SEED_RESULTS_ZIP = archive_path
